# Stage 11 — Division & Association Bottleneck Research

## Starting point

Frozen-V9 routed hybrid baseline:

- Edge Jaccard: 0.781989
- Adjusted Edge Jaccard: 0.774278
- Division Jaccard: 0.031746
- Division TP / FP / FN: 2 / 33 / 28
- Node recall: 0.959822
- Score: **0.777453**

Routing:
- 44b6 → Sparse9995
- 6bba → Baseline256

## Completed investigations

### 11.01 — Division Error Decomposition
- 30 GT divisions on Fold0
- TP = 2, FN = 28
- Detection-limited FN = 5
- Linking/policy-limited FN = 23
- 19 FN cases had parent + both daughters present but required edges missing

### 11.02 — Association Source/Dataflow Audit
- Association uses dense transformer logits
- Column-wise softmax over source nodes
- Threshold > 0.5
- Greedy child ≤ 2 / parent ≤ 1
- No spatial gate, Hungarian, mutual-NN, or ILP in current predictor

### 11.03 — Division Geometry Audit
- Current geometry gate passes only 63 / 151 GT divisions globally
- Main restriction: max parent-daughter distance ≤ 8 µm
- Relaxing geometry increased GT coverage but increased FP and reduced score
- Geometry retained only as a weak sanity filter

### 11.04 — Targeted Association Logit Replay
23 missing required daughter edges:
- column rank ≤ 3: 19 / 23
- LOW_RANK_MODEL_FAILURE: 16 / 23
- Existing association probability / geometry features do not separate TP from FP

Conclusion:
**ASSOCIATION_MODEL_SCORE_LIMITED**

### 11.05 — Division-Positive Loss Reweighting
Division positive edges are only ~0.234% of GT association edges.
Extra second-daughter supervision is only ~0.117%.

DivPos16:
- training stable
- node recall improved
- Edge Jaccard improved
- but division TP fell 2 → 1
- division FP rose 33 → 43
- final score = 0.775344

p2 recalibration did not recover the baseline.

Conclusion:
**DIVPOS16 REJECTED**

### 11.06 — Parent Division Representation Audit

Frozen 32-dim UNet parent feature linear probe:

Validation:
- prevalence = 0.10274
- PR-AUC = 0.237856
- PR lift = 2.315×
- ROC-AUC = 0.679898

Shuffle q95:
- PR-AUC = 0.179102
- ROC-AUC = 0.609504

Therefore real division information exists in the frozen visual representation.

However, on actual predicted forks:
- true TP parent ranks = 18, 24
- top 15 judgeable forks were all FP
- probe < 0.40 retains both TP but rejects 11 / 33 FP

For those 11 rejected FP forks:
- higher-prob association branch was true continuation: 7 / 11
- lower-prob branch was true continuation: 4 / 11

63.6% branch correctness is not sufficient for safe graph collapse.

Conclusion:
**PARENT_DIVISION_SIGNAL_PRESENT_BUT_NOT_ACTIONABLE_AS_SIMPLE_GATE**

Do not promote:
- parent-probe threshold post-processing
- lower-probability daughter deletion
- simple parent-only division head


In [ ]:
# ============================================================
# RUN — 11.07B Context-Aware Transformer Parent Probe
# Purpose:
#   Test whether the actual 128-d context-aware transformer
#   source state separates division vs continuation parents
#   better than the 32-d UNet-only representation.
# Changes files: NO
# Runs training: NO neural-network training
# Runs inference: YES — targeted division-frame replay
# Uses GT: YES — only for parent labels / matching
# Keep after running: YES
# ============================================================

from pathlib import Path
import sys, time
import numpy as np
import pandas as pd
import torch

from scipy.optimize import linear_sum_assignment
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

PROJECT = Path("/Users/heqiuyan/Desktop/biohub-cell-tracking")
sys.path.insert(0, str(PROJECT / "scripts"))
sys.path.insert(0, str(PROJECT / "external/official/scripts"))
sys.path.insert(0, str(PROJECT / "src"))

from replay_stage11_association import DatasetReplay, load_model
from biohub_cell_tracking.frozen_v9 import load_graph

TRAIN_ROOT = (
    PROJECT /
    "data/raw/competition/biohub-cell-tracking-during-development/train"
)

CV = pd.read_csv(PROJECT / "configs/cv_manifest.csv")
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

CHECKPOINTS = {
    "44b6": PROJECT / (
        "models/official_baseline/stage10_pilot_sparse9995_256/"
        "split_0/edge_predictor_best.pth"
    ),
    "6bba": PROJECT / (
        "models/official_baseline/stage10_pilot_baseline_256/"
        "split_0/edge_predictor_best.pth"
    ),
}

models = {}
captures = {}

for prefix, ckpt in CHECKPOINTS.items():
    model, window_size, downsample = load_model(ckpt, device)
    model.eval()

    bucket = []
    handle = model.transformer.norm_out.register_forward_hook(
        lambda module, inp, out, b=bucket:
            b.append(out.detach().cpu())
    )

    models[prefix] = (model, window_size, tuple(downsample), handle)
    captures[prefix] = bucket


SCALE = np.array([1.625, 0.40625, 0.40625], dtype=np.float32)

def match_gt_to_pred(gt_xyz, pred_xyz):
    """Thresholded one-to-one matching at the official 7 µm radius."""
    if len(gt_xyz) == 0 or len(pred_xyz) == 0:
        return []

    dist = np.linalg.norm(
        (gt_xyz[:, None, :] - pred_xyz[None, :, :]) * SCALE,
        axis=2,
    )

    n_gt, n_pred = dist.shape
    cost = np.full((n_gt, n_pred + n_gt), 1e6, dtype=np.float64)

    cost[:, :n_pred] = np.where(dist <= 7.0, dist, 1e6)

    for i in range(n_gt):
        cost[i, n_pred + i] = 7.0001

    r, c = linear_sum_assignment(cost)

    return [
        (gi, pj, float(dist[gi, pj]))
        for gi, pj in zip(r, c)
        if pj < n_pred and dist[gi, pj] <= 7.0
    ]


examples = []
started = time.time()

for n_done, meta in enumerate(CV.itertuples(index=False), 1):
    prefix = meta.prefix
    model, window_size, downsample, _ = models[prefix]

    gt = load_graph(TRAIN_ROOT / f"{meta.dataset}.geff")
    nodes = gt.node_attrs(unpack=True).to_pandas()

    edges = gt.edge_attrs(
        attr_keys=["source_id", "target_id"],
        unpack=True,
    ).to_pandas()

    outdeg = edges.groupby("source_id").size()

    div_ids = set(outdeg[outdeg == 2].index.astype(int))

    if not div_ids:
        continue

    div_times = sorted(
        nodes.loc[
            nodes["node_id"].isin(div_ids),
            "t",
        ].astype(int).unique()
    )

    replay = DatasetReplay(
        meta.dataset,
        TRAIN_ROOT,
        model,
        window_size,
        downsample,
        device,
    )

    for t in div_times:
        if t >= replay.T - 1:
            continue

        bucket = captures[prefix]
        bucket.clear()

        c_src, c_tgt, raw, _ = replay.replay_pair(t)

        if raw is None:
            continue

        # norm_out is called first for q, then for k.
        assert len(bucket) == 2
        q_context = bucket[0][0].numpy()

        assert q_context.shape[0] == len(c_src)
        assert q_context.shape[1] == 128

        frame = nodes[
            (nodes["t"].astype(int) == t)
            & nodes["node_id"].isin(outdeg.index)
        ].copy()

        if len(frame) == 0:
            continue

        gt_xyz = frame[["z", "y", "x"]].to_numpy(np.float32)

        # Replay detections are in downsampled voxel coordinates.
        pred_xyz = (
            c_src[:, 1:].astype(np.float32)
            * np.asarray(downsample, dtype=np.float32)
        )

        matches = match_gt_to_pred(gt_xyz, pred_xyz)

        for gi, pi, distance_um in matches:
            node_id = int(frame.iloc[gi]["node_id"])
            degree = int(outdeg.loc[node_id])

            if degree not in (1, 2):
                continue

            examples.append({
                "dataset": meta.dataset,
                "prefix": prefix,
                "fold": int(meta.fold),
                "t": t,
                "node_id": node_id,
                "label": int(degree == 2),
                "match_distance_um": distance_um,
                "feature": q_context[pi],
            })

    if n_done % 25 == 0:
        print(f"processed {n_done}/{len(CV)} datasets")

# Remove hooks.
for _, _, _, handle in models.values():
    handle.remove()

context_df = pd.DataFrame(examples)

train_df = context_df[context_df["fold"] != 0]
val_df = context_df[context_df["fold"] == 0]

X_train = np.stack(train_df["feature"])
y_train = train_df["label"].to_numpy()

X_val = np.stack(val_df["feature"])
y_val = val_df["label"].to_numpy()

mu_ctx = X_train.mean(axis=0)
sd_ctx = X_train.std(axis=0)
sd_ctx[sd_ctx < 1e-6] = 1.0

X_train = (X_train - mu_ctx) / sd_ctx
X_val = (X_val - mu_ctx) / sd_ctx

context_probe = LogisticRegression(
    C=1.0,
    class_weight="balanced",
    max_iter=5000,
    random_state=1640,
)

context_probe.fit(X_train, y_train)

p_train = context_probe.predict_proba(X_train)[:, 1]
p_val = context_probe.predict_proba(X_val)[:, 1]

def metrics(split, y, p):
    prevalence = float(y.mean())
    ap = average_precision_score(y, p)

    return {
        "split": split,
        "n": len(y),
        "positive": int(y.sum()),
        "prevalence": prevalence,
        "PR_AUC": ap,
        "PR_lift": ap / prevalence,
        "ROC_AUC": roc_auc_score(y, p),
    }

summary_1107b = pd.DataFrame([
    metrics("train", y_train, p_train),
    metrics("val", y_val, p_val),
])

print("\n========== 11.07B COMPLETE ==========")
print(f"elapsed       : {(time.time() - started) / 60:.1f} min")
print("examples      :", len(context_df))
print("divisions     :", int(context_df["label"].sum()))
print("feature dim   :", X_train.shape[1])
print(
    "median match  :",
    context_df["match_distance_um"].median(),
    "µm",
)

print("\n========== CONTEXT PROBE ==========")
display(summary_1107b)

print("\n========== UNET-ONLY REFERENCE ==========")
print("Val PR-AUC : 0.237856")
print("Val PR lift: 2.315136")
print("Val ROC-AUC: 0.679898")

In [ ]:
# ============================================================
# RUN — 11.07C Paired Transformer Representation Ablation
# Purpose:
#   Compare UNet, position, pre-attention, and post-attention
#   features on EXACTLY the same detected/matched parent nodes.
# Changes files: NO
# Runs training: NO neural-network training
# Runs inference: YES — targeted frozen replay (~3 min)
# Uses GT: YES — parent labels / matching only
# Keep after running: YES
# ============================================================

import time
import numpy as np
import pandas as pd

# Reload models cleanly because 11.07B removed its hooks.
models_1107c = {}
captures_1107c = {}

for prefix, ckpt in CHECKPOINTS.items():
    model, window_size, downsample = load_model(ckpt, device)
    model.eval()

    buckets = {
        "raw": [],
        "pre": [],
        "post": [],
    }

    handles = [
        model.transformer.proj.register_forward_pre_hook(
            lambda m, inp, b=buckets["raw"]:
                b.append(inp[0].detach().cpu())
        ),
        model.transformer.norm_in.register_forward_hook(
            lambda m, inp, out, b=buckets["pre"]:
                b.append(out.detach().cpu())
        ),
        model.transformer.norm_out.register_forward_hook(
            lambda m, inp, out, b=buckets["post"]:
                b.append(out.detach().cpu())
        ),
    ]

    models_1107c[prefix] = (
        model,
        window_size,
        tuple(downsample),
        handles,
    )
    captures_1107c[prefix] = buckets


rows = []
started = time.time()

for n_done, meta in enumerate(CV.itertuples(index=False), 1):
    prefix = meta.prefix

    model, window_size, downsample, _ = models_1107c[prefix]
    buckets = captures_1107c[prefix]

    gt = load_graph(TRAIN_ROOT / f"{meta.dataset}.geff")
    nodes = gt.node_attrs(unpack=True).to_pandas()

    edges = gt.edge_attrs(
        attr_keys=["source_id", "target_id"],
        unpack=True,
    ).to_pandas()

    outdeg = edges.groupby("source_id").size()
    div_ids = set(outdeg[outdeg == 2].index.astype(int))

    if not div_ids:
        continue

    div_times = sorted(
        nodes.loc[
            nodes["node_id"].isin(div_ids),
            "t",
        ].astype(int).unique()
    )

    replay = DatasetReplay(
        meta.dataset,
        TRAIN_ROOT,
        model,
        window_size,
        downsample,
        device,
    )

    for t in div_times:
        if t >= replay.T - 1:
            continue

        for b in buckets.values():
            b.clear()

        c_src, _, raw_logits, _ = replay.replay_pair(t)

        if raw_logits is None:
            continue

        # proj/norm_in/norm_out each run first for source, then target.
        assert len(buckets["raw"]) == 2
        assert len(buckets["pre"]) == 2
        assert len(buckets["post"]) == 2

        raw_src = buckets["raw"][0][0].numpy()
        pre_src = buckets["pre"][0][0].numpy()
        post_src = buckets["post"][0][0].numpy()

        assert len(raw_src) == len(c_src)
        assert len(pre_src) == len(c_src)
        assert len(post_src) == len(c_src)

        frame = nodes[
            (nodes["t"].astype(int) == t)
            & nodes["node_id"].isin(outdeg.index)
        ].copy()

        if len(frame) == 0:
            continue

        gt_xyz = frame[["z", "y", "x"]].to_numpy(np.float32)

        pred_xyz = (
            c_src[:, 1:].astype(np.float32)
            * np.asarray(downsample, dtype=np.float32)
        )

        matches = match_gt_to_pred(gt_xyz, pred_xyz)

        for gi, pi, _ in matches:
            node_id = int(frame.iloc[gi]["node_id"])
            degree = int(outdeg.loc[node_id])

            if degree not in (1, 2):
                continue

            n_unet = model.unet_out_channels

            rows.append({
                "dataset": meta.dataset,
                "prefix": prefix,
                "fold": int(meta.fold),
                "t": t,
                "node_id": node_id,
                "label": int(degree == 2),

                # Actual input to transformer = UNet + positional feature.
                "unet": raw_src[pi, :n_unet].copy(),
                "position": raw_src[pi, n_unet:].copy(),

                # norm_in(proj(input)), before cross-attention.
                "pre_attention": pre_src[pi].copy(),

                # norm_out(q), after four cross-attention blocks.
                "post_attention": post_src[pi].copy(),
            })

    if n_done % 25 == 0:
        print(f"processed {n_done}/{len(CV)} datasets")


# Remove hooks.
for _, _, _, handles in models_1107c.values():
    for handle in handles:
        handle.remove()


paired_1107c = pd.DataFrame(rows)

assert len(paired_1107c) == 1387, len(paired_1107c)
assert int(paired_1107c["label"].sum()) == 151


def probe_representation(column):
    tr = paired_1107c[paired_1107c["fold"] != 0]
    va = paired_1107c[paired_1107c["fold"] == 0]

    Xtr = np.stack(tr[column])
    Xva = np.stack(va[column])

    ytr = tr["label"].to_numpy()
    yva = va["label"].to_numpy()

    mu = Xtr.mean(axis=0)
    sd = Xtr.std(axis=0)
    sd[sd < 1e-6] = 1.0

    Xtr = (Xtr - mu) / sd
    Xva = (Xva - mu) / sd

    clf = LogisticRegression(
        C=1.0,
        class_weight="balanced",
        max_iter=5000,
        random_state=1640,
    )
    clf.fit(Xtr, ytr)

    ptr = clf.predict_proba(Xtr)[:, 1]
    pva = clf.predict_proba(Xva)[:, 1]

    prevalence = yva.mean()

    return {
        "representation": column,
        "dim": Xtr.shape[1],
        "train_AP": average_precision_score(ytr, ptr),
        "val_AP": average_precision_score(yva, pva),
        "val_PR_lift": (
            average_precision_score(yva, pva) / prevalence
        ),
        "train_AUC": roc_auc_score(ytr, ptr),
        "val_AUC": roc_auc_score(yva, pva),
    }


summary_1107c = pd.DataFrame([
    probe_representation("unet"),
    probe_representation("position"),
    probe_representation("pre_attention"),
    probe_representation("post_attention"),
])

print("\n========== 11.07C COMPLETE ==========")
print(f"elapsed  : {(time.time() - started) / 60:.1f} min")
print("examples :", len(paired_1107c))
print("positive :", int(paired_1107c["label"].sum()))

print("\n========== PAIRED REPRESENTATION ABLATION ==========")
display(summary_1107c)

In [ ]:
# ============================================================
# RUN — 11.08A Local Feature Pooling Probe — 6bba
# Purpose:
#   Compare current single-voxel UNet node features against
#   3×3×3 local mean-pooled UNet features at EXACTLY the same
#   predictor-detected parent locations.
#
# Research question:
#   Does local pooling recover division information that is
#   lost because detected centroids are offset from GT centroids?
#
# Changes files: NO
# Runs training: NO neural-network training
# Runs inference: YES
#   - exact 4-view detection TTA
#   - original-view UNet feature extraction
#   - NO transformer
# Uses GT: YES
#   - parent matching
#   - division / continuation labels
# Keep after running: YES
# ============================================================

from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
import torch

from scipy.optimize import linear_sum_assignment
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


# ------------------------------------------------------------
# 1. Project setup
# ------------------------------------------------------------

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

sys.path.insert(
    0,
    str(PROJECT / "scripts"),
)
sys.path.insert(
    0,
    str(PROJECT / "external/official/scripts"),
)
sys.path.insert(
    0,
    str(PROJECT / "src"),
)

from replay_stage11_association import (
    DatasetReplay,
    _detect_cells_pooled,
    _load_frame,
    load_model,
)

from biohub_cell_tracking.frozen_v9 import (
    load_graph,
)


TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

CV = pd.read_csv(
    PROJECT / "configs/cv_manifest.csv"
)

device = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

CHECKPOINT_6BBA = (
    PROJECT
    / "models/official_baseline/"
      "stage10_pilot_baseline_256/"
      "split_0/edge_predictor_best.pth"
)

SCALE_UM = np.array(
    [1.625, 0.40625, 0.40625],
    dtype=np.float32,
)

BATCH_WINDOWS = 4


# ------------------------------------------------------------
# 2. Load frozen 6bba routed model
# ------------------------------------------------------------

model, window_size, downsample = load_model(
    CHECKPOINT_6BBA,
    device,
)

model.eval()

downsample_arr = np.asarray(
    downsample,
    dtype=np.float32,
)

print("device      :", device)
print("window_size :", window_size)
print("downsample  :", downsample)


# ------------------------------------------------------------
# 3. GT ↔ detected-node matching
# ------------------------------------------------------------

def match_gt_to_pred(
    gt_xyz,
    pred_xyz,
):
    """
    Thresholded optimal one-to-one matching
    using the official 7 µm node radius.
    """

    if len(gt_xyz) == 0 or len(pred_xyz) == 0:
        return []

    dist = np.linalg.norm(
        (
            gt_xyz[:, None, :]
            - pred_xyz[None, :, :]
        )
        * SCALE_UM,
        axis=2,
    )

    n_gt = len(gt_xyz)
    n_pred = len(pred_xyz)

    # Dummy unmatched columns allow GT nodes to remain unmatched.
    cost = np.full(
        (n_gt, n_pred + n_gt),
        1e6,
        dtype=np.float64,
    )

    cost[:, :n_pred] = np.where(
        dist <= 7.0,
        dist,
        1e6,
    )

    for i in range(n_gt):
        cost[i, n_pred + i] = 7.0001

    r, c = linear_sum_assignment(cost)

    return [
        (
            int(gi),
            int(pj),
            float(dist[gi, pj]),
        )
        for gi, pj in zip(r, c)
        if (
            pj < n_pred
            and dist[gi, pj] <= 7.0
        )
    ]


# ------------------------------------------------------------
# 4. Normalize a two-frame window
# ------------------------------------------------------------

def normalize_window(
    window,
    replay,
):
    """
    Match DatasetReplay normalization.
    """

    return (
        (
            window
            - replay.q_low
        )
        / (
            replay.q_high
            - replay.q_low
            + 1e-6
        )
    ).clamp(0.0)


# ------------------------------------------------------------
# 5. Canonicalize model.encode() detection output
# ------------------------------------------------------------

def canonicalize_det_logits(
    det_logits,
    batch_size,
    window_len,
):
    """
    Convert detection logits to:

        (B, W, 1, Z, Y, X)

    Supports both:
      - tensor output
      - list / tuple output

    This avoids assumptions about whether encode()
    exposes batch-first or frame-first logits.
    """

    # --------------------------------------------------------
    # List / tuple:
    # usually one tensor per temporal frame.
    # --------------------------------------------------------

    if isinstance(
        det_logits,
        (list, tuple),
    ):
        assert len(det_logits) == window_len, (
            len(det_logits),
            window_len,
        )

        frames = []

        for d in det_logits:
            # Expected batched:
            # (B, 1, Z, Y, X)
            if d.ndim == 5:
                assert d.shape[0] == batch_size, (
                    d.shape,
                    batch_size,
                )

                frames.append(d)

            # Unbatched fallback:
            # (1, Z, Y, X)
            elif (
                d.ndim == 4
                and batch_size == 1
            ):
                frames.append(
                    d.unsqueeze(0)
                )

            else:
                raise RuntimeError(
                    "Unexpected detection-logit "
                    f"frame shape: {tuple(d.shape)}"
                )

        out = torch.stack(
            frames,
            dim=1,
        )

        assert out.ndim == 6

        return out


    # --------------------------------------------------------
    # Tensor forms
    # --------------------------------------------------------

    if not torch.is_tensor(det_logits):
        raise TypeError(
            "Unsupported det_logits type: "
            f"{type(det_logits)}"
        )

    # Already:
    # (B, W, 1, Z, Y, X)
    if (
        det_logits.ndim == 6
        and det_logits.shape[0] == batch_size
        and det_logits.shape[1] == window_len
    ):
        return det_logits


    # Frame-first:
    # (W, B, 1, Z, Y, X)
    if (
        det_logits.ndim == 6
        and det_logits.shape[0] == window_len
        and det_logits.shape[1] == batch_size
    ):
        return det_logits.transpose(
            0,
            1,
        )


    # Unbatched:
    # (W, 1, Z, Y, X)
    if (
        det_logits.ndim == 5
        and batch_size == 1
        and det_logits.shape[0] == window_len
    ):
        return det_logits.unsqueeze(0)


    raise RuntimeError(
        "Unexpected detection logits shape: "
        f"{tuple(det_logits.shape)} "
        f"for B={batch_size}, W={window_len}"
    )


# ------------------------------------------------------------
# 6. Exact batched 4-view detection TTA
# ------------------------------------------------------------

def encode_detection_tta(
    imgs,
):
    """
    imgs:
        (B, W, Z, Y, X)

    Returns canonicalized averaged detection logits:
        (B, W, 1, Z, Y, X)

    Original + horizontal + vertical + both flips.
    """

    B = imgs.shape[0]
    W = imgs.shape[1]

    with torch.no_grad():

        _, raw = model.encode(imgs)

        det_sum = canonicalize_det_logits(
            raw,
            B,
            W,
        ).clone()

        # Exact spatial TTA used by current inference.
        for dims in [
            (-1,),
            (-2,),
            (-2, -1),
        ]:
            flipped_imgs = imgs.flip(dims)

            _, raw_flip = model.encode(
                flipped_imgs
            )

            det_flip = canonicalize_det_logits(
                raw_flip,
                B,
                W,
            )

            # Undo image-space flip on logits.
            det_sum += det_flip.flip(dims)

    return det_sum / 4.0


# ------------------------------------------------------------
# 7. 3×3×3 local mean feature
# ------------------------------------------------------------

def mean3_index(
    feat_map,
    coords,
):
    """
    feat_map:
        (C, Z, Y, X)

    coords:
        (N, 3), feature-map coordinates [z,y,x]

    Returns:
        (N, C)

    Uses the same integer anchor as current _index_features,
    then averages the surrounding 3×3×3 feature voxels.
    """

    assert feat_map.ndim == 4
    assert coords.ndim == 2
    assert coords.shape[1] == 3

    C, Z, Y, X = feat_map.shape

    base = coords.long()

    out = torch.zeros(
        (len(base), C),
        dtype=feat_map.dtype,
        device=feat_map.device,
    )

    count = 0

    for dz in (-1, 0, 1):
        for dy in (-1, 0, 1):
            for dx in (-1, 0, 1):

                z = (
                    base[:, 0] + dz
                ).clamp(
                    0,
                    Z - 1,
                )

                y = (
                    base[:, 1] + dy
                ).clamp(
                    0,
                    Y - 1,
                )

                x = (
                    base[:, 2] + dx
                ).clamp(
                    0,
                    X - 1,
                )

                out += (
                    feat_map[
                        :,
                        z,
                        y,
                        x,
                    ]
                    .transpose(0, 1)
                )

                count += 1

    assert count == 27

    return out / 27.0


# ------------------------------------------------------------
# 8. Prepare 6bba datasets only
# ------------------------------------------------------------

cv6 = (
    CV[
        CV["prefix"] == "6bba"
    ]
    .copy()
    .reset_index(drop=True)
)

print("6bba datasets:", len(cv6))

assert len(cv6) == 128


# ------------------------------------------------------------
# 9. Main extraction
# ------------------------------------------------------------

examples_1108a = []

started = time.time()


for n_done, meta in enumerate(
    cv6.itertuples(index=False),
    start=1,
):

    # --------------------------------------------------------
    # GT graph
    # --------------------------------------------------------

    gt = load_graph(
        TRAIN_ROOT
        / f"{meta.dataset}.geff"
    )

    gt_nodes = (
        gt.node_attrs(
            unpack=True
        )
        .to_pandas()
    )

    gt_edges = (
        gt.edge_attrs(
            attr_keys=[
                "source_id",
                "target_id",
            ],
            unpack=True,
        )
        .to_pandas()
    )

    outdeg = (
        gt_edges
        .groupby("source_id")
        .size()
    )

    div_ids = set(
        outdeg[
            outdeg == 2
        ].index.astype(int)
    )

    if not div_ids:
        continue


    # --------------------------------------------------------
    # Only frames containing a GT division.
    #
    # Same population design as 11.07C:
    # all tracked ordinary/dividing parents in division frames.
    # --------------------------------------------------------

    division_times = sorted(
        gt_nodes.loc[
            gt_nodes["node_id"].isin(
                div_ids
            ),
            "t",
        ]
        .astype(int)
        .unique()
    )


    replay = DatasetReplay(
        meta.dataset,
        TRAIN_ROOT,
        model,
        window_size,
        downsample,
        device,
    )


    valid_times = [
        int(t)
        for t in division_times
        if int(t) < replay.T - 1
    ]


    # --------------------------------------------------------
    # Process several division frames together.
    # --------------------------------------------------------

    for start in range(
        0,
        len(valid_times),
        BATCH_WINDOWS,
    ):

        times = valid_times[
            start:
            start + BATCH_WINDOWS
        ]

        if not times:
            continue


        # ====================================================
        # A. Exact source detection
        #
        # Predictor source detection convention:
        #
        #   t >= 1 : source t was detected in [t-1, t],
        #            therefore source slot = 1
        #
        #   t == 0 : use [0, 1],
        #            source slot = 0
        # ====================================================

        det_windows = []
        source_slots = []

        for t in times:

            if t >= 1:
                det_frame_ids = (
                    t - 1,
                    t,
                )
                source_slot = 1

            else:
                det_frame_ids = (
                    0,
                    1,
                )
                source_slot = 0

            pair = torch.stack([
                _load_frame(
                    replay.zarr,
                    ti,
                    replay.target_shape,
                    replay.downsample,
                )
                for ti in det_frame_ids
            ])

            pair = normalize_window(
                pair,
                replay,
            )

            det_windows.append(pair)
            source_slots.append(
                source_slot
            )


        det_imgs = torch.stack(
            det_windows,
            dim=0,
        ).to(device)


        assert det_imgs.shape[0] == len(times)
        assert det_imgs.shape[1] == 2


        # Exact 4-view TTA detection.
        det_logits = encode_detection_tta(
            det_imgs
        )

        assert det_logits.ndim == 6
        assert det_logits.shape[0] == len(times)
        assert det_logits.shape[1] == 2
        assert det_logits.shape[2] == 1


        detected_sources = []


        for b, (t, slot) in enumerate(
            zip(
                times,
                source_slots,
                strict=True,
            )
        ):

            # IMPORTANT:
            # preserve channel dimension.
            #
            # Shape:
            #   (1, Z, Y, X)
            det_frame = det_logits[
                b,
                slot,
            ]


            assert det_frame.ndim == 4, (
                det_frame.shape
            )

            assert det_frame.shape[0] == 1, (
                det_frame.shape
            )


            cells = _detect_cells_pooled(
                det_frame,
                t,
                replay.cfg.det_threshold,
                replay.pool_k,
            )


            assert cells.ndim == 2

            assert cells.shape[1] == 4, (
                cells.shape
            )


            detected_sources.append(
                cells
            )


        # ====================================================
        # B. Original-view association feature maps
        #
        # This exactly targets the representation used in
        # 11.07C:
        #
        #     source feature of pair [t, t+1]
        #
        # No detection TTA contribution is allowed into the
        # UNet feature map.
        # ====================================================

        assoc_windows = []

        for t in times:

            pair = torch.stack([
                _load_frame(
                    replay.zarr,
                    ti,
                    replay.target_shape,
                    replay.downsample,
                )
                for ti in (
                    t,
                    t + 1,
                )
            ])

            pair = normalize_window(
                pair,
                replay,
            )

            assoc_windows.append(
                pair
            )


        assoc_imgs = torch.stack(
            assoc_windows,
            dim=0,
        ).to(device)


        with torch.no_grad():

            unet_out, _ = model.encode(
                assoc_imgs
            )


        # Expected:
        #
        # (B, W, C, Z, Y, X)
        assert unet_out.ndim == 6, (
            unet_out.shape
        )

        assert unet_out.shape[0] == len(times)

        assert unet_out.shape[1] == 2


        # ====================================================
        # C. GT ↔ detected parent matching + feature extraction
        # ====================================================

        for b, (t, c_src) in enumerate(
            zip(
                times,
                detected_sources,
                strict=True,
            )
        ):

            if len(c_src) == 0:
                continue


            # All tracked parents in this division frame.
            frame = gt_nodes[
                (
                    gt_nodes["t"].astype(int)
                    == t
                )
                &
                gt_nodes[
                    "node_id"
                ].isin(
                    outdeg.index
                )
            ].copy()


            if len(frame) == 0:
                continue


            gt_xyz = frame[
                ["z", "y", "x"]
            ].to_numpy(
                np.float32
            )


            # c_src columns:
            #
            # [t, z, y, x]
            #
            # in downsampled feature-grid coordinates.
            assert c_src.shape[1] == 4


            pred_xyz = (
                c_src[
                    :,
                    1:4,
                ].astype(
                    np.float32
                )
                *
                downsample_arr
            )


            matches = match_gt_to_pred(
                gt_xyz,
                pred_xyz,
            )


            if not matches:
                continue


            # Predictor-node indices selected by matching.
            pred_indices = np.asarray(
                [
                    pi
                    for _, pi, _ in matches
                ],
                dtype=np.int64,
            )


            # Current feature coordinates:
            # already in downsampled grid.
            coords_np = (
                c_src[
                    pred_indices,
                    1:4,
                ]
                .astype(
                    np.float32
                )
            )


            coords = torch.from_numpy(
                coords_np
            ).to(device)


            mask = torch.ones(
                (
                    1,
                    len(coords),
                ),
                dtype=torch.bool,
                device=device,
            )


            # Source feature map from [t,t+1].
            source_map = unet_out[
                b,
                0,
            ]


            with torch.no_grad():

                # --------------------------------------------
                # Current official association representation:
                # single integer voxel lookup.
                # --------------------------------------------

                point = model._index_features(
                    source_map.unsqueeze(0),
                    coords.unsqueeze(0),
                    mask,
                )[0]


                # --------------------------------------------
                # Candidate:
                # local 3×3×3 mean feature.
                # --------------------------------------------

                mean3 = mean3_index(
                    source_map,
                    coords,
                )


            point_np = (
                point
                .detach()
                .cpu()
                .numpy()
            )

            mean3_np = (
                mean3
                .detach()
                .cpu()
                .numpy()
            )


            assert (
                point_np.shape
                == mean3_np.shape
            )


            # ------------------------------------------------
            # Record exactly matched examples.
            # ------------------------------------------------

            for j, (
                gi,
                pi,
                distance_um,
            ) in enumerate(matches):

                node_id = int(
                    frame.iloc[
                        gi
                    ]["node_id"]
                )

                degree = int(
                    outdeg.loc[
                        node_id
                    ]
                )


                # Same binary probe labels as 11.06 / 11.07:
                #
                # 0 = ordinary continuation parent
                # 1 = division parent
                if degree not in (
                    1,
                    2,
                ):
                    continue


                examples_1108a.append({
                    "dataset":
                        meta.dataset,

                    "prefix":
                        "6bba",

                    "fold":
                        int(meta.fold),

                    "t":
                        int(t),

                    "node_id":
                        node_id,

                    "label":
                        int(
                            degree == 2
                        ),

                    "match_distance_um":
                        float(
                            distance_um
                        ),

                    "point":
                        point_np[
                            j
                        ].copy(),

                    "mean3":
                        mean3_np[
                            j
                        ].copy(),
                })


    if n_done % 25 == 0:
        elapsed_min = (
            time.time()
            - started
        ) / 60.0

        print(
            f"processed {n_done}/"
            f"{len(cv6)} datasets "
            f"({elapsed_min:.1f} min)"
        )


# ------------------------------------------------------------
# 10. Build exact probe table
# ------------------------------------------------------------

pool_df_1108a = pd.DataFrame(
    examples_1108a
)


assert len(pool_df_1108a) > 0


val_1108a = pool_df_1108a[
    pool_df_1108a["fold"] == 0
].copy()


# ------------------------------------------------------------
# Fidelity gate against 11.07D.
#
# 11.07D 6bba Fold0:
#   n        = 275
#   positive = 25
# ------------------------------------------------------------

assert len(val_1108a) == 275, (
    "Validation population mismatch: "
    f"{len(val_1108a)} != 275"
)

assert int(
    val_1108a["label"].sum()
) == 25, (
    "Validation positive mismatch: "
    f"{int(val_1108a['label'].sum())} != 25"
)


# ------------------------------------------------------------
# 11. Logistic probes
# ------------------------------------------------------------

def probe_representation(
    column,
):

    train_df = pool_df_1108a[
        pool_df_1108a["fold"] != 0
    ]

    val_df = pool_df_1108a[
        pool_df_1108a["fold"] == 0
    ]


    X_train = np.stack(
        train_df[column]
    )

    X_val = np.stack(
        val_df[column]
    )


    y_train = (
        train_df["label"]
        .to_numpy()
    )

    y_val = (
        val_df["label"]
        .to_numpy()
    )


    # Train-only standardization.
    mu = X_train.mean(
        axis=0
    )

    sd = X_train.std(
        axis=0
    )

    sd[
        sd < 1e-6
    ] = 1.0


    X_train_z = (
        X_train - mu
    ) / sd

    X_val_z = (
        X_val - mu
    ) / sd


    clf = LogisticRegression(
        C=1.0,
        class_weight="balanced",
        max_iter=5000,
        random_state=1640,
    )


    clf.fit(
        X_train_z,
        y_train,
    )


    p_train = (
        clf.predict_proba(
            X_train_z
        )[:, 1]
    )

    p_val = (
        clf.predict_proba(
            X_val_z
        )[:, 1]
    )


    train_ap = (
        average_precision_score(
            y_train,
            p_train,
        )
    )

    val_ap = (
        average_precision_score(
            y_val,
            p_val,
        )
    )


    return {
        "representation":
            column,

        "dim":
            X_train.shape[1],

        "train_AP":
            train_ap,

        "val_AP":
            val_ap,

        "val_PR_lift":
            val_ap
            / y_val.mean(),

        "train_AUC":
            roc_auc_score(
                y_train,
                p_train,
            ),

        "val_AUC":
            roc_auc_score(
                y_val,
                p_val,
            ),
    }


summary_1108a = pd.DataFrame([
    probe_representation(
        "point"
    ),
    probe_representation(
        "mean3"
    ),
])


# ------------------------------------------------------------
# 12. Final result
# ------------------------------------------------------------

elapsed_min = (
    time.time()
    - started
) / 60.0


print(
    "\n========== 11.08A COMPLETE =========="
)

print(
    f"elapsed       : "
    f"{elapsed_min:.1f} min"
)

print(
    "examples      :",
    len(pool_df_1108a),
)

print(
    "positive      :",
    int(
        pool_df_1108a[
            "label"
        ].sum()
    ),
)

print(
    "val examples  :",
    len(val_1108a),
)

print(
    "val positive  :",
    int(
        val_1108a[
            "label"
        ].sum()
    ),
)

print(
    "median match  :",
    float(
        val_1108a[
            "match_distance_um"
        ].median()
    ),
    "µm",
)


print(
    "\n========== LOCAL FEATURE POOLING =========="
)

display(
    summary_1108a
)


print(
    "\n========== FIDELITY GATE =========="
)

point_row = (
    summary_1108a[
        summary_1108a[
            "representation"
        ] == "point"
    ]
    .iloc[0]
)

print(
    "point val_AP :",
    f"{point_row['val_AP']:.6f}",
    "| expected ≈ 0.104722",
)

print(
    "point val_AUC:",
    f"{point_row['val_AUC']:.6f}",
    "| expected ≈ 0.568800",
)


print(
    "\n========== PAIRED DELTA =========="
)

mean_row = (
    summary_1108a[
        summary_1108a[
            "representation"
        ] == "mean3"
    ]
    .iloc[0]
)

print(
    "Δ val_AP :",
    f"{mean_row['val_AP'] - point_row['val_AP']:+.6f}"
)

print(
    "Δ val_AUC:",
    f"{mean_row['val_AUC'] - point_row['val_AUC']:+.6f}"
)

### 11.08A — Local Feature Pooling Audit

The fast extraction path reproduced the 6bba detected-node point-feature probe closely
(AP 0.107996 vs 0.104722 previously; AUC 0.574240 vs 0.568800), providing sufficient
fidelity for the paired pooling comparison.

Replacing the current single-voxel UNet feature lookup with a 3×3×3 local mean did
not recover division information:

- Point: AP 0.107996, AUC 0.574240
- Mean3: AP 0.107502, AUC 0.549120
- Delta: AP -0.000494, AUC -0.025120

Mean pooling slightly increased training AUC (0.671473 → 0.690950) while reducing
validation AUC, so simple local averaging does not solve the detected-node
representation problem and may blur useful localized information.

Decision: SIMPLE_LOCAL_MEAN_POOLING_REJECTED.
Do not sweep larger pooling kernels.

In [ ]:
# ============================================================
# RUN — 11.09A Joint Parent–Top2 Candidate Extraction — 6bba
# Purpose:
#   Extract parent transformer state and its top-2 predicted
#   target candidates to test whether division information is
#   present in the JOINT parent-candidate representation.
#
# Candidate selection:
#   Top-2 targets by frozen model raw association logit.
#   GT is NOT used to choose candidates.
#
# Changes files: NO
# Runs training: NO
# Runs inference: YES — batched frozen detection + transformer
# Uses GT: YES — parent matching / binary division label only
# Keep after running: YES
# ============================================================

import time
import numpy as np
import pandas as pd
import torch

from replay_stage11_association import extract_pos_features


# ------------------------------------------------------------
# Reuse validated 11.08A infrastructure:
#
# model
# cv6
# TRAIN_ROOT
# device
# downsample_arr
# SCALE_UM
# BATCH_WINDOWS
# normalize_window()
# encode_detection_tta()
# canonicalize_det_logits()
# match_gt_to_pred()
# DatasetReplay
# _load_frame
# _detect_cells_pooled
# load_graph
# ------------------------------------------------------------


def encode_pair_features_and_tta(imgs):
    """
    Return:
        unet_out:
            original-view feature maps

        det_avg:
            exact 4-view averaged detection logits,
            canonical shape (B,W,1,Z,Y,X)
    """
    B, W = imgs.shape[:2]

    with torch.no_grad():
        unet_out, raw = model.encode(imgs)

        det_sum = canonicalize_det_logits(
            raw, B, W
        ).clone()

        for dims in [
            (-1,),
            (-2,),
            (-2, -1),
        ]:
            _, raw_flip = model.encode(
                imgs.flip(dims)
            )

            det_flip = canonicalize_det_logits(
                raw_flip, B, W
            )

            det_sum += det_flip.flip(dims)

    return unet_out, det_sum / 4.0


def candidate_geometry(
    src_ds,
    tgt1_ds,
    tgt2_ds,
):
    """
    Geometry in physical µm.

    Input coordinates are on downsampled grid.
    """
    src = src_ds * downsample_arr
    d1p = tgt1_ds * downsample_arr
    d2p = tgt2_ds * downsample_arr

    v1 = (d1p - src) * SCALE_UM
    v2 = (d2p - src) * SCALE_UM

    d1 = float(np.linalg.norm(v1))
    d2 = float(np.linalg.norm(v2))
    sep = float(np.linalg.norm(v1 - v2))

    denom = (
        np.linalg.norm(v1)
        * np.linalg.norm(v2)
    )

    if denom > 1e-8:
        cosang = np.clip(
            np.dot(v1, v2) / denom,
            -1.0,
            1.0,
        )
        angle = float(
            np.degrees(np.arccos(cosang))
        )
    else:
        angle = 0.0

    balance = float(
        abs(d1 - d2)
        / (d1 + d2 + 1e-8)
    )

    return np.array(
        [d1, d2, sep, angle, balance],
        dtype=np.float32,
    )


rows_1109a = []
started = time.time()

# Capture actual post-cross-attention q and k.
context_bucket = []

hook_1109a = (
    model.transformer.norm_out
    .register_forward_hook(
        lambda module, inp, out:
            context_bucket.append(
                out.detach().cpu()
            )
    )
)


for n_done, meta in enumerate(
    cv6.itertuples(index=False),
    start=1,
):

    gt = load_graph(
        TRAIN_ROOT / f"{meta.dataset}.geff"
    )

    nodes = (
        gt.node_attrs(unpack=True)
        .to_pandas()
    )

    edges = (
        gt.edge_attrs(
            attr_keys=[
                "source_id",
                "target_id",
            ],
            unpack=True,
        )
        .to_pandas()
    )

    outdeg = edges.groupby(
        "source_id"
    ).size()

    div_ids = set(
        outdeg[
            outdeg == 2
        ].index.astype(int)
    )

    if not div_ids:
        continue

    division_times = sorted(
        nodes.loc[
            nodes["node_id"].isin(
                div_ids
            ),
            "t",
        ]
        .astype(int)
        .unique()
    )

    replay = DatasetReplay(
        meta.dataset,
        TRAIN_ROOT,
        model,
        window_size,
        downsample,
        device,
    )

    valid_times = [
        int(t)
        for t in division_times
        if int(t) < replay.T - 1
    ]


    for start in range(
        0,
        len(valid_times),
        BATCH_WINDOWS,
    ):

        times = valid_times[
            start:
            start + BATCH_WINDOWS
        ]

        if not times:
            continue


        # ====================================================
        # A. Predictor-exact source detections
        # ====================================================

        src_windows = []
        src_slots = []

        for t in times:

            if t >= 1:
                ids = (t - 1, t)
                slot = 1
            else:
                ids = (0, 1)
                slot = 0

            pair = torch.stack([
                _load_frame(
                    replay.zarr,
                    ti,
                    replay.target_shape,
                    replay.downsample,
                )
                for ti in ids
            ])

            src_windows.append(
                normalize_window(
                    pair,
                    replay,
                )
            )

            src_slots.append(slot)


        src_imgs = torch.stack(
            src_windows
        ).to(device)

        src_det = encode_detection_tta(
            src_imgs
        )


        # ====================================================
        # B. [t,t+1] original features + target detection TTA
        # ====================================================

        pair_windows = []

        for t in times:

            pair = torch.stack([
                _load_frame(
                    replay.zarr,
                    ti,
                    replay.target_shape,
                    replay.downsample,
                )
                for ti in (
                    t,
                    t + 1,
                )
            ])

            pair_windows.append(
                normalize_window(
                    pair,
                    replay,
                )
            )


        pair_imgs = torch.stack(
            pair_windows
        ).to(device)

        unet_out, pair_det = (
            encode_pair_features_and_tta(
                pair_imgs
            )
        )


        # ====================================================
        # C. Process each association pair
        # ====================================================

        for b, (t, src_slot) in enumerate(
            zip(
                times,
                src_slots,
                strict=True,
            )
        ):

            c_src = _detect_cells_pooled(
                src_det[
                    b,
                    src_slot,
                ],
                t,
                replay.cfg.det_threshold,
                replay.pool_k,
            )

            c_tgt = _detect_cells_pooled(
                pair_det[
                    b,
                    1,
                ],
                t + 1,
                replay.cfg.det_threshold,
                replay.pool_k,
            )

            assert c_src.shape[1] == 4
            assert c_tgt.shape[1] == 4

            n_src = len(c_src)
            n_tgt = len(c_tgt)

            if n_src == 0 or n_tgt < 2:
                continue


            # ------------------------------------------------
            # Exact association inputs from replay_pair().
            # ------------------------------------------------

            p_src = torch.from_numpy(
                c_src[:, 1:].astype(
                    np.float32
                )
            ).unsqueeze(0).to(device)

            p_tgt = torch.from_numpy(
                c_tgt[:, 1:].astype(
                    np.float32
                )
            ).unsqueeze(0).to(device)


            window_shape = (
                replay.window,
                *replay.image_shape[1:],
            )


            src_rel = c_src.copy()
            src_rel[:, 0] = 0

            tgt_rel = c_tgt.copy()
            tgt_rel[:, 0] = 1


            pos_src = torch.from_numpy(
                extract_pos_features(
                    src_rel,
                    window_shape,
                )
            ).unsqueeze(0).to(device)

            pos_tgt = torch.from_numpy(
                extract_pos_features(
                    tgt_rel,
                    window_shape,
                )
            ).unsqueeze(0).to(device)


            mask_src = torch.ones(
                1,
                n_src,
                dtype=torch.bool,
                device=device,
            )

            mask_tgt = torch.ones(
                1,
                n_tgt,
                dtype=torch.bool,
                device=device,
            )


            feat_src = model._index_features(
                unet_out[b:b+1, 0],
                p_src,
                mask_src,
            )

            feat_tgt = model._index_features(
                unet_out[b:b+1, 1],
                p_tgt,
                mask_tgt,
            )


            # ------------------------------------------------
            # Frozen transformer forward + q/k capture.
            # ------------------------------------------------

            context_bucket.clear()

            with torch.no_grad():

                raw = model.predict_edges(
                    feat_src,
                    feat_tgt,
                    p_src * replay.ds_arr_t,
                    p_tgt * replay.ds_arr_t,
                    pos_src,
                    pos_tgt,
                    mask_src,
                    mask_tgt,
                )[0]


            assert len(context_bucket) == 2

            q = (
                context_bucket[0][0]
                .numpy()
            )

            k = (
                context_bucket[1][0]
                .numpy()
            )

            raw_np = (
                raw.detach()
                .cpu()
                .numpy()
            )

            probs_np = (
                torch.softmax(
                    raw,
                    dim=0,
                )
                .detach()
                .cpu()
                .numpy()
            )


            assert q.shape == (
                n_src,
                128,
            )

            assert k.shape == (
                n_tgt,
                128,
            )

            assert raw_np.shape == (
                n_src,
                n_tgt,
            )


            # ------------------------------------------------
            # Match tracked GT parents to predicted source nodes.
            # ------------------------------------------------

            frame = nodes[
                (
                    nodes["t"].astype(int)
                    == t
                )
                &
                nodes[
                    "node_id"
                ].isin(
                    outdeg.index
                )
            ].copy()


            if len(frame) == 0:
                continue


            gt_xyz = frame[
                ["z", "y", "x"]
            ].to_numpy(
                np.float32
            )

            pred_xyz = (
                c_src[:, 1:4]
                .astype(np.float32)
                * downsample_arr
            )


            matches = match_gt_to_pred(
                gt_xyz,
                pred_xyz,
            )


            # ------------------------------------------------
            # For each parent:
            # choose top-2 targets ONLY by raw model score.
            # ------------------------------------------------

            for gi, pi, dist_um in matches:

                node_id = int(
                    frame.iloc[
                        gi
                    ]["node_id"]
                )

                degree = int(
                    outdeg.loc[
                        node_id
                    ]
                )

                if degree not in (1, 2):
                    continue


                score_row = raw_np[pi]

                top2 = np.argsort(
                    -score_row,
                    kind="stable",
                )[:2]

                j1 = int(top2[0])
                j2 = int(top2[1])


                raw1 = float(
                    raw_np[
                        pi,
                        j1,
                    ]
                )

                raw2 = float(
                    raw_np[
                        pi,
                        j2,
                    ]
                )

                prob1 = float(
                    probs_np[
                        pi,
                        j1,
                    ]
                )

                prob2 = float(
                    probs_np[
                        pi,
                        j2,
                    ]
                )


                geom = candidate_geometry(
                    c_src[
                        pi,
                        1:4,
                    ].astype(
                        np.float32
                    ),
                    c_tgt[
                        j1,
                        1:4,
                    ].astype(
                        np.float32
                    ),
                    c_tgt[
                        j2,
                        1:4,
                    ].astype(
                        np.float32
                    ),
                )


                # Symmetric target-context summary.
                k1 = k[j1]
                k2 = k[j2]

                k_mean = (
                    k1 + k2
                ) / 2.0

                k_absdiff = np.abs(
                    k1 - k2
                )


                score_features = np.array(
                    [
                        raw1,
                        raw2,
                        raw1 - raw2,
                        prob1,
                        prob2,
                        prob1 - prob2,
                    ],
                    dtype=np.float32,
                )


                rows_1109a.append({
                    "dataset":
                        meta.dataset,

                    "fold":
                        int(meta.fold),

                    "t":
                        int(t),

                    "node_id":
                        node_id,

                    "label":
                        int(
                            degree == 2
                        ),

                    "match_distance_um":
                        float(
                            dist_um
                        ),

                    # Existing benchmark.
                    "parent_q":
                        q[pi].copy(),

                    # Joint candidate context.
                    "k_mean":
                        k_mean.copy(),

                    "k_absdiff":
                        k_absdiff.copy(),

                    # Low-dimensional model evidence.
                    "score_features":
                        score_features,

                    # [d1,d2,sep,angle,balance]
                    "geometry":
                        geom,

                    # Provenance.
                    "top1_target":
                        int(
                            c_tgt[
                                j1,
                                0
                            ]
                        ),

                    "top2_target":
                        int(
                            c_tgt[
                                j2,
                                0
                            ]
                        ),

                    "raw1":
                        raw1,

                    "raw2":
                        raw2,

                    "prob1":
                        prob1,

                    "prob2":
                        prob2,
                })


    if n_done % 25 == 0:

        print(
            f"processed "
            f"{n_done}/{len(cv6)} "
            f"datasets "
            f"({(time.time()-started)/60:.1f} min)"
        )


hook_1109a.remove()


joint_df_1109a = pd.DataFrame(
    rows_1109a
)


# ------------------------------------------------------------
# Population fidelity
# ------------------------------------------------------------

val_1109a = joint_df_1109a[
    joint_df_1109a["fold"] == 0
]


print(
    "\n========== 11.09A COMPLETE =========="
)

print(
    "elapsed      :",
    f"{(time.time()-started)/60:.1f} min",
)

print(
    "examples     :",
    len(joint_df_1109a),
)

print(
    "positive     :",
    int(
        joint_df_1109a[
            "label"
        ].sum()
    ),
)

print(
    "val examples :",
    len(val_1109a),
)

print(
    "val positive :",
    int(
        val_1109a[
            "label"
        ].sum()
    ),
)

print(
    "parent dim   :",
    len(
        joint_df_1109a.iloc[
            0
        ]["parent_q"]
    ),
)

print(
    "candidate dim:",
    len(
        joint_df_1109a.iloc[
            0
        ]["k_mean"]
    ),
)

print(
    "score dim    :",
    len(
        joint_df_1109a.iloc[
            0
        ]["score_features"]
    ),
)

print(
    "geometry dim :",
    len(
        joint_df_1109a.iloc[
            0
        ]["geometry"]
    ),
)

In [ ]:
# ============================================================
# RUN — 11.09B Controlled Joint Parent–Candidate Probe
# Purpose:
#   Test whether joint parent + top2 candidate information
#   contains more GENERALIZABLE division signal than the
#   parent transformer state alone.
#
# Model selection:
#   C chosen independently for each representation using
#   leave-one-fold-out CV on folds 1–4 only.
#   Fold0 is untouched until final evaluation.
#
# Changes files: NO
# Runs training: YES — logistic probes only, NO neural network
# Runs inference: NO
# Uses GT: NO new GT access
# Keep after running: YES
# ============================================================

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score


# ------------------------------------------------------------
# 1. Representation definitions
# ------------------------------------------------------------

def build_X(df, representation):
    if representation == "parent_only":
        return np.stack(df["parent_q"])

    if representation == "scores_geometry":
        return np.concatenate([
            np.stack(df["score_features"]),
            np.stack(df["geometry"]),
        ], axis=1)

    if representation == "parent_candidate":
        return np.concatenate([
            np.stack(df["parent_q"]),
            np.stack(df["k_mean"]),
            np.stack(df["k_absdiff"]),
        ], axis=1)

    if representation == "full_joint":
        return np.concatenate([
            np.stack(df["parent_q"]),
            np.stack(df["k_mean"]),
            np.stack(df["k_absdiff"]),
            np.stack(df["score_features"]),
            np.stack(df["geometry"]),
        ], axis=1)

    raise ValueError(representation)


REPRESENTATIONS = [
    "parent_only",
    "scores_geometry",
    "parent_candidate",
    "full_joint",
]

C_GRID = [
    0.001,
    0.003,
    0.01,
    0.03,
    0.1,
    0.3,
    1.0,
    3.0,
    10.0,
]


# ------------------------------------------------------------
# 2. Train/evaluate one logistic probe
# ------------------------------------------------------------

def fit_eval(Xtr, ytr, Xva, yva, C):
    mu = Xtr.mean(axis=0)
    sd = Xtr.std(axis=0)
    sd[sd < 1e-6] = 1.0

    Xtr_z = (Xtr - mu) / sd
    Xva_z = (Xva - mu) / sd

    clf = LogisticRegression(
        C=C,
        penalty="l2",
        class_weight="balanced",
        max_iter=5000,
        random_state=1640,
    )

    clf.fit(Xtr_z, ytr)

    p = clf.predict_proba(Xva_z)[:, 1]

    return (
        average_precision_score(yva, p),
        roc_auc_score(yva, p),
    )


# ------------------------------------------------------------
# 3. Inner CV on folds 1–4 ONLY
# ------------------------------------------------------------

train_outer = joint_df_1109a[
    joint_df_1109a["fold"] != 0
].copy()

inner_folds = sorted(
    train_outer["fold"].unique()
)

assert inner_folds == [1, 2, 3, 4], inner_folds

cv_rows = []

for rep in REPRESENTATIONS:
    X_all = build_X(train_outer, rep)
    y_all = train_outer["label"].to_numpy()
    fold_all = train_outer["fold"].to_numpy()

    for C in C_GRID:
        aps = []
        aucs = []

        for held_fold in inner_folds:
            tr_mask = fold_all != held_fold
            va_mask = fold_all == held_fold

            ap, auc = fit_eval(
                X_all[tr_mask],
                y_all[tr_mask],
                X_all[va_mask],
                y_all[va_mask],
                C,
            )

            aps.append(ap)
            aucs.append(auc)

        cv_rows.append({
            "representation": rep,
            "C": C,
            "mean_inner_AP": np.mean(aps),
            "mean_inner_AUC": np.mean(aucs),
            "sd_inner_AP": np.std(aps),
        })


cv_1109b = pd.DataFrame(cv_rows)


# ------------------------------------------------------------
# 4. Select C by mean inner AP
#    Tie-break with mean inner AUC.
# ------------------------------------------------------------

best_rows = []

for rep in REPRESENTATIONS:
    sub = cv_1109b[
        cv_1109b["representation"] == rep
    ].sort_values(
        ["mean_inner_AP", "mean_inner_AUC"],
        ascending=False,
    )

    best_rows.append(sub.iloc[0])

best_1109b = pd.DataFrame(best_rows).reset_index(drop=True)


# ------------------------------------------------------------
# 5. Final untouched Fold0 evaluation
# ------------------------------------------------------------

train_df = joint_df_1109a[
    joint_df_1109a["fold"] != 0
].copy()

val_df = joint_df_1109a[
    joint_df_1109a["fold"] == 0
].copy()

final_rows = []

for rep in REPRESENTATIONS:
    C = float(
        best_1109b.loc[
            best_1109b["representation"] == rep,
            "C",
        ].iloc[0]
    )

    Xtr = build_X(train_df, rep)
    Xva = build_X(val_df, rep)

    ytr = train_df["label"].to_numpy()
    yva = val_df["label"].to_numpy()

    mu = Xtr.mean(axis=0)
    sd = Xtr.std(axis=0)
    sd[sd < 1e-6] = 1.0

    Xtr_z = (Xtr - mu) / sd
    Xva_z = (Xva - mu) / sd

    clf = LogisticRegression(
        C=C,
        penalty="l2",
        class_weight="balanced",
        max_iter=5000,
        random_state=1640,
    )

    clf.fit(Xtr_z, ytr)

    ptr = clf.predict_proba(Xtr_z)[:, 1]
    pva = clf.predict_proba(Xva_z)[:, 1]

    val_ap = average_precision_score(yva, pva)

    final_rows.append({
        "representation": rep,
        "dim": Xtr.shape[1],
        "selected_C": C,
        "inner_AP": float(
            best_1109b.loc[
                best_1109b["representation"] == rep,
                "mean_inner_AP",
            ].iloc[0]
        ),
        "train_AP": average_precision_score(ytr, ptr),
        "val_AP": val_ap,
        "val_PR_lift": val_ap / yva.mean(),
        "train_AUC": roc_auc_score(ytr, ptr),
        "val_AUC": roc_auc_score(yva, pva),
    })


summary_1109b = pd.DataFrame(final_rows)

print("========== 11.09B COMPLETE ==========")
print("train examples :", len(train_df))
print("val examples   :", len(val_df))
print("val positive   :", int(val_df["label"].sum()))

print("\n========== SELECTED REGULARIZATION ==========")
display(
    best_1109b[
        [
            "representation",
            "C",
            "mean_inner_AP",
            "mean_inner_AUC",
            "sd_inner_AP",
        ]
    ]
)

print("\n========== JOINT REPRESENTATION PROBE ==========")
display(summary_1109b)

parent = summary_1109b[
    summary_1109b["representation"] == "parent_only"
].iloc[0]

print("\n========== DELTA VS PARENT ==========")

for _, row in summary_1109b.iterrows():
    print(
        f"{row['representation']:18s} "
        f"ΔAP={row['val_AP'] - parent['val_AP']:+.6f}  "
        f"ΔAUC={row['val_AUC'] - parent['val_AUC']:+.6f}"
    )

In [ ]:
# ============================================================
# RESULT — 11.09B Compact Result Display
# Purpose:
#   Re-display the completed 11.09B model-selection and final
#   Fold0 results without rerunning any logistic probes.
# Changes files: NO
# Runs training/inference: NO
# Uses GT: NO
# Keep after running: YES
# ============================================================

print("========== SELECTED REGULARIZATION ==========")

print(
    best_1109b[
        [
            "representation",
            "C",
            "mean_inner_AP",
            "mean_inner_AUC",
            "sd_inner_AP",
        ]
    ]
    .round(6)
    .to_string(index=False)
)

print("\n========== JOINT REPRESENTATION PROBE ==========")

print(
    summary_1109b[
        [
            "representation",
            "dim",
            "selected_C",
            "inner_AP",
            "train_AP",
            "val_AP",
            "val_PR_lift",
            "train_AUC",
            "val_AUC",
        ]
    ]
    .round(6)
    .to_string(index=False)
)

parent = summary_1109b.loc[
    summary_1109b["representation"] == "parent_only"
].iloc[0]

print("\n========== DELTA VS PARENT ==========")

for _, row in summary_1109b.iterrows():
    print(
        f"{row['representation']:18s} "
        f"ΔAP={row['val_AP'] - parent['val_AP']:+.6f}  "
        f"ΔAUC={row['val_AUC'] - parent['val_AUC']:+.6f}"
    )

In [ ]:
# ============================================================
# SETUP — 11.09C All-Frame Joint-Feature Extractor
# Purpose:
#   Define fast helpers for evaluating the frozen joint
#   representation on ALL tracked Fold0 6bba parents.
# Changes files: NO
# Runs training/inference: NO
# Uses GT: NO
# Keep after running: YES
# ============================================================

PRED_6BBA = (
    PROJECT
    / "predictions/heqiuyan/"
      "stage10_infer_baseline_256_det995/split_0"
)

VAL6 = (
    CV[
        (CV["prefix"] == "6bba")
        & (CV["fold"] == 0)
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(VAL6) == 25


def pred_cells_from_graph(pred_nodes, t):
    """Return predictor nodes as [t,z,y,x] on downsampled grid."""
    f = pred_nodes[
        pred_nodes["t"].astype(int) == int(t)
    ]

    if len(f) == 0:
        return np.empty((0, 4), dtype=np.int16)

    xyz = f[["z", "y", "x"]].to_numpy(np.float32)
    xyz_ds = xyz / downsample_arr

    # Predictor detections should lie exactly on its downsampled grid.
    assert np.max(
        np.abs(xyz_ds - np.rint(xyz_ds))
    ) < 1e-4

    return np.column_stack([
        np.full(len(f), t),
        np.rint(xyz_ds),
    ]).astype(np.int16)


def make_pair_batch(replay, times):
    windows = []

    for t in times:
        pair = torch.stack([
            _load_frame(
                replay.zarr,
                ti,
                replay.target_shape,
                replay.downsample,
            )
            for ti in (t, t + 1)
        ])

        windows.append(
            normalize_window(pair, replay)
        )

    return torch.stack(windows).to(device)

In [ ]:
# ============================================================
# RUN — 11.09C All-Frame Deployment-Prior Audit — 6bba Fold0
# Purpose:
#   Test parent-only and joint division representations under
#   the natural all-frame GT parent prevalence, rather than
#   division-frame-enriched sampling.
# Changes files: NO
# Runs training: YES — logistic probes only
# Runs inference: YES — frozen original-view UNet + transformer
# Uses GT: YES — matching / division labels only
# Keep after running: YES
# ============================================================

import time

rows = []
started = time.time()
BATCH_PAIRS = 4

context_bucket = []

hook = model.transformer.norm_out.register_forward_hook(
    lambda m, inp, out:
        context_bucket.append(out.detach().cpu())
)

for d_i, meta in enumerate(VAL6.itertuples(index=False), 1):

    gt = load_graph(TRAIN_ROOT / f"{meta.dataset}.geff")
    pred = load_graph(PRED_6BBA / f"{meta.dataset}.geff")

    nodes = gt.node_attrs(unpack=True).to_pandas()
    pred_nodes = pred.node_attrs(unpack=True).to_pandas()

    edges = gt.edge_attrs(
        attr_keys=["source_id", "target_id"],
        unpack=True,
    ).to_pandas()

    outdeg = edges.groupby("source_id").size()

    parent_nodes = nodes[
        nodes["node_id"].isin(outdeg.index)
    ]

    times = sorted(
        t for t in parent_nodes["t"].astype(int).unique()
        if t < 99
    )

    division_times = set(
        nodes.loc[
            nodes["node_id"].isin(
                outdeg[outdeg == 2].index
            ),
            "t",
        ].astype(int)
    )

    replay = DatasetReplay(
        meta.dataset,
        TRAIN_ROOT,
        model,
        window_size,
        downsample,
        device,
    )

    for start in range(0, len(times), BATCH_PAIRS):
        batch_times = times[start:start + BATCH_PAIRS]

        imgs = make_pair_batch(replay, batch_times)

        with torch.no_grad():
            unet_out, _ = model.encode(imgs)

        for b, t in enumerate(batch_times):

            c_src = pred_cells_from_graph(pred_nodes, t)
            c_tgt = pred_cells_from_graph(pred_nodes, t + 1)

            if len(c_src) == 0 or len(c_tgt) < 2:
                continue

            n_src, n_tgt = len(c_src), len(c_tgt)

            p_src = torch.from_numpy(
                c_src[:, 1:].astype(np.float32)
            ).unsqueeze(0).to(device)

            p_tgt = torch.from_numpy(
                c_tgt[:, 1:].astype(np.float32)
            ).unsqueeze(0).to(device)

            window_shape = (
                replay.window,
                *replay.image_shape[1:],
            )

            src_rel = c_src.copy()
            src_rel[:, 0] = 0

            tgt_rel = c_tgt.copy()
            tgt_rel[:, 0] = 1

            pos_src = torch.from_numpy(
                extract_pos_features(src_rel, window_shape)
            ).unsqueeze(0).to(device)

            pos_tgt = torch.from_numpy(
                extract_pos_features(tgt_rel, window_shape)
            ).unsqueeze(0).to(device)

            mask_src = torch.ones(
                1, n_src,
                dtype=torch.bool,
                device=device,
            )

            mask_tgt = torch.ones(
                1, n_tgt,
                dtype=torch.bool,
                device=device,
            )

            feat_src = model._index_features(
                unet_out[b:b+1, 0],
                p_src,
                mask_src,
            )

            feat_tgt = model._index_features(
                unet_out[b:b+1, 1],
                p_tgt,
                mask_tgt,
            )

            context_bucket.clear()

            with torch.no_grad():
                raw = model.predict_edges(
                    feat_src,
                    feat_tgt,
                    p_src * replay.ds_arr_t,
                    p_tgt * replay.ds_arr_t,
                    pos_src,
                    pos_tgt,
                    mask_src,
                    mask_tgt,
                )[0]

            assert len(context_bucket) == 2

            q = context_bucket[0][0].numpy()
            k = context_bucket[1][0].numpy()

            raw_np = raw.cpu().numpy()

            probs_np = torch.softmax(
                raw, dim=0
            ).cpu().numpy()

            gt_frame = nodes[
                (nodes["t"].astype(int) == t)
                & nodes["node_id"].isin(outdeg.index)
            ].copy()

            gt_xyz = gt_frame[
                ["z", "y", "x"]
            ].to_numpy(np.float32)

            pred_xyz = (
                c_src[:, 1:4].astype(np.float32)
                * downsample_arr
            )

            matches = match_gt_to_pred(
                gt_xyz,
                pred_xyz,
            )

            for gi, pi, dist_um in matches:

                node_id = int(
                    gt_frame.iloc[gi]["node_id"]
                )

                degree = int(outdeg.loc[node_id])

                if degree not in (1, 2):
                    continue

                top2 = np.argsort(
                    -raw_np[pi],
                    kind="stable",
                )[:2]

                j1, j2 = map(int, top2)

                raw1 = float(raw_np[pi, j1])
                raw2 = float(raw_np[pi, j2])

                prob1 = float(probs_np[pi, j1])
                prob2 = float(probs_np[pi, j2])

                geom = candidate_geometry(
                    c_src[pi, 1:4].astype(np.float32),
                    c_tgt[j1, 1:4].astype(np.float32),
                    c_tgt[j2, 1:4].astype(np.float32),
                )

                rows.append({
                    "dataset": meta.dataset,
                    "fold": 0,
                    "t": int(t),
                    "node_id": node_id,
                    "label": int(degree == 2),
                    "division_frame": int(t in division_times),
                    "match_distance_um": float(dist_um),

                    "parent_q": q[pi].copy(),
                    "k_mean": ((k[j1] + k[j2]) / 2).copy(),
                    "k_absdiff": np.abs(k[j1] - k[j2]).copy(),

                    "score_features": np.array([
                        raw1,
                        raw2,
                        raw1 - raw2,
                        prob1,
                        prob2,
                        prob1 - prob2,
                    ], dtype=np.float32),

                    "geometry": geom,

                    # Save candidates for the next daughter-recall audit.
                    "top1_xyz": (
                        c_tgt[j1, 1:4].astype(np.float32)
                        * downsample_arr
                    ),
                    "top2_xyz": (
                        c_tgt[j2, 1:4].astype(np.float32)
                        * downsample_arr
                    ),
                })

    if d_i % 5 == 0:
        print(
            f"processed {d_i}/25 datasets "
            f"({(time.time()-started)/60:.1f} min)"
        )

hook.remove()

allframe_1109c = pd.DataFrame(rows)

print("\n========== EXTRACTION ==========")
print("examples :", len(allframe_1109c))
print("positive :", int(allframe_1109c["label"].sum()))
print(
    "prevalence:",
    f"{allframe_1109c['label'].mean():.6f}"
)


# ------------------------------------------------------------
# Fidelity: division-frame subset must reproduce 11.09A
# population.
# ------------------------------------------------------------

audit_subset = allframe_1109c[
    allframe_1109c["division_frame"] == 1
]

print("\n========== FIDELITY ==========")
print("division-frame n   :", len(audit_subset))
print("division-frame pos :", int(audit_subset["label"].sum()))


# ------------------------------------------------------------
# Fit on folds1–4 enriched training set exactly as 11.09B,
# then evaluate on natural-prior all-frame Fold0.
# ------------------------------------------------------------

natural_rows = []
probabilities = {}

for rep in REPRESENTATIONS:

    C = float(
        best_1109b.loc[
            best_1109b["representation"] == rep,
            "C",
        ].iloc[0]
    )

    Xtr = build_X(joint_df_1109a, rep)
    ytr = joint_df_1109a["label"].to_numpy()

    Xva = build_X(allframe_1109c, rep)
    yva = allframe_1109c["label"].to_numpy()

    mu = Xtr.mean(0)
    sd = Xtr.std(0)
    sd[sd < 1e-6] = 1.0

    clf = LogisticRegression(
        C=C,
        class_weight="balanced",
        max_iter=5000,
        random_state=1640,
    )

    clf.fit(
        (Xtr - mu) / sd,
        ytr,
    )

    p = clf.predict_proba(
        (Xva - mu) / sd
    )[:, 1]

    probabilities[rep] = p

    ap = average_precision_score(yva, p)

    natural_rows.append({
        "representation": rep,
        "selected_C": C,
        "prevalence": yva.mean(),
        "AP": ap,
        "PR_lift": ap / yva.mean(),
        "AUC": roc_auc_score(yva, p),
    })


summary_1109c = pd.DataFrame(natural_rows)

print("\n========== NATURAL-PRIOR PROBE ==========")
print(
    summary_1109c
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# Full-joint top-K operational ranking.
# ------------------------------------------------------------

y = allframe_1109c["label"].to_numpy()
p = probabilities["full_joint"]

order = np.argsort(-p)

top_rows = []

for K in (25, 50, 100, 250, 500):
    idx = order[:K]
    tp = int(y[idx].sum())

    top_rows.append({
        "top_K": K,
        "TP": tp,
        "FP": K - tp,
        "precision": tp / K,
        "division_recall": tp / y.sum(),
    })

topk_1109c = pd.DataFrame(top_rows)

print("\n========== FULL-JOINT TOP-K ==========")
print(
    topk_1109c
    .round(6)
    .to_string(index=False)
)

print(
    "\n========== 11.09C COMPLETE =========="
)
print(
    "elapsed:",
    f"{(time.time()-started)/60:.1f} min"
)

In [ ]:
# ============================================================
# RESULT — 11.09D Frame-Relative Joint-Score Ranking Audit
# Purpose:
#   Test whether true division parents stand out relative to
#   other parents in the SAME frame, despite poor global
#   absolute-score calibration.
# Changes files: NO
# Runs training/inference: NO
# Uses GT: NO new GT access — reuses existing labels
# Keep after running: YES
# ============================================================

import numpy as np
import pandas as pd

audit_1109d = allframe_1109c.copy()

p = np.asarray(
    probabilities["full_joint"],
    dtype=np.float64,
)

assert len(p) == len(audit_1109d)

audit_1109d["joint_score"] = p


# ------------------------------------------------------------
# 1. Frame-relative statistics
# ------------------------------------------------------------

group_cols = [
    "dataset",
    "t",
]

g = audit_1109d.groupby(
    group_cols,
    sort=False,
)["joint_score"]


# rank=1 means highest-scoring parent in the frame.
audit_1109d["frame_rank"] = g.rank(
    method="first",
    ascending=False,
)

audit_1109d["frame_n"] = g.transform(
    "size"
)


# Percentile:
# 1.0 = highest score in frame
# 0.0 ≈ lowest score in frame
audit_1109d["frame_percentile"] = (
    1.0
    - (
        audit_1109d["frame_rank"] - 1.0
    )
    / np.maximum(
        audit_1109d["frame_n"] - 1.0,
        1.0,
    )
)


frame_mean = g.transform("mean")
frame_std = g.transform("std").fillna(0.0)

audit_1109d["frame_z"] = (
    audit_1109d["joint_score"]
    - frame_mean
) / np.maximum(
    frame_std,
    1e-8,
)


# ------------------------------------------------------------
# 2. Dataset-relative percentile as a secondary check
# ------------------------------------------------------------

gd = audit_1109d.groupby(
    "dataset",
    sort=False,
)["joint_score"]

audit_1109d["dataset_rank"] = gd.rank(
    method="first",
    ascending=False,
)

audit_1109d["dataset_n"] = gd.transform(
    "size"
)

audit_1109d["dataset_percentile"] = (
    1.0
    - (
        audit_1109d["dataset_rank"] - 1.0
    )
    / np.maximum(
        audit_1109d["dataset_n"] - 1.0,
        1.0,
    )
)


# ------------------------------------------------------------
# 3. Metrics
# ------------------------------------------------------------

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

y = audit_1109d["label"].to_numpy()

score_defs = {
    "global_raw":
        audit_1109d["joint_score"].to_numpy(),

    "frame_percentile":
        audit_1109d["frame_percentile"].to_numpy(),

    "frame_z":
        audit_1109d["frame_z"].to_numpy(),

    "dataset_percentile":
        audit_1109d["dataset_percentile"].to_numpy(),
}

metric_rows = []

for name, score in score_defs.items():

    ap = average_precision_score(
        y,
        score,
    )

    metric_rows.append({
        "score": name,
        "AP": ap,
        "PR_lift": ap / y.mean(),
        "AUC": roc_auc_score(
            y,
            score,
        ),
    })

summary_1109d = pd.DataFrame(
    metric_rows
)


# ------------------------------------------------------------
# 4. Exact local ranks of the 25 divisions
# ------------------------------------------------------------

positives_1109d = (
    audit_1109d[
        audit_1109d["label"] == 1
    ][
        [
            "dataset",
            "t",
            "joint_score",
            "frame_rank",
            "frame_n",
            "frame_percentile",
            "frame_z",
        ]
    ]
    .sort_values(
        [
            "frame_rank",
            "joint_score",
        ],
        ascending=[
            True,
            False,
        ],
    )
)


rank_rows = []

for k in [
    1,
    2,
    3,
    5,
    10,
]:

    count = int(
        (
            positives_1109d[
                "frame_rank"
            ] <= k
        ).sum()
    )

    rank_rows.append({
        "within_frame_top_k": k,
        "division_count": count,
        "division_recall": (
            count
            / len(positives_1109d)
        ),
    })

frame_rank_recall_1109d = pd.DataFrame(
    rank_rows
)


# ------------------------------------------------------------
# 5. Global Top-K after frame normalization
# ------------------------------------------------------------

topk_rows = []

for score_name in [
    "frame_percentile",
    "frame_z",
]:

    score = audit_1109d[
        score_name
    ].to_numpy()

    order = np.argsort(
        -score,
        kind="stable",
    )

    for K in [
        25,
        50,
        100,
        250,
        500,
    ]:

        idx = order[:K]

        tp = int(
            y[idx].sum()
        )

        topk_rows.append({
            "score": score_name,
            "top_K": K,
            "TP": tp,
            "FP": K - tp,
            "precision": tp / K,
            "division_recall": (
                tp / y.sum()
            ),
        })

topk_1109d = pd.DataFrame(
    topk_rows
)


# ------------------------------------------------------------
# 6. Output
# ------------------------------------------------------------

print(
    "========== 11.09D FRAME-RELATIVE METRICS =========="
)

print(
    summary_1109d
    .round(6)
    .to_string(index=False)
)


print(
    "\n========== TRUE-DIVISION FRAME RANKS =========="
)

print(
    frame_rank_recall_1109d
    .round(6)
    .to_string(index=False)
)


print(
    "\n========== DIVISION RANK QUANTILES =========="
)

print(
    positives_1109d[
        "frame_rank"
    ]
    .quantile(
        [
            0.0,
            0.25,
            0.5,
            0.75,
            1.0,
        ]
    )
    .to_string()
)


print(
    "\n========== NORMALIZED GLOBAL TOP-K =========="
)

print(
    topk_1109d
    .round(6)
    .to_string(index=False)
)

In [ ]:
# ============================================================
# RESULT — 11.09E Division-Frame Event Detection Audit
# Purpose:
#   Determine whether the remaining bottleneck is identifying
#   WHEN a division occurs, after showing that the correct
#   parent is usually top-ranked within true division frames.
#
# Tests frame-level signals derived from existing joint scores:
#   - maximum parent score
#   - top1 - top2 margin
#   - top1 relative to frame score distribution
#
# Changes files: NO
# Runs training/inference: NO
# Uses GT: NO new GT access — reuses existing labels
# Keep after running: YES
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


# ------------------------------------------------------------
# 1. Attach all four existing probe scores
# ------------------------------------------------------------

frame_parent_df = allframe_1109c.copy()

for rep, p in probabilities.items():
    assert len(p) == len(frame_parent_df)
    frame_parent_df[f"score_{rep}"] = np.asarray(
        p,
        dtype=np.float64,
    )


# ------------------------------------------------------------
# 2. Aggregate parent predictions to frame-level features
# ------------------------------------------------------------

frame_rows = []

for (dataset, t), g in frame_parent_df.groupby(
    ["dataset", "t"],
    sort=False,
):

    row = {
        "dataset": dataset,
        "t": int(t),

        # A frame is positive if at least one tracked parent
        # undergoes a GT division.
        "label": int(g["label"].max()),

        "n_parents": len(g),
    }

    for rep in REPRESENTATIONS:

        s = np.sort(
            g[f"score_{rep}"].to_numpy()
        )[::-1]

        top1 = float(s[0])
        top2 = float(s[1]) if len(s) >= 2 else float(s[0])

        row[f"{rep}_max"] = top1
        row[f"{rep}_margin"] = top1 - top2

        row[f"{rep}_mean"] = float(
            s.mean()
        )

        row[f"{rep}_std"] = float(
            s.std()
        )

        row[f"{rep}_top3mean"] = float(
            s[:min(3, len(s))].mean()
        )

        row[f"{rep}_max_z"] = (
            (top1 - s.mean())
            / max(s.std(), 1e-8)
        )

    frame_rows.append(row)


frame_df_1109e = pd.DataFrame(frame_rows)

y_frame = frame_df_1109e[
    "label"
].to_numpy()


print(
    "========== FRAME POPULATION =========="
)

print(
    "frames          :",
    len(frame_df_1109e),
)

print(
    "division frames :",
    int(y_frame.sum()),
)

print(
    "frame prevalence:",
    f"{y_frame.mean():.6f}",
)


# ------------------------------------------------------------
# 3. Pre-declared frame-level signals
# ------------------------------------------------------------

SCORE_DEFS = {
    # Does one parent simply get a very high score?
    "parent_only_max":
        frame_df_1109e["parent_only_max"].to_numpy(),

    "scores_geometry_max":
        frame_df_1109e["scores_geometry_max"].to_numpy(),

    "parent_candidate_max":
        frame_df_1109e["parent_candidate_max"].to_numpy(),

    "full_joint_max":
        frame_df_1109e["full_joint_max"].to_numpy(),

    # Is there one parent standing out from the rest?
    "full_joint_margin":
        frame_df_1109e["full_joint_margin"].to_numpy(),

    # Does the frame contain several elevated candidates?
    "full_joint_top3mean":
        frame_df_1109e["full_joint_top3mean"].to_numpy(),

    # Is the strongest parent unusually strong relative
    # to its own frame distribution?
    "full_joint_max_z":
        frame_df_1109e["full_joint_max_z"].to_numpy(),
}


# ------------------------------------------------------------
# 4. Frame-level AP / AUC
# ------------------------------------------------------------

metric_rows = []

for name, score in SCORE_DEFS.items():

    ap = average_precision_score(
        y_frame,
        score,
    )

    metric_rows.append({
        "frame_score": name,
        "AP": ap,
        "PR_lift": (
            ap / y_frame.mean()
        ),
        "AUC": roc_auc_score(
            y_frame,
            score,
        ),
    })


summary_1109e = (
    pd.DataFrame(metric_rows)
    .sort_values(
        "AP",
        ascending=False,
    )
    .reset_index(drop=True)
)


print(
    "\n========== FRAME EVENT METRICS =========="
)

print(
    summary_1109e
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 5. Top-K frame retrieval for each promising full-joint score
# ------------------------------------------------------------

topk_rows = []

for score_name in [
    "full_joint_max",
    "full_joint_margin",
    "full_joint_top3mean",
    "full_joint_max_z",
]:

    score = SCORE_DEFS[
        score_name
    ]

    order = np.argsort(
        -score,
        kind="stable",
    )

    for K in [
        25,
        50,
        100,
        250,
        500,
    ]:

        K_eff = min(
            K,
            len(order),
        )

        idx = order[:K_eff]

        tp = int(
            y_frame[idx].sum()
        )

        topk_rows.append({
            "frame_score": score_name,
            "top_K": K_eff,
            "division_frames": tp,
            "false_frames": K_eff - tp,
            "precision": (
                tp / K_eff
            ),
            "event_recall": (
                tp / y_frame.sum()
            ),
        })


topk_1109e = pd.DataFrame(
    topk_rows
)


print(
    "\n========== FRAME TOP-K =========="
)

print(
    topk_1109e
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 6. End-to-end decomposition:
#
# If we selected the correct frame, how often would choosing
# that frame's highest-scoring parent recover the real parent?
# ------------------------------------------------------------

parent_rank = (
    frame_parent_df
    .groupby(
        ["dataset", "t"],
        sort=False,
    )["score_full_joint"]
    .rank(
        method="first",
        ascending=False,
    )
)

frame_parent_df[
    "full_joint_frame_rank"
] = parent_rank


frame_outcomes = []

for (dataset, t), g in frame_parent_df.groupby(
    ["dataset", "t"],
    sort=False,
):

    positives = g[
        g["label"] == 1
    ]

    if len(positives) == 0:
        correct_top1 = 0
        correct_top5 = 0
    else:
        ranks = positives[
            "full_joint_frame_rank"
        ].to_numpy()

        correct_top1 = int(
            np.any(ranks <= 1)
        )

        correct_top5 = int(
            np.any(ranks <= 5)
        )

    frame_outcomes.append({
        "dataset": dataset,
        "t": int(t),
        "correct_parent_top1":
            correct_top1,
        "correct_parent_top5":
            correct_top5,
    })


frame_outcomes = pd.DataFrame(
    frame_outcomes
)


frame_eval = frame_df_1109e.merge(
    frame_outcomes,
    on=[
        "dataset",
        "t",
    ],
    how="left",
)


# ------------------------------------------------------------
# Use best *predeclared* full-joint frame statistic only for
# descriptive decomposition.
# ------------------------------------------------------------

candidate_frame_scores = [
    "full_joint_max",
    "full_joint_margin",
    "full_joint_top3mean",
    "full_joint_max_z",
]

best_frame_score = (
    summary_1109e[
        summary_1109e[
            "frame_score"
        ].isin(
            candidate_frame_scores
        )
    ]
    .iloc[0][
        "frame_score"
    ]
)

print(
    "\n========== BEST DESCRIPTIVE FRAME SIGNAL =========="
)

print(
    best_frame_score
)


score = frame_eval[
    best_frame_score
].to_numpy()

order = np.argsort(
    -score,
    kind="stable",
)


end_rows = []

for K in [
    25,
    50,
    100,
    250,
    500,
]:

    K_eff = min(
        K,
        len(order),
    )

    selected = frame_eval.iloc[
        order[:K_eff]
    ]

    true_frames = selected[
        selected["label"] == 1
    ]

    n_true_frames = len(
        true_frames
    )

    parent1 = int(
        true_frames[
            "correct_parent_top1"
        ].sum()
    )

    parent5 = int(
        true_frames[
            "correct_parent_top5"
        ].sum()
    )

    end_rows.append({
        "top_K_frames":
            K_eff,

        "true_division_frames":
            n_true_frames,

        "event_recall":
            n_true_frames
            / y_frame.sum(),

        "correct_parent_top1":
            parent1,

        "end_to_end_top1_recall":
            parent1
            / y_frame.sum(),

        "correct_parent_top5":
            parent5,

        "end_to_end_top5_recall":
            parent5
            / y_frame.sum(),
    })


end_to_end_1109e = pd.DataFrame(
    end_rows
)


print(
    "\n========== TWO-STAGE DECOMPOSITION =========="
)

print(
    end_to_end_1109e
    .round(6)
    .to_string(index=False)
)

In [ ]:
# ============================================================
# RESULT — 11.10A Temporal Division-Event Signal Audit
# Purpose:
#   Test whether division frames are transient temporal peaks
#   in full-joint score, rather than simply high absolute-score
#   frames.
# Changes files: NO
# Runs training/inference: NO
# Uses GT: NO new GT access — reuses existing frame labels
# Keep after running: YES
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

df = (
    frame_df_1109e[
        ["dataset", "t", "label", "full_joint_max"]
    ]
    .copy()
    .sort_values(["dataset", "t"])
    .reset_index(drop=True)
)

g = df.groupby("dataset", sort=False)

# ------------------------------------------------------------
# Neighbor scores, respecting real consecutive frame numbers.
# ------------------------------------------------------------

for k in (1, 2):
    prev_s = g["full_joint_max"].shift(k)
    next_s = g["full_joint_max"].shift(-k)

    prev_t = g["t"].shift(k)
    next_t = g["t"].shift(-k)

    df[f"prev{k}"] = prev_s.where(
        df["t"] - prev_t == k
    )
    df[f"next{k}"] = next_s.where(
        next_t - df["t"] == k
    )

s = df["full_joint_max"]

# Require both ±1 neighbors for the primary audit.
valid1 = df["prev1"].notna() & df["next1"].notna()

df["rise_prev1"] = s - df["prev1"]
df["drop_next1"] = s - df["next1"]

df["contrast1"] = (
    s - 0.5 * (df["prev1"] + df["next1"])
)

df["peak1"] = (
    s - np.maximum(df["prev1"], df["next1"])
)

# Wider ±2 contrast when all four neighbors exist.
valid2 = (
    valid1
    & df["prev2"].notna()
    & df["next2"].notna()
)

df["contrast2"] = (
    s
    - (
        df["prev2"]
        + df["prev1"]
        + df["next1"]
        + df["next2"]
    ) / 4.0
)


def evaluate(name, score, mask):
    y = df.loc[mask, "label"].to_numpy()
    x = df.loc[mask, score].to_numpy()

    ap = average_precision_score(y, x)

    return {
        "signal": name,
        "n": len(y),
        "positive": int(y.sum()),
        "prevalence": float(y.mean()),
        "AP": ap,
        "PR_lift": ap / y.mean(),
        "AUC": roc_auc_score(y, x),
    }


rows = [
    evaluate(
        "absolute_score",
        "full_joint_max",
        valid1,
    ),
    evaluate(
        "rise_prev1",
        "rise_prev1",
        valid1,
    ),
    evaluate(
        "drop_next1",
        "drop_next1",
        valid1,
    ),
    evaluate(
        "contrast1",
        "contrast1",
        valid1,
    ),
    evaluate(
        "peak1",
        "peak1",
        valid1,
    ),
    evaluate(
        "contrast2",
        "contrast2",
        valid2,
    ),
]

summary_1110a = pd.DataFrame(rows)

print("========== 11.10A TEMPORAL SIGNAL METRICS ==========")
print(
    summary_1110a
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# Top-K retrieval for the temporal signals.
# ------------------------------------------------------------

topk_rows = []

for signal, mask in [
    ("absolute_score", valid1),
    ("contrast1", valid1),
    ("peak1", valid1),
    ("contrast2", valid2),
]:
    score_col = (
        "full_joint_max"
        if signal == "absolute_score"
        else signal
    )

    sub = df.loc[mask].copy()
    y = sub["label"].to_numpy()
    score = sub[score_col].to_numpy()

    order = np.argsort(-score, kind="stable")

    for K in (25, 50, 100, 250, 500):
        K_eff = min(K, len(order))
        idx = order[:K_eff]
        tp = int(y[idx].sum())

        topk_rows.append({
            "signal": signal,
            "top_K": K_eff,
            "TP": tp,
            "FP": K_eff - tp,
            "precision": tp / K_eff,
            "event_recall": tp / y.sum(),
        })

topk_1110a = pd.DataFrame(topk_rows)

print("\n========== TEMPORAL TOP-K ==========")
print(
    topk_1110a
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# Positive-event trajectory around t = division frame.
# ------------------------------------------------------------

pos = df[df["label"] == 1]

trajectory = pd.DataFrame({
    "offset": [-2, -1, 0, 1, 2],
    "mean_score": [
        pos["prev2"].mean(),
        pos["prev1"].mean(),
        pos["full_joint_max"].mean(),
        pos["next1"].mean(),
        pos["next2"].mean(),
    ],
    "median_score": [
        pos["prev2"].median(),
        pos["prev1"].median(),
        pos["full_joint_max"].median(),
        pos["next1"].median(),
        pos["next2"].median(),
    ],
})

print("\n========== DIVISION EVENT TRAJECTORY ==========")
print(
    trajectory
    .round(6)
    .to_string(index=False)
)

In [ ]:
# ============================================================
# RESULT — 11.10B Same-Lineage Temporal Score Audit
# Purpose:
#   Test whether the SAME cell's joint division score rises
#   before division, rather than comparing unrelated frame
#   winners across time.
#
# This is a teacher-forced CAPACITY AUDIT:
#   GT lineage is used ONLY to identify the previous same-cell
#   node. It is NOT a deployable inference rule.
#
# Changes files: NO
# Runs training/inference: NO
# Uses GT: YES — lineage alignment only
# Keep after running: YES
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


# ------------------------------------------------------------
# 1. Attach frozen full-joint score
# ------------------------------------------------------------

track_df = allframe_1109c[
    [
        "dataset",
        "t",
        "node_id",
        "label",
    ]
].copy()

p = np.asarray(
    probabilities["full_joint"],
    dtype=np.float64,
)

assert len(p) == len(track_df)

# Work in log-odds space so temporal differences are not
# distorted by probability saturation near 0 or 1.
eps = 1e-6
p_clip = np.clip(p, eps, 1.0 - eps)

track_df["score"] = p
track_df["logit"] = np.log(
    p_clip / (1.0 - p_clip)
)


# ------------------------------------------------------------
# 2. Recover GT predecessor links
# ------------------------------------------------------------

pred_rows = []

for dataset in track_df["dataset"].unique():

    gt = load_graph(
        TRAIN_ROOT / f"{dataset}.geff"
    )

    edges = (
        gt.edge_attrs(
            attr_keys=[
                "source_id",
                "target_id",
            ],
            unpack=True,
        )
        .to_pandas()
    )

    nodes = (
        gt.node_attrs(unpack=True)
        .to_pandas()
        .set_index("node_id")
    )

    for e in edges.itertuples(index=False):

        src = int(e.source_id)
        tgt = int(e.target_id)

        if (
            src not in nodes.index
            or tgt not in nodes.index
        ):
            continue

        t_src = int(
            nodes.loc[src, "t"]
        )

        t_tgt = int(
            nodes.loc[tgt, "t"]
        )

        if t_tgt - t_src != 1:
            continue

        pred_rows.append({
            "dataset": dataset,
            "node_id": tgt,
            "prev_node_id": src,
        })


pred_map = pd.DataFrame(pred_rows)

# Forest structure should give at most one predecessor.
assert not pred_map.duplicated(
    ["dataset", "node_id"]
).any()


# ------------------------------------------------------------
# 3. Join current node to previous same-lineage node
# ------------------------------------------------------------

current = track_df.merge(
    pred_map,
    on=[
        "dataset",
        "node_id",
    ],
    how="left",
)


prev_lookup = track_df[
    [
        "dataset",
        "node_id",
        "t",
        "score",
        "logit",
    ]
].rename(
    columns={
        "node_id": "prev_node_id",
        "t": "prev_t",
        "score": "prev_score",
        "logit": "prev_logit",
    }
)


aligned = current.merge(
    prev_lookup,
    on=[
        "dataset",
        "prev_node_id",
    ],
    how="left",
)


# Require exact consecutive predecessor.
valid1 = (
    aligned["prev_logit"].notna()
    &
    (
        aligned["t"]
        - aligned["prev_t"]
        == 1
    )
)

aligned1 = aligned[
    valid1
].copy()


# ------------------------------------------------------------
# 4. One-frame same-cell temporal change
# ------------------------------------------------------------

aligned1["track_delta1"] = (
    aligned1["logit"]
    - aligned1["prev_logit"]
)


# ------------------------------------------------------------
# 5. Recover second predecessor
# ------------------------------------------------------------

pred_map2 = pred_map.rename(
    columns={
        "node_id":
            "prev_node_id",

        "prev_node_id":
            "prev2_node_id",
    }
)


aligned2 = aligned1.merge(
    pred_map2,
    on=[
        "dataset",
        "prev_node_id",
    ],
    how="left",
)


prev2_lookup = track_df[
    [
        "dataset",
        "node_id",
        "t",
        "logit",
    ]
].rename(
    columns={
        "node_id":
            "prev2_node_id",

        "t":
            "prev2_t",

        "logit":
            "prev2_logit",
    }
)


aligned2 = aligned2.merge(
    prev2_lookup,
    on=[
        "dataset",
        "prev2_node_id",
    ],
    how="left",
)


valid2 = (
    aligned2["prev2_logit"].notna()
    &
    (
        aligned2["t"]
        - aligned2["prev2_t"]
        == 2
    )
)

aligned2 = aligned2[
    valid2
].copy()


aligned2["track_delta2avg"] = (
    aligned2["logit"]
    - 0.5 * (
        aligned2["prev_logit"]
        + aligned2["prev2_logit"]
    )
)


# ------------------------------------------------------------
# 6. Fair paired evaluation
# ------------------------------------------------------------

def metric_row(
    name,
    df,
    column,
):

    y = df[
        "label"
    ].to_numpy()

    score = df[
        column
    ].to_numpy()

    ap = average_precision_score(
        y,
        score,
    )

    return {
        "signal": name,
        "n": len(df),
        "positive": int(y.sum()),
        "prevalence": float(
            y.mean()
        ),
        "AP": ap,
        "PR_lift": (
            ap / y.mean()
        ),
        "AUC": roc_auc_score(
            y,
            score,
        ),
    }


rows = [
    # Same exact population for absolute vs delta1.
    metric_row(
        "absolute_logit_1",
        aligned1,
        "logit",
    ),

    metric_row(
        "same_cell_delta1",
        aligned1,
        "track_delta1",
    ),

    # Same exact population for absolute vs delta2.
    metric_row(
        "absolute_logit_2",
        aligned2,
        "logit",
    ),

    metric_row(
        "same_cell_delta2avg",
        aligned2,
        "track_delta2avg",
    ),
]


summary_1110b = pd.DataFrame(
    rows
)


print(
    "========== 11.10B SAME-LINEAGE METRICS =========="
)

print(
    summary_1110b
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 7. Top-K on the one-frame aligned population
# ------------------------------------------------------------

topk_rows = []

for name, col in [
    (
        "absolute_logit",
        "logit",
    ),
    (
        "same_cell_delta1",
        "track_delta1",
    ),
]:

    y = aligned1[
        "label"
    ].to_numpy()

    score = aligned1[
        col
    ].to_numpy()

    order = np.argsort(
        -score,
        kind="stable",
    )

    for K in (
        25,
        50,
        100,
        250,
        500,
    ):

        K_eff = min(
            K,
            len(order),
        )

        idx = order[:K_eff]

        tp = int(
            y[idx].sum()
        )

        topk_rows.append({
            "signal": name,
            "top_K": K_eff,
            "TP": tp,
            "FP": K_eff - tp,
            "precision":
                tp / K_eff,
            "division_recall":
                tp / y.sum(),
        })


topk_1110b = pd.DataFrame(
    topk_rows
)


print(
    "\n========== SAME-LINEAGE TOP-K =========="
)

print(
    topk_1110b
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 8. Division-parent trajectory summary
# ------------------------------------------------------------

div1 = aligned1[
    aligned1["label"] == 1
]

trajectory_1110b = pd.DataFrame({
    "quantity": [
        "previous_same_cell",
        "division_parent",
        "delta",
    ],

    "mean": [
        div1[
            "prev_score"
        ].mean(),

        div1[
            "score"
        ].mean(),

        div1[
            "track_delta1"
        ].mean(),
    ],

    "median": [
        div1[
            "prev_score"
        ].median(),

        div1[
            "score"
        ].median(),

        div1[
            "track_delta1"
        ].median(),
    ],
})


print(
    "\n========== DIVISION SAME-CELL TRAJECTORY =========="
)

print(
    trajectory_1110b
    .round(6)
    .to_string(index=False)
)


print(
    "\n========== COVERAGE =========="
)

print(
    "all examples        :",
    len(track_df),
)

print(
    "1-step aligned      :",
    len(aligned1),
)

print(
    "1-step divisions    :",
    int(
        aligned1[
            "label"
        ].sum()
    ),
)

print(
    "2-step aligned      :",
    len(aligned2),
)

print(
    "2-step divisions    :",
    int(
        aligned2[
            "label"
        ].sum()
    ),
)

In [ ]:
# ============================================================
# RESULT — 11.10C Same-Lineage Representation-Change Audit
# Purpose:
#   Test whether the frozen parent/candidate representation
#   changes abnormally before division even though scalar
#   probability differences were not discriminative.
# Changes files: NO
# Runs training/inference: NO
# Uses GT: YES — reuses GT predecessor alignment from 11.10B
# Keep after running: YES
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score


# ------------------------------------------------------------
# 1. Current-node table
# ------------------------------------------------------------

cur = allframe_1109c.reset_index(drop=True).copy()

cur["full_joint_score"] = np.asarray(
    probabilities["full_joint"],
    dtype=np.float64,
)

feature_cols = [
    "parent_q",
    "k_mean",
    "k_absdiff",
    "score_features",
    "geometry",
]


# ------------------------------------------------------------
# 2. Attach GT predecessor and its frozen representation
# ------------------------------------------------------------

cur = cur.merge(
    pred_map,
    on=["dataset", "node_id"],
    how="left",
)

prev = allframe_1109c.reset_index(drop=True).copy()

prev["full_joint_score"] = np.asarray(
    probabilities["full_joint"],
    dtype=np.float64,
)

rename = {
    "node_id": "prev_node_id",
    "t": "prev_t",
    "full_joint_score": "prev_full_joint_score",
}

for c in feature_cols:
    rename[c] = f"prev_{c}"

prev = prev[
    ["dataset", "node_id", "t", "full_joint_score"]
    + feature_cols
].rename(columns=rename)

aligned_1110c = cur.merge(
    prev,
    on=["dataset", "prev_node_id"],
    how="left",
)

aligned_1110c = aligned_1110c[
    aligned_1110c["prev_t"].notna()
    & (
        aligned_1110c["t"]
        - aligned_1110c["prev_t"]
        == 1
    )
].copy()

assert int(aligned_1110c["label"].sum()) == 24


# ------------------------------------------------------------
# 3. Unsupervised standardized representation-change magnitude
# ------------------------------------------------------------

def standardized_change(df, cols):
    current = np.concatenate(
        [np.stack(df[c]) for c in cols],
        axis=1,
    )

    previous = np.concatenate(
        [np.stack(df[f"prev_{c}"]) for c in cols],
        axis=1,
    )

    # No labels used: scale dimensions only to make units comparable.
    scale = np.std(
        np.concatenate([current, previous], axis=0),
        axis=0,
    )

    scale[scale < 1e-6] = 1.0

    delta = (current - previous) / scale

    return np.sqrt(
        np.mean(delta ** 2, axis=1)
    )


aligned_1110c["parent_change"] = standardized_change(
    aligned_1110c,
    ["parent_q"],
)

aligned_1110c["candidate_change"] = standardized_change(
    aligned_1110c,
    ["k_mean", "k_absdiff"],
)

aligned_1110c["score_geom_change"] = standardized_change(
    aligned_1110c,
    ["score_features", "geometry"],
)

aligned_1110c["full_rep_change"] = standardized_change(
    aligned_1110c,
    feature_cols,
)


# ------------------------------------------------------------
# 4. Cosine change in parent / candidate context
# ------------------------------------------------------------

def cosine_change(df, col):
    a = np.stack(df[col])
    b = np.stack(df[f"prev_{col}"])

    denom = (
        np.linalg.norm(a, axis=1)
        * np.linalg.norm(b, axis=1)
        + 1e-8
    )

    return 1.0 - (
        np.sum(a * b, axis=1) / denom
    )


aligned_1110c["parent_cos_change"] = cosine_change(
    aligned_1110c,
    "parent_q",
)

aligned_1110c["candidate_cos_change"] = cosine_change(
    aligned_1110c,
    "k_mean",
)


# ------------------------------------------------------------
# 5. Evaluate
# ------------------------------------------------------------

y = aligned_1110c["label"].to_numpy()

signals = {
    "absolute_score":
        aligned_1110c["full_joint_score"].to_numpy(),

    "parent_change":
        aligned_1110c["parent_change"].to_numpy(),

    "candidate_change":
        aligned_1110c["candidate_change"].to_numpy(),

    "score_geom_change":
        aligned_1110c["score_geom_change"].to_numpy(),

    "full_rep_change":
        aligned_1110c["full_rep_change"].to_numpy(),

    "parent_cos_change":
        aligned_1110c["parent_cos_change"].to_numpy(),

    "candidate_cos_change":
        aligned_1110c["candidate_cos_change"].to_numpy(),
}

rows = []

for name, score in signals.items():
    ap = average_precision_score(y, score)

    rows.append({
        "signal": name,
        "AP": ap,
        "PR_lift": ap / y.mean(),
        "AUC": roc_auc_score(y, score),
    })

summary_1110c = (
    pd.DataFrame(rows)
    .sort_values("AP", ascending=False)
    .reset_index(drop=True)
)

print("========== 11.10C REPRESENTATION CHANGE ==========")

print(
    summary_1110c
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 6. Top-K operational comparison
# ------------------------------------------------------------

topk_rows = []

for name in [
    "absolute_score",
    "parent_change",
    "candidate_change",
    "full_rep_change",
]:

    score = signals[name]
    order = np.argsort(-score, kind="stable")

    for K in (25, 50, 100, 250, 500):
        idx = order[:K]
        tp = int(y[idx].sum())

        topk_rows.append({
            "signal": name,
            "top_K": K,
            "TP": tp,
            "FP": K - tp,
            "precision": tp / K,
            "division_recall": tp / y.sum(),
        })

topk_1110c = pd.DataFrame(topk_rows)

print("\n========== REPRESENTATION-CHANGE TOP-K ==========")

print(
    topk_1110c
    .round(6)
    .to_string(index=False)
)

### 11.10 — Temporal Post-processing Audit

Temporal post-processing of the frozen pairwise association representation was rejected.

Although true division parents showed higher full-joint scores at the division frame,
simple temporal transformations consistently reduced discrimination:

- Absolute full-joint score: AP 0.027969, AUC 0.963205
- Same-cell score delta: AP 0.013651, AUC 0.814433
- Same-cell 2-frame delta: AP 0.018719, AUC 0.892980
- Full representation change magnitude: AP 0.001950, AUC 0.675196
- Parent/candidate cosine or norm changes were similarly weak.

The frozen t→t+1 representation therefore contains strong parent-identity information
but does not expose division onset through simple temporal score or representation changes.

Decision:
FROZEN_TEMPORAL_POSTPROCESSING_REJECTED.

Next direction:
learn an explicit multi-frame division-event representation rather than deriving temporal
event signals post hoc from the existing pairwise association state.

In [ ]:
# ============================================================
# PATCH — 11.09C Remove Fold0 Probe-Training Leakage
# Purpose:
#   Refit all natural-prior probes using ONLY folds 1–4.
#   Fold0 remains completely untouched until evaluation.
#
# Fixes:
#   Previous 11.09C accidentally fit the logistic probe on
#   joint_df_1109a INCLUDING Fold0 before evaluating Fold0.
#
# Changes files: NO
# Runs training: YES — logistic regression only
# Runs inference: NO
# Uses GT: NO new GT access
# Keep after running: NO — delete after corrected results are recorded
# ============================================================

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


# ------------------------------------------------------------
# 1. Strict outer-train / outer-val separation
# ------------------------------------------------------------

probe_train_1109c = joint_df_1109a[
    joint_df_1109a["fold"] != 0
].copy()

probe_val_1109c = allframe_1109c.copy()

assert set(
    probe_train_1109c["fold"].unique()
) == {1, 2, 3, 4}

assert set(
    probe_val_1109c["fold"].unique()
) == {0}

assert len(probe_train_1109c) == 1037
assert len(probe_val_1109c) == 20635

assert int(
    probe_val_1109c["label"].sum()
) == 25


# ------------------------------------------------------------
# 2. Refit each representation using folds 1–4 ONLY
# ------------------------------------------------------------

natural_rows = []
probabilities = {}

for rep in REPRESENTATIONS:

    selected_C = float(
        best_1109b.loc[
            best_1109b["representation"] == rep,
            "C",
        ].iloc[0]
    )

    Xtr = build_X(
        probe_train_1109c,
        rep,
    )

    ytr = (
        probe_train_1109c[
            "label"
        ].to_numpy()
    )

    Xva = build_X(
        probe_val_1109c,
        rep,
    )

    yva = (
        probe_val_1109c[
            "label"
        ].to_numpy()
    )

    # Train-fold-only standardization.
    mu = Xtr.mean(axis=0)

    sd = Xtr.std(axis=0)
    sd[sd < 1e-6] = 1.0

    Xtr_z = (Xtr - mu) / sd
    Xva_z = (Xva - mu) / sd

    clf = LogisticRegression(
        C=selected_C,
        class_weight="balanced",
        max_iter=5000,
        random_state=1640,
    )

    clf.fit(
        Xtr_z,
        ytr,
    )

    pva = clf.predict_proba(
        Xva_z
    )[:, 1]

    probabilities[rep] = pva

    ap = average_precision_score(
        yva,
        pva,
    )

    natural_rows.append({
        "representation":
            rep,

        "selected_C":
            selected_C,

        "prevalence":
            float(yva.mean()),

        "AP":
            ap,

        "PR_lift":
            ap / yva.mean(),

        "AUC":
            roc_auc_score(
                yva,
                pva,
            ),
    })


summary_1109c = pd.DataFrame(
    natural_rows
)


# ------------------------------------------------------------
# 3. Recompute full-joint global Top-K
# ------------------------------------------------------------

y = probe_val_1109c[
    "label"
].to_numpy()

p = probabilities[
    "full_joint"
]

order = np.argsort(
    -p,
    kind="stable",
)

top_rows = []

for K in (
    25,
    50,
    100,
    250,
    500,
):

    idx = order[:K]

    tp = int(
        y[idx].sum()
    )

    top_rows.append({
        "top_K":
            K,

        "TP":
            tp,

        "FP":
            K - tp,

        "precision":
            tp / K,

        "division_recall":
            tp / y.sum(),
    })


topk_1109c = pd.DataFrame(
    top_rows
)


# ------------------------------------------------------------
# 4. Corrected output
# ------------------------------------------------------------

print(
    "========== 11.09C CLEAN SPLIT =========="
)

print(
    "probe train examples :",
    len(probe_train_1109c),
)

print(
    "probe train folds    :",
    sorted(
        probe_train_1109c[
            "fold"
        ].unique()
    ),
)

print(
    "Fold0 eval examples  :",
    len(probe_val_1109c),
)

print(
    "Fold0 positives      :",
    int(y.sum()),
)


print(
    "\n========== CLEAN NATURAL-PRIOR PROBE =========="
)

print(
    summary_1109c
    .round(6)
    .to_string(index=False)
)


print(
    "\n========== CLEAN FULL-JOINT TOP-K =========="
)

print(
    topk_1109c
    .round(6)
    .to_string(index=False)
)

In [ ]:
# ============================================================
# RESULT — 11.09D Clean-Split Frame-Relative Ranking Audit
# Purpose:
#   Recompute within-frame division-parent ranking using the
#   corrected folds1–4-trained full-joint probe probabilities.
# Changes files: NO
# Runs training/inference: NO
# Uses GT: NO new GT access
# Keep after running: YES
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

audit_1109d_clean = allframe_1109c.copy()

p = np.asarray(
    probabilities["full_joint"],
    dtype=np.float64,
)

assert len(p) == len(audit_1109d_clean)

audit_1109d_clean["joint_score"] = p

g = audit_1109d_clean.groupby(
    ["dataset", "t"],
    sort=False,
)["joint_score"]

# 1 = highest-scoring parent in its frame.
audit_1109d_clean["frame_rank"] = g.rank(
    method="first",
    ascending=False,
)

audit_1109d_clean["frame_n"] = g.transform("size")

audit_1109d_clean["frame_percentile"] = (
    1.0
    - (
        audit_1109d_clean["frame_rank"] - 1.0
    )
    / np.maximum(
        audit_1109d_clean["frame_n"] - 1.0,
        1.0,
    )
)

frame_mean = g.transform("mean")
frame_std = g.transform("std").fillna(0.0)

audit_1109d_clean["frame_z"] = (
    audit_1109d_clean["joint_score"] - frame_mean
) / np.maximum(frame_std, 1e-8)


# ------------------------------------------------------------
# Global metrics
# ------------------------------------------------------------

y = audit_1109d_clean["label"].to_numpy()

metric_rows = []

for name, score in {
    "global_raw":
        audit_1109d_clean["joint_score"].to_numpy(),

    "frame_percentile":
        audit_1109d_clean["frame_percentile"].to_numpy(),

    "frame_z":
        audit_1109d_clean["frame_z"].to_numpy(),
}.items():

    ap = average_precision_score(y, score)

    metric_rows.append({
        "score": name,
        "AP": ap,
        "PR_lift": ap / y.mean(),
        "AUC": roc_auc_score(y, score),
    })

summary_1109d_clean = pd.DataFrame(metric_rows)


# ------------------------------------------------------------
# True-division within-frame ranks
# ------------------------------------------------------------

positives = audit_1109d_clean[
    audit_1109d_clean["label"] == 1
].copy()

rank_rows = []

for k in (1, 2, 3, 5, 10):

    n = int(
        (positives["frame_rank"] <= k).sum()
    )

    rank_rows.append({
        "within_frame_top_k": k,
        "division_count": n,
        "division_recall": n / len(positives),
    })

rank_summary_1109d_clean = pd.DataFrame(rank_rows)


print(
    "========== 11.09D CLEAN FRAME-RELATIVE METRICS =========="
)

print(
    summary_1109d_clean
    .round(6)
    .to_string(index=False)
)

print(
    "\n========== CLEAN TRUE-DIVISION FRAME RANKS =========="
)

print(
    rank_summary_1109d_clean
    .round(6)
    .to_string(index=False)
)

print(
    "\n========== CLEAN DIVISION RANK QUANTILES =========="
)

print(
    positives["frame_rank"]
    .quantile([0, 0.25, 0.5, 0.75, 1.0])
    .to_string()
)

In [ ]:
# ============================================================
# RESULT — 11.09E Clean Division-Frame Event Audit
# Purpose:
#   Recompute division-frame detection using ONLY clean
#   folds1–4-trained probe probabilities.
# Changes files: NO
# Runs training/inference: NO
# Uses GT: NO new GT access
# Keep after running: YES
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

df = allframe_1109c.copy()

for rep in REPRESENTATIONS:
    df[f"score_{rep}"] = np.asarray(
        probabilities[rep],
        dtype=np.float64,
    )

frame_rows = []

for (dataset, t), g in df.groupby(
    ["dataset", "t"],
    sort=False,
):
    row = {
        "dataset": dataset,
        "t": int(t),
        "label": int(g["label"].max()),
        "n_parents": len(g),
    }

    for rep in REPRESENTATIONS:
        s = np.sort(
            g[f"score_{rep}"].to_numpy()
        )[::-1]

        top1 = float(s[0])
        top2 = float(s[1]) if len(s) > 1 else top1

        row[f"{rep}_max"] = top1
        row[f"{rep}_margin"] = top1 - top2
        row[f"{rep}_top3mean"] = float(
            s[:min(3, len(s))].mean()
        )

    frame_rows.append(row)

frame_df_1109e_clean = pd.DataFrame(frame_rows)
y_frame = frame_df_1109e_clean["label"].to_numpy()

print("========== CLEAN FRAME POPULATION ==========")
print("frames          :", len(frame_df_1109e_clean))
print("division frames :", int(y_frame.sum()))
print("prevalence      :", f"{y_frame.mean():.6f}")


# ------------------------------------------------------------
# Frame-level signals
# ------------------------------------------------------------

score_defs = {
    "parent_only_max":
        frame_df_1109e_clean["parent_only_max"].to_numpy(),

    "scores_geometry_max":
        frame_df_1109e_clean["scores_geometry_max"].to_numpy(),

    "parent_candidate_max":
        frame_df_1109e_clean["parent_candidate_max"].to_numpy(),

    "full_joint_max":
        frame_df_1109e_clean["full_joint_max"].to_numpy(),

    "full_joint_margin":
        frame_df_1109e_clean["full_joint_margin"].to_numpy(),

    "full_joint_top3mean":
        frame_df_1109e_clean["full_joint_top3mean"].to_numpy(),
}

rows = []

for name, score in score_defs.items():
    ap = average_precision_score(y_frame, score)

    rows.append({
        "frame_score": name,
        "AP": ap,
        "PR_lift": ap / y_frame.mean(),
        "AUC": roc_auc_score(y_frame, score),
    })

summary_1109e_clean = (
    pd.DataFrame(rows)
    .sort_values("AP", ascending=False)
    .reset_index(drop=True)
)

print("\n========== CLEAN FRAME EVENT METRICS ==========")
print(
    summary_1109e_clean
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# Operational Top-K for full_joint_max
# ------------------------------------------------------------

score = score_defs["full_joint_max"]
order = np.argsort(-score, kind="stable")

top_rows = []

for K in (25, 50, 100, 250, 500):
    idx = order[:K]
    tp = int(y_frame[idx].sum())

    top_rows.append({
        "top_K_frames": K,
        "true_division_frames": tp,
        "false_frames": K - tp,
        "precision": tp / K,
        "event_recall": tp / y_frame.sum(),
    })

topk_1109e_clean = pd.DataFrame(top_rows)

print("\n========== CLEAN FRAME TOP-K ==========")
print(
    topk_1109e_clean
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# Two-stage:
# select frame by full_joint_max;
# within selected true frame select highest joint-score parent.
# ------------------------------------------------------------

parent_df = audit_1109d_clean.copy()

best_parent = (
    parent_df
    .sort_values(
        ["dataset", "t", "joint_score"],
        ascending=[True, True, False],
    )
    .groupby(["dataset", "t"], sort=False)
    .first()
    .reset_index()
)

parent_correct = (
    best_parent[
        ["dataset", "t", "label"]
    ]
    .rename(columns={"label": "top1_parent_correct"})
)

frame_eval = frame_df_1109e_clean.merge(
    parent_correct,
    on=["dataset", "t"],
    how="left",
)

score = frame_eval["full_joint_max"].to_numpy()
order = np.argsort(-score, kind="stable")

end_rows = []

for K in (25, 50, 100, 250, 500):
    selected = frame_eval.iloc[order[:K]]
    true_frames = selected[selected["label"] == 1]

    event_tp = len(true_frames)
    parent_tp = int(
        true_frames["top1_parent_correct"].sum()
    )

    end_rows.append({
        "top_K_frames": K,
        "event_TP": event_tp,
        "event_recall": event_tp / y_frame.sum(),
        "correct_parent_top1": parent_tp,
        "end_to_end_recall": parent_tp / y_frame.sum(),
    })

two_stage_1109e_clean = pd.DataFrame(end_rows)

print("\n========== CLEAN TWO-STAGE DECOMPOSITION ==========")
print(
    two_stage_1109e_clean
    .round(6)
    .to_string(index=False)
)

In [ ]:
# ============================================================
# RESULT — 11.10A Clean Temporal Division-Event Audit
# Purpose:
#   Re-evaluate simple temporal event signals using the
#   corrected folds1–4-trained full-joint probe.
# Changes files: NO
# Runs training/inference: NO
# Uses GT: NO new GT access
# Keep after running: YES
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


# ------------------------------------------------------------
# 1. Clean frame-level score table
# ------------------------------------------------------------

df = (
    frame_df_1109e_clean[
        [
            "dataset",
            "t",
            "label",
            "full_joint_max",
        ]
    ]
    .copy()
    .sort_values(["dataset", "t"])
    .reset_index(drop=True)
)

g = df.groupby(
    "dataset",
    sort=False,
)


# ------------------------------------------------------------
# 2. Exact consecutive temporal neighbors
# ------------------------------------------------------------

for k in (1, 2):

    prev_score = g[
        "full_joint_max"
    ].shift(k)

    next_score = g[
        "full_joint_max"
    ].shift(-k)

    prev_t = g[
        "t"
    ].shift(k)

    next_t = g[
        "t"
    ].shift(-k)

    df[f"prev{k}"] = prev_score.where(
        df["t"] - prev_t == k
    )

    df[f"next{k}"] = next_score.where(
        next_t - df["t"] == k
    )


s = df["full_joint_max"]

valid1 = (
    df["prev1"].notna()
    & df["next1"].notna()
)

valid2 = (
    valid1
    & df["prev2"].notna()
    & df["next2"].notna()
)


# ------------------------------------------------------------
# 3. Predeclared temporal signals
# ------------------------------------------------------------

df["rise_prev1"] = (
    s - df["prev1"]
)

df["drop_next1"] = (
    s - df["next1"]
)

df["contrast1"] = (
    s
    - 0.5
    * (
        df["prev1"]
        + df["next1"]
    )
)

df["peak1"] = (
    s
    - np.maximum(
        df["prev1"],
        df["next1"],
    )
)

df["contrast2"] = (
    s
    - (
        df["prev2"]
        + df["prev1"]
        + df["next1"]
        + df["next2"]
    ) / 4.0
)


# ------------------------------------------------------------
# 4. Paired metric evaluation
# ------------------------------------------------------------

def evaluate(name, column, mask):

    y = df.loc[
        mask,
        "label",
    ].to_numpy()

    score = df.loc[
        mask,
        column,
    ].to_numpy()

    ap = average_precision_score(
        y,
        score,
    )

    return {
        "signal": name,
        "n": len(y),
        "positive": int(y.sum()),
        "prevalence": float(y.mean()),
        "AP": ap,
        "PR_lift": ap / y.mean(),
        "AUC": roc_auc_score(
            y,
            score,
        ),
    }


summary_1110a_clean = pd.DataFrame([
    evaluate(
        "absolute_score",
        "full_joint_max",
        valid1,
    ),

    evaluate(
        "rise_prev1",
        "rise_prev1",
        valid1,
    ),

    evaluate(
        "drop_next1",
        "drop_next1",
        valid1,
    ),

    evaluate(
        "contrast1",
        "contrast1",
        valid1,
    ),

    evaluate(
        "peak1",
        "peak1",
        valid1,
    ),

    evaluate(
        "contrast2",
        "contrast2",
        valid2,
    ),
])


print(
    "========== 11.10A CLEAN TEMPORAL METRICS =========="
)

print(
    summary_1110a_clean
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 5. Operational Top-K
# ------------------------------------------------------------

topk_rows = []

for signal, column, mask in [
    (
        "absolute_score",
        "full_joint_max",
        valid1,
    ),
    (
        "contrast1",
        "contrast1",
        valid1,
    ),
    (
        "peak1",
        "peak1",
        valid1,
    ),
    (
        "contrast2",
        "contrast2",
        valid2,
    ),
]:

    sub = df.loc[mask]

    y = sub[
        "label"
    ].to_numpy()

    score = sub[
        column
    ].to_numpy()

    order = np.argsort(
        -score,
        kind="stable",
    )

    for K in (
        25,
        50,
        100,
        250,
        500,
    ):

        idx = order[:K]

        tp = int(
            y[idx].sum()
        )

        topk_rows.append({
            "signal": signal,
            "top_K": K,
            "TP": tp,
            "FP": K - tp,
            "precision": tp / K,
            "event_recall":
                tp / y.sum(),
        })


topk_1110a_clean = pd.DataFrame(
    topk_rows
)


print(
    "\n========== CLEAN TEMPORAL TOP-K =========="
)

print(
    topk_1110a_clean
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 6. True division-event score trajectory
# ------------------------------------------------------------

pos = df[
    df["label"] == 1
]

trajectory_1110a_clean = pd.DataFrame({
    "offset": [-2, -1, 0, 1, 2],

    "mean_score": [
        pos["prev2"].mean(),
        pos["prev1"].mean(),
        pos["full_joint_max"].mean(),
        pos["next1"].mean(),
        pos["next2"].mean(),
    ],

    "median_score": [
        pos["prev2"].median(),
        pos["prev1"].median(),
        pos["full_joint_max"].median(),
        pos["next1"].median(),
        pos["next2"].median(),
    ],
})


print(
    "\n========== CLEAN DIVISION EVENT TRAJECTORY =========="
)

print(
    trajectory_1110a_clean
    .round(6)
    .to_string(index=False)
)

In [ ]:
# ============================================================
# RESULT — 11.12A Multi-frame Supervision Eligibility Census
# Purpose:
#   Count clean tracked-parent examples that have:
#     t-2 -> t-1 -> t -> t+1
#   lineage coverage, so they can supervise a 4-frame
#   division-event capacity experiment.
# Changes files: NO
# Runs training/inference: NO
# Uses GT: YES — topology/count census only
# Keep after running: YES
# ============================================================

import numpy as np
import pandas as pd

manifest = pd.read_csv(
    PROJECT / "configs/cv_manifest.csv"
)

manifest = manifest[
    manifest["prefix"] == "6bba"
][["dataset", "fold", "prefix"]].copy()

rows = []

for r in manifest.itertuples(index=False):

    g = load_graph(
        TRAIN_ROOT / f"{r.dataset}.geff"
    )

    nodes = (
        g.node_attrs(
            attr_keys=["node_id", "t"]
        )
        .to_pandas()
    )

    edges = (
        g.edge_attrs(
            attr_keys=["source_id", "target_id"]
        )
        .to_pandas()
    )

    t_of = dict(
        zip(
            nodes["node_id"],
            nodes["t"],
        )
    )

    predecessors = {}
    successors = {}

    for e in edges.itertuples(index=False):
        s = int(e.source_id)
        d = int(e.target_id)

        successors.setdefault(s, []).append(d)
        predecessors.setdefault(d, []).append(s)

    for node_id, t in t_of.items():

        children = successors.get(node_id, [])

        # Only label-clean tracked parents:
        # ordinary=1 child, division=2 children.
        if len(children) not in (1, 2):
            continue

        p1 = predecessors.get(node_id, [])

        if len(p1) != 1:
            continue

        p1 = p1[0]

        if t_of.get(p1) != t - 1:
            continue

        p2 = predecessors.get(p1, [])

        if len(p2) != 1:
            continue

        p2 = p2[0]

        if t_of.get(p2) != t - 2:
            continue

        # Children must really belong to t+1.
        if not all(
            t_of.get(c) == t + 1
            for c in children
        ):
            continue

        rows.append({
            "dataset": r.dataset,
            "fold": int(r.fold),
            "node_id": int(node_id),
            "t": int(t),
            "label": int(len(children) == 2),
        })


eligible_1112a = pd.DataFrame(rows)

summary_1112a = (
    eligible_1112a
    .groupby("fold")
    .agg(
        examples=("label", "size"),
        divisions=("label", "sum"),
    )
    .reset_index()
)

summary_1112a["prevalence"] = (
    summary_1112a["divisions"]
    / summary_1112a["examples"]
)

print("========== 11.12A MULTI-FRAME ELIGIBILITY ==========")
print(
    summary_1112a
    .round(6)
    .to_string(index=False)
)

print("\n========== OUTER SPLIT ==========")

train_mask = eligible_1112a["fold"] != 0
val_mask = eligible_1112a["fold"] == 0

print(
    "train folds1-4:",
    int(train_mask.sum()),
    "examples /",
    int(
        eligible_1112a.loc[
            train_mask,
            "label",
        ].sum()
    ),
    "divisions",
)

print(
    "Fold0:",
    int(val_mask.sum()),
    "examples /",
    int(
        eligible_1112a.loc[
            val_mask,
            "label",
        ].sum()
    ),
    "divisions",
)

print(
    "\nTotal retained divisions:",
    int(eligible_1112a["label"].sum()),
)

### 11.12E — One-Dataset Extractor Smoke Gate

`6bba_df673a83` completed successfully with the new reusable extractor.

- 645 eligible parent rows
- 4 division positives
- 98 source frames
- median GT match distance: 1.8617 µm
- maximum GT match distance: 6.9182 µm
- all feature schemas and finite-value checks passed
- elapsed: 57.8 s

Decision: ONE_DATASET_EXTRACTION_PASS.

Full folds1–4 extraction is deferred until numerical fidelity against the
previous validated Stage-11 extraction is established.

In [ ]:
# ============================================================
# RUN — 11.12G One-Dataset Legacy-Schedule Reference
# Purpose:
#   Regenerate Stage11 joint features for 6bba_df673a83 using
#   the OLD validated DatasetReplay.replay_pair(t) schedule:
#
#       source detection: [t-1, t]
#       association pair: [t, t+1]
#
#   This provides an independent numerical reference for the
#   new cached/batched extractor.
#
# Changes files: YES
#   - reports/stage11_division_probe/
#       stage11_multiframe_fidelity_legacy.pkl
#
# Runs training: NO
# Runs inference: YES — one dataset only
# Uses GT: YES — matching/labels only, after model inference
# Keep after running: NO — delete after fidelity comparison
# ============================================================

from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
import torch


# ------------------------------------------------------------
# 1. Paths / imports
# ------------------------------------------------------------

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

CHECKPOINT = (
    PROJECT
    / "models/official_baseline/"
      "stage10_pilot_baseline_256/"
      "split_0/edge_predictor_best.pth"
)

OUT = (
    PROJECT
    / "reports/stage11_division_probe/"
      "stage11_multiframe_fidelity_legacy.pkl"
)

dataset = "6bba_df673a83"
fold = 0

sys.path.insert(
    0,
    str(PROJECT / "scripts"),
)

import stage11_multiframe_utils as mf
import run_stage11_multiframe_extract as runner

from replay_stage11_association import DatasetReplay


# ------------------------------------------------------------
# 2. Environment / model
# ------------------------------------------------------------

assert Path(sys.executable) == (
    PROJECT / ".venv/bin/python"
)

device = runner.resolve_device("mps")

model, window_size, downsample = runner.load_model(
    CHECKPOINT,
    device,
)

model.eval()

print("========== LEGACY REFERENCE SETUP ==========")
print("dataset     :", dataset)
print("device      :", device)
print("window_size :", window_size)
print("downsample  :", downsample)

assert int(window_size) == 2
assert tuple(downsample) == (1, 4, 4)


# ------------------------------------------------------------
# 3. Exact old DatasetReplay
# ------------------------------------------------------------

replay = DatasetReplay(
    dataset,
    TRAIN_ROOT,
    model,
    window_size,
    downsample,
    device,
)

downsample_arr = np.asarray(
    downsample,
    dtype=np.float32,
)

assert np.allclose(
    np.asarray(replay.scale, dtype=np.float32),
    mf.SCALE_UM,
)


# ------------------------------------------------------------
# 4. GT tables — labels enter only AFTER replay_pair()
# ------------------------------------------------------------

gt = runner.load_graph(
    TRAIN_ROOT / f"{dataset}.geff"
)

gt_nodes = (
    gt.node_attrs(
        attr_keys=[
            "node_id",
            "t",
            "z",
            "y",
            "x",
        ]
    )
    .to_pandas()
)

gt_edges = (
    gt.edge_attrs(
        attr_keys=[
            "source_id",
            "target_id",
        ]
    )
    .to_pandas()
)

degrees = mf.gt_child_degrees(
    gt_nodes,
    gt_edges,
)

division_times = mf.division_frame_times(
    gt_nodes,
    degrees,
)

labelled_ids = set(
    degrees.index.astype(int)
)


# ------------------------------------------------------------
# 5. Capture exact post-attention q / k contexts
# ------------------------------------------------------------

context_bucket = []

hook = (
    model.transformer.norm_out
    .register_forward_hook(
        lambda module, inp, out:
            context_bucket.append(
                out.detach().cpu()
            )
    )
)


# ------------------------------------------------------------
# 6. OLD replay schedule — one pair at a time
# ------------------------------------------------------------

rows = []

t0 = time.perf_counter()

try:

    for t in range(int(replay.T) - 1):

        # Critical:
        # clear BEFORE replay_pair because predict_edges()
        # inside replay_pair triggers norm_out twice: q and k.
        context_bucket.clear()

        with torch.no_grad():

            c_src, c_tgt, raw_np, probs_np = (
                replay.replay_pair(t)
            )

        # No association prediction occurred.
        if (
            raw_np is None
            or probs_np is None
            or len(c_src) == 0
            or len(c_tgt) < 2
        ):
            continue

        if len(context_bucket) != 2:
            raise RuntimeError(
                "Expected exactly 2 post-attention contexts "
                f"at t={t}; got {len(context_bucket)}"
            )

        q = (
            context_bucket[0][0]
            .numpy()
        )

        k = (
            context_bucket[1][0]
            .numpy()
        )

        n_src = len(c_src)
        n_tgt = len(c_tgt)

        if q.shape != (n_src, 128):
            raise RuntimeError(
                f"Bad q shape at t={t}: {q.shape}"
            )

        if k.shape != (n_tgt, 128):
            raise RuntimeError(
                f"Bad k shape at t={t}: {k.shape}"
            )


        # ----------------------------------------------------
        # GT enters only here.
        # ----------------------------------------------------

        frame_gt = gt_nodes[
            (gt_nodes["t"].astype(int) == t)
            & (
                gt_nodes["node_id"]
                .astype(int)
                .isin(labelled_ids)
            )
        ]

        if len(frame_gt) == 0:
            continue

        gt_xyz = (
            frame_gt[
                ["z", "y", "x"]
            ]
            .to_numpy(np.float32)
        )

        # Detected coords are on downsampled grid.
        pred_xyz = (
            c_src[:, 1:4]
            .astype(np.float32)
            * downsample_arr
        )

        matches = mf.match_gt_to_pred(
            gt_xyz,
            pred_xyz,
            mf.SCALE_UM,
        )

        is_division_frame = bool(
            t in division_times
        )


        # ----------------------------------------------------
        # Feature assembly — same pure semantics as 11.09A
        # ----------------------------------------------------

        for gi, pi, dist_um in matches:

            node_id = int(
                frame_gt.iloc[gi][
                    "node_id"
                ]
            )

            label = mf.eligible_label(
                int(
                    degrees.loc[node_id]
                )
            )

            if label is None:
                continue

            j1, j2 = mf.top2_by_raw_logit(
                raw_np[pi]
            )

            raw1 = float(
                raw_np[pi, j1]
            )

            raw2 = float(
                raw_np[pi, j2]
            )

            prob1 = float(
                probs_np[pi, j1]
            )

            prob2 = float(
                probs_np[pi, j2]
            )

            k1 = k[j1]
            k2 = k[j2]

            rows.append(
                {
                    "dataset":
                        dataset,

                    "fold":
                        int(fold),

                    "t":
                        int(t),

                    "node_id":
                        node_id,

                    "label":
                        int(label),

                    "division_frame":
                        is_division_frame,

                    "match_distance_um":
                        float(dist_um),

                    "parent_q":
                        q[pi]
                        .astype(np.float32)
                        .copy(),

                    "k_mean":
                        (
                            (k1 + k2) / 2.0
                        )
                        .astype(np.float32),

                    "k_absdiff":
                        np.abs(k1 - k2)
                        .astype(np.float32),

                    "score_features":
                        mf.score_features(
                            raw1,
                            raw2,
                            prob1,
                            prob2,
                        ),

                    "geometry":
                        mf.candidate_geometry(
                            c_src[
                                pi,
                                1:4,
                            ].astype(
                                np.float32
                            ),

                            c_tgt[
                                j1,
                                1:4,
                            ].astype(
                                np.float32
                            ),

                            c_tgt[
                                j2,
                                1:4,
                            ].astype(
                                np.float32
                            ),

                            downsample_arr,
                            mf.SCALE_UM,
                        ),

                    # Despite historical column naming,
                    # Stage11 coordinate convention is Z,Y,X.
                    "top1_xyz":
                        c_tgt[
                            j1,
                            1:4,
                        ]
                        .astype(np.float32)
                        .copy(),

                    "top2_xyz":
                        c_tgt[
                            j2,
                            1:4,
                        ]
                        .astype(np.float32)
                        .copy(),
                }
            )

finally:

    hook.remove()


elapsed = (
    time.perf_counter()
    - t0
)


# ------------------------------------------------------------
# 7. Persist and validate
# ------------------------------------------------------------

legacy_1112g = pd.DataFrame(
    rows,
    columns=mf.OUTPUT_COLUMNS,
)

mf.validate_schema(
    legacy_1112g
)

OUT.parent.mkdir(
    parents=True,
    exist_ok=True,
)

legacy_1112g.to_pickle(
    OUT
)


# ------------------------------------------------------------
# 8. Concise result
# ------------------------------------------------------------

print(
    "\n========== LEGACY REFERENCE SUMMARY =========="
)

print(
    "rows            :",
    len(legacy_1112g),
)

print(
    "positives       :",
    int(
        legacy_1112g[
            "label"
        ].sum()
    ),
)

print(
    "division frames :",
    int(
        legacy_1112g.loc[
            legacy_1112g[
                "division_frame"
            ],
            "t",
        ].nunique()
    ),
)

print(
    "source frames   :",
    legacy_1112g[
        "t"
    ].nunique(),
)

print(
    "match median um :",
    round(
        float(
            legacy_1112g[
                "match_distance_um"
            ].median()
        ),
        6,
    ),
)

print(
    "match max um    :",
    round(
        float(
            legacy_1112g[
                "match_distance_um"
            ].max()
        ),
        6,
    ),
)

print(
    f"elapsed         : {elapsed:.1f}s"
)

print(
    "written         :",
    OUT,
)

print(
    "\n========== 11.12G DECISION =========="
)

print(
    "LEGACY_REFERENCE_GENERATED: PASS"
)

In [ ]:
# ============================================================
# RESULT — 11.12H Exact New-vs-Legacy Extractor Fidelity Gate
# Purpose:
#   Compare the new cached/batched extractor numerically
#   against the independently regenerated legacy replay
#   schedule on the exact same dataset.
#
# Changes files: NO
# Runs training/inference: NO
# Uses GT: NO
# Keep after running: YES
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

REPORT_DIR = (
    PROJECT
    / "reports/stage11_division_probe"
)

NEW_PATH = (
    REPORT_DIR
    / "stage11_multiframe_fidelity_fold0.pkl"
)

LEGACY_PATH = (
    REPORT_DIR
    / "stage11_multiframe_fidelity_legacy.pkl"
)

new = pd.read_pickle(NEW_PATH)
legacy = pd.read_pickle(LEGACY_PATH)

KEY = [
    "dataset",
    "t",
    "node_id",
]


# ------------------------------------------------------------
# 1. Row identity
# ------------------------------------------------------------

assert not new.duplicated(KEY).any()
assert not legacy.duplicated(KEY).any()

new_keys = set(
    map(
        tuple,
        new[KEY].to_numpy(),
    )
)

legacy_keys = set(
    map(
        tuple,
        legacy[KEY].to_numpy(),
    )
)

print(
    "========== ROW IDENTITY =========="
)

print(
    "new rows       :",
    len(new),
)

print(
    "legacy rows    :",
    len(legacy),
)

print(
    "missing in new :",
    len(legacy_keys - new_keys),
)

print(
    "extra in new   :",
    len(new_keys - legacy_keys),
)

assert new_keys == legacy_keys


# ------------------------------------------------------------
# 2. One-to-one aligned table
# ------------------------------------------------------------

m = legacy.merge(
    new,
    on=KEY,
    suffixes=(
        "_legacy",
        "_new",
    ),
    validate="one_to_one",
)

assert len(m) == len(new)


# ------------------------------------------------------------
# 3. Exact categorical fidelity
# ------------------------------------------------------------

print(
    "\n========== CATEGORICAL FIDELITY =========="
)

categorical_cols = [
    "fold",
    "label",
    "division_frame",
]

for col in categorical_cols:

    exact = np.array_equal(
        m[f"{col}_legacy"].to_numpy(),
        m[f"{col}_new"].to_numpy(),
    )

    print(
        f"{col:20s}: exact={exact}"
    )

    assert exact


# ------------------------------------------------------------
# 4. Scalar fidelity
# ------------------------------------------------------------

dist_delta = float(
    np.max(
        np.abs(
            m[
                "match_distance_um_legacy"
            ].to_numpy(dtype=np.float64)
            -
            m[
                "match_distance_um_new"
            ].to_numpy(dtype=np.float64)
        )
    )
)

print(
    "\n========== SCALAR FIDELITY =========="
)

print(
    f"{'match_distance_um':20s}: "
    f"max_delta={dist_delta:.3e}"
)


# ------------------------------------------------------------
# 5. Array-feature fidelity
# ------------------------------------------------------------

ARRAY_COLS = [
    "parent_q",
    "k_mean",
    "k_absdiff",
    "score_features",
    "geometry",
    "top1_xyz",
    "top2_xyz",
]

max_deltas = {}

print(
    "\n========== FEATURE FIDELITY =========="
)

for col in ARRAY_COLS:

    a = np.stack(
        m[
            f"{col}_legacy"
        ].to_numpy()
    ).astype(np.float64)

    b = np.stack(
        m[
            f"{col}_new"
        ].to_numpy()
    ).astype(np.float64)

    assert a.shape == b.shape

    delta = float(
        np.max(
            np.abs(a - b)
        )
    )

    max_deltas[col] = delta

    exact = np.array_equal(
        a,
        b,
    )

    print(
        f"{col:20s}: "
        f"exact={str(exact):5s} "
        f"max_delta={delta:.3e}"
    )


# ------------------------------------------------------------
# 6. Candidate identity fidelity
# ------------------------------------------------------------

candidate_exact = (
    np.array_equal(
        np.stack(
            m["top1_xyz_legacy"]
        ),
        np.stack(
            m["top1_xyz_new"]
        ),
    )
    and
    np.array_equal(
        np.stack(
            m["top2_xyz_legacy"]
        ),
        np.stack(
            m["top2_xyz_new"]
        ),
    )
)

print(
    "\n========== CANDIDATE IDENTITY =========="
)

print(
    "top1/top2 exact:",
    candidate_exact,
)


# ------------------------------------------------------------
# 7. Decision
# ------------------------------------------------------------

TOL = 1e-5

pass_gate = (
    dist_delta <= TOL
    and candidate_exact
    and all(
        delta <= TOL
        for delta in max_deltas.values()
    )
)

print(
    "\n========== 11.12H DECISION =========="
)

print(
    "FIDELITY:",
    "PASS"
    if pass_gate
    else "FAIL",
)

print(
    "tolerance:",
    TOL,
)

if not pass_gate:
    raise AssertionError(
        "New extractor does not reproduce "
        "legacy Stage11 semantics."
    )

In [ ]:
# ============================================================
# RUN — 11.12I Full Folds1–4 Natural-Prior Feature Extraction
# Purpose:
#   Extract the validated Stage11 joint representation for ALL
#   6bba datasets in outer-training folds 1–4.
#
#   This produces the natural-prior training feature table for
#   the multi-frame division-capacity experiment.
#
# Changes files: YES
#   - reports/stage11_division_probe/
#       stage11_multiframe_folds1to4.pkl
#       stage11_multiframe_folds1to4.log
#
# Runs training: NO
# Runs inference: YES — 103 6bba datasets
# Uses GT: YES — matching / labels only, after inference
# Keep after running: NO — delete after successful extraction
# ============================================================

from pathlib import Path
import subprocess
import time
import sys
import pandas as pd
import numpy as np

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

DATA_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development"
)

MANIFEST = (
    PROJECT
    / "configs/cv_manifest.csv"
)

CHECKPOINT = (
    PROJECT
    / "models/official_baseline/"
      "stage10_pilot_baseline_256/"
      "split_0/edge_predictor_best.pth"
)

CHECKPOINT_SHA = (
    "8bddad68aaabf240d36eab9cda158f12"
    "d4969fa1cd6b21277570043ef83ab689"
)

RUNNER = (
    PROJECT
    / "scripts/run_stage11_multiframe_extract.py"
)

REPORT_DIR = (
    PROJECT
    / "reports/stage11_division_probe"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OUT = (
    REPORT_DIR
    / "stage11_multiframe_folds1to4.pkl"
)

LOG = (
    REPORT_DIR
    / "stage11_multiframe_folds1to4.log"
)


# ------------------------------------------------------------
# 1. Environment / selection sanity
# ------------------------------------------------------------

assert Path(sys.executable) == (
    PROJECT / ".venv/bin/python"
)

assert Path(sys.prefix) == (
    PROJECT / ".venv"
)

assert MANIFEST.exists()
assert CHECKPOINT.exists()
assert RUNNER.exists()

manifest = pd.read_csv(
    MANIFEST
)

selected = manifest[
    (manifest["prefix"] == "6bba")
    & (
        manifest["fold"]
        .astype(int)
        .isin([1, 2, 3, 4])
    )
].copy()

assert len(selected) == 103, (
    f"Expected 103 folds1–4 6bba datasets, "
    f"got {len(selected)}"
)

print(
    "========== 11.12I EXTRACTION SETUP =========="
)

print(
    "Python   :",
    sys.executable,
)

print(
    "datasets :",
    len(selected),
)

print(
    "folds    :",
    sorted(
        selected["fold"]
        .astype(int)
        .unique()
    ),
)

print(
    "output   :",
    OUT,
)

print(
    "log      :",
    LOG,
)

print(
    "\nStarting full folds1–4 extraction."
)

print(
    "Expected runtime: approximately 1.5–2 hours."
)


# ------------------------------------------------------------
# 2. Run — stdout/stderr go entirely to log
# ------------------------------------------------------------

cmd = [
    sys.executable,
    str(RUNNER),

    "--data-root",
    str(DATA_ROOT),

    "--manifest",
    str(MANIFEST),

    "--checkpoint",
    str(CHECKPOINT),

    "--checkpoint-sha256",
    CHECKPOINT_SHA,

    "--output",
    str(OUT),

    "--folds",
    "1,2,3,4",

    "--prefix",
    "6bba",

    "--device",
    "mps",

    "--batch-size",
    "4",
]

t0 = time.perf_counter()

with LOG.open("w") as log:
    result = subprocess.run(
        cmd,
        cwd=PROJECT,
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
    )

elapsed = (
    time.perf_counter()
    - t0
)


# ------------------------------------------------------------
# 3. Failure handling
# ------------------------------------------------------------

print(
    "\n========== RUN RESULT =========="
)

print(
    "Exit code:",
    result.returncode,
)

print(
    f"Elapsed  : {elapsed / 60:.1f} min"
)

if result.returncode != 0:

    print(
        "\n========== LOG TAIL =========="
    )

    print(
        "\n".join(
            LOG.read_text().splitlines()[-60:]
        )
    )

    raise RuntimeError(
        "11.12I folds1–4 extraction failed"
    )


# ------------------------------------------------------------
# 4. Load and validate result
# ------------------------------------------------------------

assert OUT.exists()

train_features_1112i = pd.read_pickle(
    OUT
)

EXPECTED_COLUMNS = [
    "dataset",
    "fold",
    "t",
    "node_id",
    "label",
    "division_frame",
    "match_distance_um",
    "parent_q",
    "k_mean",
    "k_absdiff",
    "score_features",
    "geometry",
    "top1_xyz",
    "top2_xyz",
]

assert list(
    train_features_1112i.columns
) == EXPECTED_COLUMNS

assert set(
    train_features_1112i[
        "fold"
    ].astype(int).unique()
) == {1, 2, 3, 4}

assert set(
    train_features_1112i[
        "dataset"
    ].unique()
).issubset(
    set(selected["dataset"])
)

assert set(
    train_features_1112i[
        "label"
    ].unique()
).issubset({0, 1})


# ------------------------------------------------------------
# 5. Feature integrity
# ------------------------------------------------------------

FEATURE_DIMS = {
    "parent_q": 128,
    "k_mean": 128,
    "k_absdiff": 128,
    "score_features": 6,
    "geometry": 5,
    "top1_xyz": 3,
    "top2_xyz": 3,
}

for col, dim in FEATURE_DIMS.items():

    arr = np.stack(
        train_features_1112i[
            col
        ].to_numpy()
    )

    assert arr.shape == (
        len(train_features_1112i),
        dim,
    )

    assert np.isfinite(
        arr
    ).all()


# ------------------------------------------------------------
# 6. Compact summary
# ------------------------------------------------------------

summary = (
    train_features_1112i
    .groupby("fold")
    .agg(
        datasets=(
            "dataset",
            "nunique",
        ),
        rows=(
            "label",
            "size",
        ),
        positives=(
            "label",
            "sum",
        ),
    )
    .reset_index()
)

summary["prevalence"] = (
    summary["positives"]
    / summary["rows"]
)

print(
    "\n========== FOLDS1–4 EXTRACTION SUMMARY =========="
)

print(
    summary
    .round(6)
    .to_string(index=False)
)

print(
    "\nTOTAL"
)

print(
    "datasets :",
    train_features_1112i[
        "dataset"
    ].nunique(),
)

print(
    "rows     :",
    len(train_features_1112i),
)

print(
    "positives:",
    int(
        train_features_1112i[
            "label"
        ].sum()
    ),
)

print(
    "median match distance:",
    round(
        float(
            train_features_1112i[
                "match_distance_um"
            ].median()
        ),
        6,
    ),
    "um",
)


print(
    "\n========== LOG TAIL =========="
)

print(
    "\n".join(
        LOG.read_text().splitlines()[-15:]
    )
)


print(
    "\n========== 11.12I DECISION =========="
)

print(
    "FOLDS1TO4_EXTRACTION: PASS"
)

In [ ]:
# ============================================================
# RESULT — 11.12K Multi-frame Lineage Join Coverage
# Purpose:
#   Join the persisted natural-prior feature tables along the
#   SAME GT lineage to attach:
#
#       q_prev1 = parent context at t-1
#       q_prev2 = parent context at t-2
#
#   This is a teacher-forced CAPACITY AUDIT only:
#   GT lineage is used for temporal alignment, not deployment.
#
# Changes files: YES
#   - reports/stage11_division_probe/
#       stage11_multiframe_lineage_join.pkl
#
# Runs training/inference: NO
# Uses GT: YES — predecessor identity only
# Keep after running: YES
# ============================================================

from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

REPORT_DIR = (
    PROJECT
    / "reports/stage11_division_probe"
)

TRAIN_FEATURES = (
    REPORT_DIR
    / "stage11_multiframe_folds1to4.pkl"
)

FOLD0_FEATURES = (
    REPORT_DIR
    / "stage11_multiframe_fold0.pkl"
)

OUT = (
    REPORT_DIR
    / "stage11_multiframe_lineage_join.pkl"
)

sys.path.insert(
    0,
    str(PROJECT / "scripts"),
)

import run_stage11_multiframe_extract as runner


# ------------------------------------------------------------
# 1. Load persisted clean-split feature tables
# ------------------------------------------------------------

train_df = pd.read_pickle(
    TRAIN_FEATURES
)

fold0_df = pd.read_pickle(
    FOLD0_FEATURES
)

assert set(
    train_df["fold"].astype(int).unique()
) == {1, 2, 3, 4}

assert set(
    fold0_df["fold"].astype(int).unique()
) == {0}

base = pd.concat(
    [
        train_df,
        fold0_df,
    ],
    ignore_index=True,
)

KEY = [
    "dataset",
    "t",
    "node_id",
]

assert not base.duplicated(KEY).any()

print(
    "========== BASE FEATURE POPULATION =========="
)

print(
    "rows      :",
    len(base),
)

print(
    "positives :",
    int(base["label"].sum()),
)

print(
    "datasets  :",
    base["dataset"].nunique(),
)


# ------------------------------------------------------------
# 2. Build exact GT predecessor maps
# ------------------------------------------------------------

lineage_parts = []

for dataset in sorted(
    base["dataset"].unique()
):

    gt = runner.load_graph(
        TRAIN_ROOT
        / f"{dataset}.geff"
    )

    nodes = (
        gt.node_attrs(
            attr_keys=[
                "node_id",
                "t",
            ]
        )
        .to_pandas()
    )

    edges = (
        gt.edge_attrs(
            attr_keys=[
                "source_id",
                "target_id",
            ]
        )
        .to_pandas()
    )

    t_of = {
        int(n):
            int(t)
        for n, t
        in zip(
            nodes["node_id"],
            nodes["t"],
        )
    }

    predecessor = {}

    for e in edges.itertuples(
        index=False
    ):

        src = int(
            e.source_id
        )

        tgt = int(
            e.target_id
        )

        # Stage-11 GT contract:
        # only consecutive temporal lineage edges.
        if (
            t_of.get(tgt)
            != t_of.get(src, -999) + 1
        ):
            continue

        if tgt in predecessor:
            raise RuntimeError(
                f"GT node has >1 predecessor: "
                f"{dataset} node={tgt}"
            )

        predecessor[tgt] = src


    sub = base[
        base["dataset"] == dataset
    ][
        [
            "dataset",
            "t",
            "node_id",
        ]
    ].copy()

    prev1_ids = []
    prev1_ts = []

    prev2_ids = []
    prev2_ts = []

    for r in sub.itertuples(
        index=False
    ):

        node_id = int(
            r.node_id
        )

        t = int(
            r.t
        )

        p1 = predecessor.get(
            node_id
        )

        if (
            p1 is None
            or t_of.get(p1) != t - 1
        ):
            prev1_ids.append(
                np.nan
            )

            prev1_ts.append(
                np.nan
            )

            prev2_ids.append(
                np.nan
            )

            prev2_ts.append(
                np.nan
            )

            continue

        prev1_ids.append(
            p1
        )

        prev1_ts.append(
            t - 1
        )

        p2 = predecessor.get(
            p1
        )

        if (
            p2 is None
            or t_of.get(p2) != t - 2
        ):
            prev2_ids.append(
                np.nan
            )

            prev2_ts.append(
                np.nan
            )

        else:
            prev2_ids.append(
                p2
            )

            prev2_ts.append(
                t - 2
            )


    sub[
        "prev1_node_id"
    ] = prev1_ids

    sub[
        "prev1_t"
    ] = prev1_ts

    sub[
        "prev2_node_id"
    ] = prev2_ids

    sub[
        "prev2_t"
    ] = prev2_ts

    lineage_parts.append(
        sub
    )


lineage = pd.concat(
    lineage_parts,
    ignore_index=True,
)


# ------------------------------------------------------------
# 3. Attach lineage IDs to current representations
# ------------------------------------------------------------

joined = base.merge(
    lineage,
    on=KEY,
    how="left",
    validate="one_to_one",
)


# ------------------------------------------------------------
# 4. Build q(t-1) lookup
# ------------------------------------------------------------

q1_lookup = base[
    [
        "dataset",
        "t",
        "node_id",
        "parent_q",
    ]
].rename(
    columns={
        "t":
            "prev1_t",

        "node_id":
            "prev1_node_id",

        "parent_q":
            "q_prev1",
    }
)

joined = joined.merge(
    q1_lookup,
    on=[
        "dataset",
        "prev1_t",
        "prev1_node_id",
    ],
    how="left",
    validate="many_to_one",
)


# ------------------------------------------------------------
# 5. Build q(t-2) lookup
# ------------------------------------------------------------

q2_lookup = base[
    [
        "dataset",
        "t",
        "node_id",
        "parent_q",
    ]
].rename(
    columns={
        "t":
            "prev2_t",

        "node_id":
            "prev2_node_id",

        "parent_q":
            "q_prev2",
    }
)

joined = joined.merge(
    q2_lookup,
    on=[
        "dataset",
        "prev2_t",
        "prev2_node_id",
    ],
    how="left",
    validate="many_to_one",
)


# ------------------------------------------------------------
# 6. Coverage flags
# ------------------------------------------------------------

joined[
    "has_q_prev1"
] = joined[
    "q_prev1"
].notna()

joined[
    "has_q_prev2"
] = (
    joined["q_prev1"].notna()
    &
    joined["q_prev2"].notna()
)


# ------------------------------------------------------------
# 7. Validate temporal feature dimensions
# ------------------------------------------------------------

for col in [
    "q_prev1",
    "q_prev2",
]:

    nonnull = joined[
        joined[col].notna()
    ][col]

    if len(nonnull):

        arr = np.stack(
            nonnull.to_numpy()
        )

        assert arr.shape[1] == 128

        assert np.isfinite(
            arr
        ).all()


# ------------------------------------------------------------
# 8. Coverage summary by fold
# ------------------------------------------------------------

rows = []

for fold in range(5):

    f = joined[
        joined["fold"].astype(int)
        == fold
    ]

    f1 = f[
        f["has_q_prev1"]
    ]

    f2 = f[
        f["has_q_prev2"]
    ]

    rows.append({
        "fold":
            fold,

        "all_rows":
            len(f),

        "all_pos":
            int(
                f["label"].sum()
            ),

        "q1_rows":
            len(f1),

        "q1_pos":
            int(
                f1["label"].sum()
            ),

        "q2_rows":
            len(f2),

        "q2_pos":
            int(
                f2["label"].sum()
            ),

        "q2_row_recall":
            len(f2)
            / max(len(f), 1),

        "q2_pos_recall":
            (
                f2["label"].sum()
                / max(
                    f["label"].sum(),
                    1,
                )
            ),
    })


coverage_1112k = pd.DataFrame(
    rows
)


print(
    "\n========== 11.12K LINEAGE COVERAGE =========="
)

print(
    coverage_1112k
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 9. Outer-split totals on common X2 population
# ------------------------------------------------------------

train_common = joined[
    (joined["fold"].astype(int) != 0)
    &
    joined["has_q_prev2"]
]

val_common = joined[
    (joined["fold"].astype(int) == 0)
    &
    joined["has_q_prev2"]
]


print(
    "\n========== X2 COMMON POPULATION =========="
)

print(
    "train folds1-4 :",
    len(train_common),
    "rows /",
    int(
        train_common[
            "label"
        ].sum()
    ),
    "positives",
)

print(
    "Fold0          :",
    len(val_common),
    "rows /",
    int(
        val_common[
            "label"
        ].sum()
    ),
    "positives",
)

print(
    "Fold0 prevalence:",
    f"{val_common['label'].mean():.6f}",
)


# ------------------------------------------------------------
# 10. Persist joined table
# ------------------------------------------------------------

joined.to_pickle(
    OUT
)

print(
    "\nwritten:",
    OUT
)

print(
    "\n========== 11.12K DECISION =========="
)

print(
    "LINEAGE_JOIN: PASS"
)

In [ ]:
# ============================================================
# RESULT — 11.12L Clean Multi-frame Division Capacity Probe
# Purpose:
#   Test whether SAME-LINEAGE historical transformer contexts
#   add division-event information beyond the current-frame
#   full-joint representation.
#
# Outer protocol:
#   - Hyperparameter selection: folds 1–4 ONLY
#   - Final evaluation: Fold0 ONCE
#   - All representations evaluated on the identical X2-common
#     population to eliminate history-availability confounding.
#
# Changes files: NO
# Runs training: YES — logistic probes only
# Runs inference: NO
# Uses GT: NO new GT access — reuses lineage alignment
# Keep after running: YES
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


# ------------------------------------------------------------
# 1. Load common multi-frame population
# ------------------------------------------------------------

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

JOINED_PATH = (
    PROJECT
    / "reports/stage11_division_probe/"
      "stage11_multiframe_lineage_join.pkl"
)

joined = pd.read_pickle(
    JOINED_PATH
)

common = joined[
    joined["has_q_prev2"]
].copy()

train = common[
    common["fold"].astype(int) != 0
].copy()

val = common[
    common["fold"].astype(int) == 0
].copy()

assert set(
    train["fold"].astype(int).unique()
) == {1, 2, 3, 4}

assert set(
    val["fold"].astype(int).unique()
) == {0}

assert len(train) == 78562
assert int(train["label"].sum()) == 78

assert len(val) == 18504
assert int(val["label"].sum()) == 21


print(
    "========== COMMON POPULATION =========="
)

print(
    "train:",
    len(train),
    "rows /",
    int(train["label"].sum()),
    "positives",
)

print(
    "Fold0:",
    len(val),
    "rows /",
    int(val["label"].sum()),
    "positives",
)

print(
    "Fold0 prevalence:",
    f"{val['label'].mean():.6f}",
)


# ------------------------------------------------------------
# 2. Fixed feature definitions
# ------------------------------------------------------------

def stack_col(df, col):
    return np.stack(
        df[col].to_numpy()
    ).astype(np.float64)


def build_representation(df, name):

    q_t = stack_col(
        df,
        "parent_q",
    )

    k_mean = stack_col(
        df,
        "k_mean",
    )

    k_absdiff = stack_col(
        df,
        "k_absdiff",
    )

    score = stack_col(
        df,
        "score_features",
    )

    geometry = stack_col(
        df,
        "geometry",
    )

    q1 = stack_col(
        df,
        "q_prev1",
    )

    q2 = stack_col(
        df,
        "q_prev2",
    )

    X0 = np.concatenate(
        [
            q_t,
            k_mean,
            k_absdiff,
            score,
            geometry,
        ],
        axis=1,
    )

    if name == "X0_current":
        return X0

    if name == "X1_prev1":
        return np.concatenate(
            [
                X0,
                q1,
            ],
            axis=1,
        )

    if name == "X2_prev2":
        return np.concatenate(
            [
                X0,
                q1,
                q2,
            ],
            axis=1,
        )

    if name == "X3_temporal_delta":
        return np.concatenate(
            [
                X0,
                q1,
                q2,

                # Explicit signed temporal changes.
                q_t - q1,
                q1 - q2,
            ],
            axis=1,
        )

    raise ValueError(
        f"Unknown representation: {name}"
    )


REPRESENTATIONS = [
    "X0_current",
    "X1_prev1",
    "X2_prev2",
    "X3_temporal_delta",
]

for rep in REPRESENTATIONS:
    print(
        rep,
        build_representation(
            train.iloc[:2],
            rep,
        ).shape[1],
        "dims",
    )


# ------------------------------------------------------------
# 3. Predeclared regularization grid
# ------------------------------------------------------------

C_GRID = [
    0.001,
    0.003,
    0.01,
    0.03,
    0.1,
    0.3,
    1.0,
    3.0,
]

INNER_FOLDS = [
    1,
    2,
    3,
    4,
]


# ------------------------------------------------------------
# 4. Inner-CV hyperparameter selection
#
# IMPORTANT:
#   Each inner split computes normalization ONLY from that
#   inner-training subset.
# ------------------------------------------------------------

inner_rows = []

for rep in REPRESENTATIONS:

    for C in C_GRID:

        fold_metrics = []

        for held_fold in INNER_FOLDS:

            inner_train = train[
                train["fold"].astype(int)
                != held_fold
            ]

            inner_val = train[
                train["fold"].astype(int)
                == held_fold
            ]

            Xtr = build_representation(
                inner_train,
                rep,
            )

            ytr = inner_train[
                "label"
            ].to_numpy()

            Xva = build_representation(
                inner_val,
                rep,
            )

            yva = inner_val[
                "label"
            ].to_numpy()


            # -----------------------------------------------
            # Train-split-only standardization
            # -----------------------------------------------

            mu = Xtr.mean(
                axis=0
            )

            sd = Xtr.std(
                axis=0
            )

            sd[
                sd < 1e-6
            ] = 1.0

            Xtr = (
                Xtr - mu
            ) / sd

            Xva = (
                Xva - mu
            ) / sd


            clf = LogisticRegression(
                C=C,
                class_weight="balanced",
                max_iter=5000,
                random_state=1640,
            )

            clf.fit(
                Xtr,
                ytr,
            )

            p = clf.predict_proba(
                Xva
            )[:, 1]

            fold_metrics.append(
                (
                    average_precision_score(
                        yva,
                        p,
                    ),

                    roc_auc_score(
                        yva,
                        p,
                    ),
                )
            )


        aps = np.asarray(
            [
                x[0]
                for x in fold_metrics
            ]
        )

        aucs = np.asarray(
            [
                x[1]
                for x in fold_metrics
            ]
        )

        inner_rows.append(
            {
                "representation":
                    rep,

                "C":
                    C,

                "mean_AP":
                    float(
                        aps.mean()
                    ),

                "sd_AP":
                    float(
                        aps.std(
                            ddof=0
                        )
                    ),

                "mean_AUC":
                    float(
                        aucs.mean()
                    ),
            }
        )


inner_1112l = pd.DataFrame(
    inner_rows
)


# ------------------------------------------------------------
# 5. Select C independently for each representation
#
# Primary criterion: mean inner AP
# Tie-breaker: mean inner AUC
# ------------------------------------------------------------

best_rows = []

for rep in REPRESENTATIONS:

    sub = (
        inner_1112l[
            inner_1112l[
                "representation"
            ] == rep
        ]
        .sort_values(
            [
                "mean_AP",
                "mean_AUC",
            ],
            ascending=[
                False,
                False,
            ],
        )
    )

    best_rows.append(
        sub.iloc[0]
    )


best_1112l = pd.DataFrame(
    best_rows
).reset_index(
    drop=True
)


print(
    "\n========== INNER-CV SELECTION =========="
)

print(
    best_1112l[
        [
            "representation",
            "C",
            "mean_AP",
            "sd_AP",
            "mean_AUC",
        ]
    ]
    .round(6)
    .to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 6. Final outer fit on folds1–4, Fold0 evaluated ONCE
# ------------------------------------------------------------

ytr = train[
    "label"
].to_numpy()

yva = val[
    "label"
].to_numpy()

outer_rows = []
outer_probabilities = {}

for rep in REPRESENTATIONS:

    C = float(
        best_1112l.loc[
            best_1112l[
                "representation"
            ] == rep,
            "C",
        ].iloc[0]
    )

    Xtr = build_representation(
        train,
        rep,
    )

    Xva = build_representation(
        val,
        rep,
    )


    # --------------------------------------------------------
    # Outer-training-only standardization
    # --------------------------------------------------------

    mu = Xtr.mean(
        axis=0
    )

    sd = Xtr.std(
        axis=0
    )

    sd[
        sd < 1e-6
    ] = 1.0

    Xtr = (
        Xtr - mu
    ) / sd

    Xva = (
        Xva - mu
    ) / sd


    clf = LogisticRegression(
        C=C,
        class_weight="balanced",
        max_iter=5000,
        random_state=1640,
    )

    clf.fit(
        Xtr,
        ytr,
    )

    p = clf.predict_proba(
        Xva
    )[:, 1]

    outer_probabilities[
        rep
    ] = p

    ap = average_precision_score(
        yva,
        p,
    )

    auc = roc_auc_score(
        yva,
        p,
    )

    outer_rows.append(
        {
            "representation":
                rep,

            "dims":
                Xtr.shape[1],

            "selected_C":
                C,

            "Fold0_AP":
                ap,

            "PR_lift":
                ap
                / yva.mean(),

            "Fold0_AUC":
                auc,
        }
    )


outer_1112l = pd.DataFrame(
    outer_rows
)


print(
    "\n========== FOLD0 CLEAN MULTI-FRAME RESULTS =========="
)

print(
    outer_1112l
    .round(6)
    .to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 7. Delta relative to same-population X0 baseline
# ------------------------------------------------------------

baseline = (
    outer_1112l[
        outer_1112l[
            "representation"
        ] == "X0_current"
    ]
    .iloc[0]
)

delta = outer_1112l.copy()

delta[
    "delta_AP_vs_X0"
] = (
    delta["Fold0_AP"]
    - baseline["Fold0_AP"]
)

delta[
    "delta_AUC_vs_X0"
] = (
    delta["Fold0_AUC"]
    - baseline["Fold0_AUC"]
)


print(
    "\n========== DELTA VS X0 =========="
)

print(
    delta[
        [
            "representation",
            "delta_AP_vs_X0",
            "delta_AUC_vs_X0",
        ]
    ]
    .round(6)
    .to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 8. Operational Top-K on exactly the same Fold0 population
# ------------------------------------------------------------

topk_rows = []

for rep in REPRESENTATIONS:

    p = outer_probabilities[
        rep
    ]

    order = np.argsort(
        -p,
        kind="stable",
    )

    for K in [
        25,
        50,
        100,
        250,
        500,
    ]:

        idx = order[:K]

        tp = int(
            yva[idx].sum()
        )

        topk_rows.append(
            {
                "representation":
                    rep,

                "top_K":
                    K,

                "TP":
                    tp,

                "FP":
                    K - tp,

                "precision":
                    tp / K,

                "division_recall":
                    tp / yva.sum(),
            }
        )


topk_1112l = pd.DataFrame(
    topk_rows
)


print(
    "\n========== FOLD0 TOP-K =========="
)

print(
    topk_1112l
    .round(6)
    .to_string(
        index=False
    )
)

### 11.12 — Clean Multi-frame Historical-Context Capacity Audit

A leakage-free multi-frame capacity experiment was performed on the same
X2-common population:

- folds 1–4: 78,562 rows / 78 divisions
- Fold0: 18,504 rows / 21 divisions

Representations:
- X0: current parent-candidate joint representation
- X1: X0 + same-lineage q(t-1)
- X2: X1 + same-lineage q(t-2)
- X3: X2 + explicit signed temporal context differences

Adding frozen historical transformer contexts did not produce a consistent
improvement in division discrimination.

Fold0:
- X0 AP 0.014166, AUC 0.852415
- X1 AP 0.014897, AUC 0.854520
- X2 AP 0.013643, AUC 0.852335
- X3 AP 0.009104, AUC 0.827468

Although X1 showed a very small Fold0 gain, its inner-CV AP was lower than X0
(0.026922 vs 0.032193), and operational Top-K gains were inconsistent.
Longer history and explicit temporal differences degraded performance.

Decision:
FROZEN_HISTORICAL_CONTEXT_CONCAT_REJECTED.

Interpretation:
Historical information is not absent, but simply concatenating frozen
association-transformer states does not create a robust division-event
representation. The next hypothesis should operate before or inside the
representation learning process, using multi-frame morphology/context jointly.

In [ ]:
# ============================================================
# RESULT — 11.13C Frozen W2-vs-W4 Representation Perturbation
# Purpose:
#   Measure how much the SAME detected source/target nodes
#   change when the frozen TemporalUNet sees:
#
#       W2: [t, t+1]
#       W4: [t-2, t-1, t, t+1]
#
#   Detections and node coordinates are fixed to the validated
#   W2 replay, isolating temporal-context effects inside UNet.
#
# Changes files: NO
# Runs training: NO
# Runs inference: YES — one dataset only
# Uses GT: NO
# Keep after running: YES
# ============================================================

from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
import torch

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

CHECKPOINT = (
    PROJECT
    / "models/official_baseline/"
      "stage10_pilot_baseline_256/"
      "split_0/edge_predictor_best.pth"
)

sys.path.insert(
    0,
    str(PROJECT / "scripts"),
)

import run_stage11_multiframe_extract as runner

from replay_stage11_association import DatasetReplay
from train_unet_transformer_mps import extract_pos_features


# ------------------------------------------------------------
# 1. Setup
# ------------------------------------------------------------

dataset = "6bba_df673a83"

device = runner.resolve_device(
    "mps"
)

model, window_size, downsample = (
    runner.load_model(
        CHECKPOINT,
        device,
    )
)

model.eval()

assert window_size == 2
assert tuple(downsample) == (1, 4, 4)

replay = DatasetReplay(
    dataset,
    TRAIN_ROOT,
    model,
    window_size,
    downsample,
    device,
)

_load_frame = (
    DatasetReplay
    ._encode
    .__globals__[
        "_load_frame"
    ]
)

ds_arr = torch.tensor(
    downsample,
    dtype=torch.float32,
    device=device,
)


# ------------------------------------------------------------
# 2. Helper: original-view UNet encoding only
#
# No detection TTA is required here because detections are
# fixed to W2 replay outputs.
# ------------------------------------------------------------

def encode_original(frame_indices):

    imgs = torch.stack([
        _load_frame(
            replay.zarr,
            t,
            replay.target_shape,
            replay.downsample,
        )
        for t in frame_indices
    ])

    imgs = (
        (
            imgs - replay.q_low
        )
        / (
            replay.q_high
            - replay.q_low
            + 1e-6
        )
    ).clamp(0.0)

    imgs = (
        imgs
        .unsqueeze(0)
        .to(device)
    )

    with torch.no_grad():
        unet_out, _ = model.encode(
            imgs
        )

    return unet_out


# ------------------------------------------------------------
# 3. Helper: q/k/raw for fixed detected nodes
# ------------------------------------------------------------

def association_state(
    unet_src,
    unet_tgt,
    c_src,
    c_tgt,
):

    n_src = len(c_src)
    n_tgt = len(c_tgt)

    p_src = torch.from_numpy(
        c_src[:, 1:].astype(
            np.float32
        )
    ).unsqueeze(0).to(device)

    p_tgt = torch.from_numpy(
        c_tgt[:, 1:].astype(
            np.float32
        )
    ).unsqueeze(0).to(device)

    # Keep edge-transformer temporal positional semantics
    # identical to the original t -> t+1 pair.
    window_shape = (
        2,
        *replay.image_shape[1:],
    )

    src_rel = c_src.copy()
    tgt_rel = c_tgt.copy()

    src_rel[:, 0] = 0
    tgt_rel[:, 0] = 1

    pos_src = torch.from_numpy(
        extract_pos_features(
            src_rel,
            window_shape,
        )
    ).unsqueeze(0).to(device)

    pos_tgt = torch.from_numpy(
        extract_pos_features(
            tgt_rel,
            window_shape,
        )
    ).unsqueeze(0).to(device)

    mask_src = torch.ones(
        1,
        n_src,
        dtype=torch.bool,
        device=device,
    )

    mask_tgt = torch.ones(
        1,
        n_tgt,
        dtype=torch.bool,
        device=device,
    )

    feat_src = model._index_features(
        unet_src,
        p_src,
        mask_src,
    )

    feat_tgt = model._index_features(
        unet_tgt,
        p_tgt,
        mask_tgt,
    )

    bucket = []

    hook = (
        model.transformer.norm_out
        .register_forward_hook(
            lambda module, inp, out:
                bucket.append(
                    out.detach().cpu()
                )
        )
    )

    try:
        with torch.no_grad():
            raw = model.predict_edges(
                feat_src,
                feat_tgt,
                p_src * ds_arr,
                p_tgt * ds_arr,
                pos_src,
                pos_tgt,
                mask_src,
                mask_tgt,
            )[0]
    finally:
        hook.remove()

    assert len(bucket) == 2

    q = bucket[0][0].numpy()
    k = bucket[1][0].numpy()

    return (
        q,
        k,
        raw.detach().cpu().numpy(),
    )


# ------------------------------------------------------------
# 4. Compare W2 vs W4 over all valid t >= 2
# ------------------------------------------------------------

rows = []

t0 = time.perf_counter()

for t in range(
    2,
    int(replay.T) - 1,
):

    # Exact validated W2 detections.
    with torch.no_grad():
        c_src, c_tgt, _, _ = (
            replay.replay_pair(t)
        )

    if (
        len(c_src) == 0
        or len(c_tgt) < 2
    ):
        continue


    # W2 association feature context.
    w2 = encode_original(
        [t, t + 1]
    )

    q2, k2, raw2 = association_state(
        w2[:, 0],
        w2[:, 1],
        c_src,
        c_tgt,
    )


    # W4 context.
    w4 = encode_original(
        [
            t - 2,
            t - 1,
            t,
            t + 1,
        ]
    )

    q4, k4, raw4 = association_state(
        w4[:, 2],
        w4[:, 3],
        c_src,
        c_tgt,
    )


    # Same-node representation differences.
    q_delta = np.sqrt(
        np.mean(
            (q4 - q2) ** 2,
            axis=1,
        )
    )

    k_delta = np.sqrt(
        np.mean(
            (k4 - k2) ** 2,
            axis=1,
        )
    )


    # Top-1 target according to raw association logits.
    top2 = np.argmax(
        raw2,
        axis=1,
    )

    top4 = np.argmax(
        raw4,
        axis=1,
    )

    rows.append({
        "t":
            t,

        "n_src":
            len(c_src),

        "n_tgt":
            len(c_tgt),

        "q_rmse_mean":
            float(
                q_delta.mean()
            ),

        "q_rmse_median":
            float(
                np.median(
                    q_delta
                )
            ),

        "k_rmse_mean":
            float(
                k_delta.mean()
            ),

        "raw_rmse":
            float(
                np.sqrt(
                    np.mean(
                        (raw4 - raw2) ** 2
                    )
                )
            ),

        "top1_same_rate":
            float(
                np.mean(
                    top2 == top4
                )
            ),
    })


audit_1113c = pd.DataFrame(
    rows
)

elapsed = (
    time.perf_counter()
    - t0
)


# ------------------------------------------------------------
# 5. Summary
# ------------------------------------------------------------

print(
    "========== 11.13C W2-vs-W4 PERTURBATION =========="
)

print(
    "frame pairs:",
    len(audit_1113c),
)

print(
    f"elapsed    : {elapsed:.1f}s"
)


summary = pd.DataFrame({
    "metric": [
        "q_rmse_mean",
        "q_rmse_median",
        "k_rmse_mean",
        "raw_rmse",
        "top1_same_rate",
    ],

    "mean": [
        audit_1113c[
            "q_rmse_mean"
        ].mean(),

        audit_1113c[
            "q_rmse_median"
        ].mean(),

        audit_1113c[
            "k_rmse_mean"
        ].mean(),

        audit_1113c[
            "raw_rmse"
        ].mean(),

        audit_1113c[
            "top1_same_rate"
        ].mean(),
    ],

    "median": [
        audit_1113c[
            "q_rmse_mean"
        ].median(),

        audit_1113c[
            "q_rmse_median"
        ].median(),

        audit_1113c[
            "k_rmse_mean"
        ].median(),

        audit_1113c[
            "raw_rmse"
        ].median(),

        audit_1113c[
            "top1_same_rate"
        ].median(),
    ],
})

print(
    "\n========== SUMMARY =========="
)

print(
    summary
    .round(6)
    .to_string(index=False)
)


print(
    "\n========== FRAME-LEVEL QUANTILES =========="
)

print(
    audit_1113c[
        [
            "q_rmse_mean",
            "raw_rmse",
            "top1_same_rate",
        ]
    ]
    .quantile(
        [0, .25, .5, .75, 1]
    )
    .round(6)
    .to_string()
)

In [ ]:
# ============================================================
# RUN — 11.13D Enriched Frozen-W4 Feature Extraction
# Purpose:
#   Extract W4 temporally-mixed joint representations ONLY on
#   division-containing frames, using the exact same W2-detected
#   parent population as the persisted validated feature tables.
#
#   This is a cheap CAPACITY GATE before any expensive
#   natural-prior W4 extraction.
#
# Changes files: YES
#   - reports/stage11_division_probe/
#       stage11_w4_enriched.pkl
#
# Runs training: NO
# Runs inference: YES — division-containing frames only
# Uses GT: YES — only to recover the already-selected W2 parent
#                  row's source-node correspondence
# Keep after running: NO — delete after successful extraction
# ============================================================

from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
import torch

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

REPORT_DIR = (
    PROJECT
    / "reports/stage11_division_probe"
)

CHECKPOINT = (
    PROJECT
    / "models/official_baseline/"
      "stage10_pilot_baseline_256/"
      "split_0/edge_predictor_best.pth"
)

OUT = (
    REPORT_DIR
    / "stage11_w4_enriched.pkl"
)

sys.path.insert(
    0,
    str(PROJECT / "scripts"),
)

import stage11_multiframe_utils as mf
import run_stage11_multiframe_extract as runner

from replay_stage11_association import DatasetReplay
from train_unet_transformer_mps import extract_pos_features


# ------------------------------------------------------------
# 1. Load validated W2 natural-prior tables
# ------------------------------------------------------------

w2 = pd.concat(
    [
        pd.read_pickle(
            REPORT_DIR
            / "stage11_multiframe_folds1to4.pkl"
        ),
        pd.read_pickle(
            REPORT_DIR
            / "stage11_multiframe_fold0.pkl"
        ),
    ],
    ignore_index=True,
)

target = (
    w2[
        w2["division_frame"]
        & (w2["t"].astype(int) >= 2)
    ]
    .copy()
)

KEY = [
    "dataset",
    "t",
    "node_id",
]

assert not target.duplicated(KEY).any()

print(
    "========== 11.13D TARGET POPULATION =========="
)

print(
    "datasets :",
    target["dataset"].nunique(),
)

print(
    "frames   :",
    target[
        ["dataset", "t"]
    ].drop_duplicates().shape[0],
)

print(
    "rows     :",
    len(target),
)

print(
    "positives:",
    int(target["label"].sum()),
)


# ------------------------------------------------------------
# 2. Frozen model
# ------------------------------------------------------------

device = runner.resolve_device(
    "mps"
)

model, window_size, downsample = (
    runner.load_model(
        CHECKPOINT,
        device,
    )
)

model.eval()

assert window_size == 2
assert tuple(downsample) == (1, 4, 4)

downsample_arr = np.asarray(
    downsample,
    dtype=np.float32,
)


# ------------------------------------------------------------
# 3. Low-level frame loader
# ------------------------------------------------------------

_load_frame = (
    DatasetReplay
    ._encode
    .__globals__[
        "_load_frame"
    ]
)


def encode_w4(replay, t):
    """Original-view W4 UNet features for [t-2,t-1,t,t+1]."""

    indices = [
        t - 2,
        t - 1,
        t,
        t + 1,
    ]

    imgs = torch.stack(
        [
            _load_frame(
                replay.zarr,
                ti,
                replay.target_shape,
                replay.downsample,
            )
            for ti in indices
        ]
    )

    imgs = (
        (
            imgs - replay.q_low
        )
        / (
            replay.q_high
            - replay.q_low
            + 1e-6
        )
    ).clamp(0.0)

    imgs = (
        imgs
        .unsqueeze(0)
        .to(device)
    )

    with torch.no_grad():
        unet_out, _ = model.encode(
            imgs
        )

    return unet_out


# ------------------------------------------------------------
# 4. Association state using FIXED W2 detections
# ------------------------------------------------------------

def w4_association_state(
    replay,
    unet_out,
    c_src,
    c_tgt,
):

    n_src = len(c_src)
    n_tgt = len(c_tgt)

    p_src = torch.from_numpy(
        c_src[:, 1:].astype(np.float32)
    ).unsqueeze(0).to(device)

    p_tgt = torch.from_numpy(
        c_tgt[:, 1:].astype(np.float32)
    ).unsqueeze(0).to(device)

    # Keep the edge-transformer pair semantics identical
    # to the original t -> t+1 model.
    pair_shape = (
        2,
        *replay.image_shape[1:],
    )

    src_rel = c_src.copy()
    tgt_rel = c_tgt.copy()

    src_rel[:, 0] = 0
    tgt_rel[:, 0] = 1

    pos_src = torch.from_numpy(
        extract_pos_features(
            src_rel,
            pair_shape,
        )
    ).unsqueeze(0).to(device)

    pos_tgt = torch.from_numpy(
        extract_pos_features(
            tgt_rel,
            pair_shape,
        )
    ).unsqueeze(0).to(device)

    mask_src = torch.ones(
        1,
        n_src,
        dtype=torch.bool,
        device=device,
    )

    mask_tgt = torch.ones(
        1,
        n_tgt,
        dtype=torch.bool,
        device=device,
    )

    feat_src = model._index_features(
        unet_out[:, 2],
        p_src,
        mask_src,
    )

    feat_tgt = model._index_features(
        unet_out[:, 3],
        p_tgt,
        mask_tgt,
    )

    bucket = []

    hook = (
        model.transformer.norm_out
        .register_forward_hook(
            lambda module, inp, out:
                bucket.append(
                    out.detach().cpu()
                )
        )
    )

    try:
        with torch.no_grad():
            raw = model.predict_edges(
                feat_src,
                feat_tgt,
                p_src * replay.ds_arr_t,
                p_tgt * replay.ds_arr_t,
                pos_src,
                pos_tgt,
                mask_src,
                mask_tgt,
            )[0]
    finally:
        hook.remove()

    assert len(bucket) == 2

    q = bucket[0][0].numpy()
    k = bucket[1][0].numpy()

    raw_np = (
        raw.detach()
        .cpu()
        .numpy()
    )

    probs_np = (
        torch.softmax(
            raw,
            dim=0,
        )
        .detach()
        .cpu()
        .numpy()
    )

    return q, k, raw_np, probs_np


# ------------------------------------------------------------
# 5. Extract W4 features for exactly the W2 target rows
# ------------------------------------------------------------

rows = []

t0 = time.perf_counter()

for dataset, ds_target in target.groupby(
    "dataset",
    sort=True,
):

    fold = int(
        ds_target["fold"].iloc[0]
    )

    replay = DatasetReplay(
        dataset,
        TRAIN_ROOT,
        model,
        window_size,
        downsample,
        device,
    )

    gt = runner.load_graph(
        TRAIN_ROOT
        / f"{dataset}.geff"
    )

    gt_nodes = (
        gt.node_attrs(
            attr_keys=[
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
        .set_index("node_id")
    )

    for t, frame_rows in ds_target.groupby(
        "t",
        sort=True,
    ):

        t = int(t)

        # Exact validated W2 detections.
        with torch.no_grad():
            c_src, c_tgt, _, _ = (
                replay.replay_pair(t)
            )

        if (
            len(c_src) == 0
            or len(c_tgt) < 2
        ):
            raise RuntimeError(
                f"Unexpected missing W2 detections: "
                f"{dataset} t={t}"
            )

        # W4 changes ONLY representation context.
        unet_w4 = encode_w4(
            replay,
            t,
        )

        q, k, raw, probs = (
            w4_association_state(
                replay,
                unet_w4,
                c_src,
                c_tgt,
            )
        )

        # Recover the exact source detection corresponding to
        # each already-persisted W2 GT node row.
        frame_rows = (
            frame_rows
            .sort_values("node_id")
            .reset_index(drop=True)
        )

        node_ids = (
            frame_rows[
                "node_id"
            ]
            .astype(int)
            .to_numpy()
        )

        gt_xyz = (
            gt_nodes.loc[
                node_ids,
                ["z", "y", "x"],
            ]
            .to_numpy(
                dtype=np.float32
            )
        )

        pred_xyz = (
            c_src[:, 1:4]
            .astype(np.float32)
            * downsample_arr
        )

        matches = mf.match_gt_to_pred(
            gt_xyz,
            pred_xyz,
            mf.SCALE_UM,
        )

        assert len(matches) == len(
            frame_rows
        ), (
            f"W2 population rematch mismatch: "
            f"{dataset} t={t}, "
            f"{len(matches)} != {len(frame_rows)}"
        )

        for gi, pi, _ in matches:

            old = frame_rows.iloc[gi]

            j1, j2 = (
                mf.top2_by_raw_logit(
                    raw[pi]
                )
            )

            raw1 = float(
                raw[pi, j1]
            )

            raw2 = float(
                raw[pi, j2]
            )

            prob1 = float(
                probs[pi, j1]
            )

            prob2 = float(
                probs[pi, j2]
            )

            k1 = k[j1]
            k2 = k[j2]

            top1 = (
                c_tgt[
                    j1,
                    1:4,
                ]
                .astype(np.float32)
                .copy()
            )

            top2 = (
                c_tgt[
                    j2,
                    1:4,
                ]
                .astype(np.float32)
                .copy()
            )

            old_top1 = np.asarray(
                old["top1_xyz"],
                dtype=np.float32,
            )

            old_top2 = np.asarray(
                old["top2_xyz"],
                dtype=np.float32,
            )

            rows.append(
                {
                    "dataset":
                        dataset,

                    "fold":
                        fold,

                    "t":
                        t,

                    "node_id":
                        int(old["node_id"]),

                    "label":
                        int(old["label"]),

                    "parent_q":
                        q[pi]
                        .astype(np.float32)
                        .copy(),

                    "k_mean":
                        (
                            (k1 + k2) / 2.0
                        )
                        .astype(np.float32),

                    "k_absdiff":
                        np.abs(
                            k1 - k2
                        )
                        .astype(np.float32),

                    "score_features":
                        mf.score_features(
                            raw1,
                            raw2,
                            prob1,
                            prob2,
                        ),

                    "geometry":
                        mf.candidate_geometry(
                            c_src[
                                pi,
                                1:4,
                            ].astype(
                                np.float32
                            ),
                            top1,
                            top2,
                            downsample_arr,
                            mf.SCALE_UM,
                        ),

                    "top1_xyz":
                        top1,

                    "top2_xyz":
                        top2,

                    "top1_same_w2":
                        bool(
                            np.array_equal(
                                top1,
                                old_top1,
                            )
                        ),

                    "pair_same_w2":
                        bool(
                            np.array_equal(
                                top1,
                                old_top1,
                            )
                            and
                            np.array_equal(
                                top2,
                                old_top2,
                            )
                        ),
                }
            )

        del unet_w4


w4_enriched_1113d = pd.DataFrame(
    rows
)

elapsed = (
    time.perf_counter()
    - t0
)


# ------------------------------------------------------------
# 6. Exact row-population check
# ------------------------------------------------------------

w4_keys = set(
    map(
        tuple,
        w4_enriched_1113d[
            KEY
        ].to_numpy(),
    )
)

target_keys = set(
    map(
        tuple,
        target[
            KEY
        ].to_numpy(),
    )
)

assert w4_keys == target_keys


# ------------------------------------------------------------
# 7. Representation perturbation on matched rows
# ------------------------------------------------------------

aligned = target.merge(
    w4_enriched_1113d,
    on=KEY,
    suffixes=(
        "_w2",
        "_w4",
    ),
    validate="one_to_one",
)

q2 = np.stack(
    aligned[
        "parent_q_w2"
    ]
)

q4 = np.stack(
    aligned[
        "parent_q_w4"
    ]
)

q_rmse = np.sqrt(
    np.mean(
        (q4 - q2) ** 2,
        axis=1,
    )
)


# ------------------------------------------------------------
# 8. Persist + concise summary
# ------------------------------------------------------------

w4_enriched_1113d.to_pickle(
    OUT
)

print(
    "\n========== 11.13D W4 ENRICHED EXTRACTION =========="
)

print(
    "rows        :",
    len(w4_enriched_1113d),
)

print(
    "positives   :",
    int(
        w4_enriched_1113d[
            "label"
        ].sum()
    ),
)

print(
    "datasets    :",
    w4_enriched_1113d[
        "dataset"
    ].nunique(),
)

print(
    "frame pairs :",
    target[
        ["dataset", "t"]
    ].drop_duplicates().shape[0],
)

print(
    f"elapsed     : {elapsed / 60:.1f} min"
)

print(
    "q RMSE mean :",
    round(
        float(
            q_rmse.mean()
        ),
        6,
    ),
)

print(
    "q RMSE med  :",
    round(
        float(
            np.median(
                q_rmse
            )
        ),
        6,
    ),
)

print(
    "top1 same   :",
    round(
        float(
            w4_enriched_1113d[
                "top1_same_w2"
            ].mean()
        ),
        6,
    ),
)

print(
    "top2-pair same:",
    round(
        float(
            w4_enriched_1113d[
                "pair_same_w2"
            ].mean()
        ),
        6,
    ),
)

print(
    "written     :",
    OUT,
)

print(
    "\n========== 11.13D DECISION =========="
)

print(
    "W4_ENRICHED_EXTRACTION: PASS"
)

In [ ]:
# ============================================================
# RESULT — 11.13E Clean W2-vs-W4 Enriched Capacity Gate
# Purpose:
#   Compare frozen W2 and frozen W4 joint representations on
#   EXACTLY the same parent rows from true division frames.
#
# Scientific question:
#   Does internal W4 TemporalUNet context improve division-
#   parent discrimination (WHO), before paying for full
#   natural-prior W4 extraction?
#
# Protocol:
#   - Same dataset/t/node_id rows for W2 and W4
#   - C selected using folds 1–4 ONLY
#   - Fold0 evaluated once after selection
#   - Train-only standardization in every split
#
# Changes files: NO
# Runs training: YES — logistic probes only
# Runs inference: NO
# Uses GT: NO new GT access
# Keep after running: YES
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

REPORT_DIR = (
    PROJECT
    / "reports/stage11_division_probe"
)

W2_PATHS = [
    REPORT_DIR / "stage11_multiframe_folds1to4.pkl",
    REPORT_DIR / "stage11_multiframe_fold0.pkl",
]

W4_PATH = (
    REPORT_DIR
    / "stage11_w4_enriched.pkl"
)

KEY = [
    "dataset",
    "t",
    "node_id",
]


# ------------------------------------------------------------
# 1. Load exact common enriched population
# ------------------------------------------------------------

w2_all = pd.concat(
    [
        pd.read_pickle(p)
        for p in W2_PATHS
    ],
    ignore_index=True,
)

w4 = pd.read_pickle(
    W4_PATH
)

w2_target = (
    w2_all[
        w2_all["division_frame"]
        & (w2_all["t"].astype(int) >= 2)
    ]
    .copy()
)

assert not w2_target.duplicated(KEY).any()
assert not w4.duplicated(KEY).any()

assert set(
    map(tuple, w2_target[KEY].to_numpy())
) == set(
    map(tuple, w4[KEY].to_numpy())
)

paired = w2_target.merge(
    w4,
    on=KEY,
    suffixes=("_w2", "_w4"),
    validate="one_to_one",
)

assert len(paired) == 1291

assert np.array_equal(
    paired["label_w2"].to_numpy(),
    paired["label_w4"].to_numpy(),
)

# Fold is also required to agree exactly.
assert np.array_equal(
    paired["fold_w2"].astype(int).to_numpy(),
    paired["fold_w4"].astype(int).to_numpy(),
)

paired["fold"] = (
    paired["fold_w2"]
    .astype(int)
)

paired["label"] = (
    paired["label_w2"]
    .astype(int)
)


# ------------------------------------------------------------
# 2. Population summary
# ------------------------------------------------------------

population = (
    paired
    .groupby("fold")
    .agg(
        rows=("label", "size"),
        positives=("label", "sum"),
    )
    .reset_index()
)

population["prevalence"] = (
    population["positives"]
    / population["rows"]
)

print(
    "========== 11.13E COMMON ENRICHED POPULATION =========="
)

print(
    population
    .round(6)
    .to_string(index=False)
)

print(
    "\nTOTAL:",
    len(paired),
    "rows /",
    int(paired["label"].sum()),
    "positives",
)


# ------------------------------------------------------------
# 3. Exact 395-d joint representation
# ------------------------------------------------------------

def stack_col(df, col):
    return np.stack(
        df[col].to_numpy()
    ).astype(np.float64)


def build_X(df, version):

    suffix = (
        "_w2"
        if version == "W2"
        else "_w4"
    )

    return np.concatenate(
        [
            stack_col(
                df,
                f"parent_q{suffix}",
            ),
            stack_col(
                df,
                f"k_mean{suffix}",
            ),
            stack_col(
                df,
                f"k_absdiff{suffix}",
            ),
            stack_col(
                df,
                f"score_features{suffix}",
            ),
            stack_col(
                df,
                f"geometry{suffix}",
            ),
        ],
        axis=1,
    )


assert build_X(
    paired.iloc[:2],
    "W2",
).shape[1] == 395

assert build_X(
    paired.iloc[:2],
    "W4",
).shape[1] == 395


# ------------------------------------------------------------
# 4. Outer split
# ------------------------------------------------------------

train = paired[
    paired["fold"] != 0
].copy()

val = paired[
    paired["fold"] == 0
].copy()

assert set(
    train["fold"].unique()
) == {1, 2, 3, 4}

assert set(
    val["fold"].unique()
) == {0}

print(
    "\n========== OUTER SPLIT =========="
)

print(
    "train folds1–4:",
    len(train),
    "rows /",
    int(train["label"].sum()),
    "positives",
)

print(
    "Fold0:",
    len(val),
    "rows /",
    int(val["label"].sum()),
    "positives",
)


# ------------------------------------------------------------
# 5. Predeclared C grid
# ------------------------------------------------------------

C_GRID = [
    0.001,
    0.003,
    0.01,
    0.03,
    0.1,
    0.3,
    1.0,
    3.0,
    10.0,
]

VERSIONS = [
    "W2",
    "W4",
]

INNER_FOLDS = [
    1,
    2,
    3,
    4,
]


# ------------------------------------------------------------
# 6. Inner-CV selection
# ------------------------------------------------------------

inner_rows = []

for version in VERSIONS:

    for C in C_GRID:

        fold_rows = []

        for held_fold in INNER_FOLDS:

            tr = train[
                train["fold"] != held_fold
            ]

            va = train[
                train["fold"] == held_fold
            ]

            Xtr = build_X(
                tr,
                version,
            )

            Xva = build_X(
                va,
                version,
            )

            ytr = tr[
                "label"
            ].to_numpy()

            yva = va[
                "label"
            ].to_numpy()

            # Train-split-only normalization.
            mu = Xtr.mean(
                axis=0
            )

            sd = Xtr.std(
                axis=0
            )

            sd[
                sd < 1e-6
            ] = 1.0

            Xtr = (
                Xtr - mu
            ) / sd

            Xva = (
                Xva - mu
            ) / sd

            clf = LogisticRegression(
                C=C,
                class_weight="balanced",
                max_iter=5000,
                random_state=1640,
            )

            clf.fit(
                Xtr,
                ytr,
            )

            p = clf.predict_proba(
                Xva
            )[:, 1]

            fold_rows.append(
                {
                    "AP":
                        average_precision_score(
                            yva,
                            p,
                        ),

                    "AUC":
                        roc_auc_score(
                            yva,
                            p,
                        ),
                }
            )

        fm = pd.DataFrame(
            fold_rows
        )

        inner_rows.append(
            {
                "version":
                    version,

                "C":
                    C,

                "mean_AP":
                    fm["AP"].mean(),

                "sd_AP":
                    fm["AP"].std(
                        ddof=0
                    ),

                "mean_AUC":
                    fm["AUC"].mean(),
            }
        )


inner_1113e = pd.DataFrame(
    inner_rows
)

best_rows = []

for version in VERSIONS:

    sub = (
        inner_1113e[
            inner_1113e["version"]
            == version
        ]
        .sort_values(
            [
                "mean_AP",
                "mean_AUC",
            ],
            ascending=[
                False,
                False,
            ],
        )
    )

    best_rows.append(
        sub.iloc[0]
    )

best_1113e = pd.DataFrame(
    best_rows
).reset_index(drop=True)


print(
    "\n========== INNER-CV SELECTION =========="
)

print(
    best_1113e
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 7. Final Fold0 evaluation
# ------------------------------------------------------------

outer_rows = []
outer_prob = {}

ytr = train[
    "label"
].to_numpy()

yva = val[
    "label"
].to_numpy()

for version in VERSIONS:

    C = float(
        best_1113e.loc[
            best_1113e["version"]
            == version,
            "C",
        ].iloc[0]
    )

    Xtr = build_X(
        train,
        version,
    )

    Xva = build_X(
        val,
        version,
    )

    # Outer-train-only normalization.
    mu = Xtr.mean(
        axis=0
    )

    sd = Xtr.std(
        axis=0
    )

    sd[
        sd < 1e-6
    ] = 1.0

    Xtr = (
        Xtr - mu
    ) / sd

    Xva = (
        Xva - mu
    ) / sd

    clf = LogisticRegression(
        C=C,
        class_weight="balanced",
        max_iter=5000,
        random_state=1640,
    )

    clf.fit(
        Xtr,
        ytr,
    )

    p = clf.predict_proba(
        Xva
    )[:, 1]

    outer_prob[
        version
    ] = p

    ap = average_precision_score(
        yva,
        p,
    )

    auc = roc_auc_score(
        yva,
        p,
    )

    outer_rows.append(
        {
            "version":
                version,

            "selected_C":
                C,

            "Fold0_AP":
                ap,

            "PR_lift":
                ap
                / yva.mean(),

            "Fold0_AUC":
                auc,
        }
    )


outer_1113e = pd.DataFrame(
    outer_rows
)


print(
    "\n========== FOLD0 W2-vs-W4 RESULTS =========="
)

print(
    outer_1113e
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 8. Direct W4 - W2 delta
# ------------------------------------------------------------

w2_result = (
    outer_1113e[
        outer_1113e["version"]
        == "W2"
    ]
    .iloc[0]
)

w4_result = (
    outer_1113e[
        outer_1113e["version"]
        == "W4"
    ]
    .iloc[0]
)

print(
    "\n========== W4 DELTA =========="
)

print(
    "inner mean AP delta:",
    round(
        float(
            best_1113e.loc[
                best_1113e["version"]
                == "W4",
                "mean_AP",
            ].iloc[0]
            -
            best_1113e.loc[
                best_1113e["version"]
                == "W2",
                "mean_AP",
            ].iloc[0]
        ),
        6,
    ),
)

print(
    "inner mean AUC delta:",
    round(
        float(
            best_1113e.loc[
                best_1113e["version"]
                == "W4",
                "mean_AUC",
            ].iloc[0]
            -
            best_1113e.loc[
                best_1113e["version"]
                == "W2",
                "mean_AUC",
            ].iloc[0]
        ),
        6,
    ),
)

print(
    "Fold0 AP delta:",
    round(
        float(
            w4_result[
                "Fold0_AP"
            ]
            -
            w2_result[
                "Fold0_AP"
            ]
        ),
        6,
    ),
)

print(
    "Fold0 AUC delta:",
    round(
        float(
            w4_result[
                "Fold0_AUC"
            ]
            -
            w2_result[
                "Fold0_AUC"
            ]
        ),
        6,
    ),
)


# ------------------------------------------------------------
# 9. Fold0 Top-K
# ------------------------------------------------------------

topk_rows = []

for version in VERSIONS:

    p = outer_prob[
        version
    ]

    order = np.argsort(
        -p,
        kind="stable",
    )

    for K in [
        10,
        20,
        30,
        50,
        100,
    ]:

        K_eff = min(
            K,
            len(order),
        )

        idx = order[:K_eff]

        tp = int(
            yva[idx].sum()
        )

        topk_rows.append(
            {
                "version":
                    version,

                "top_K":
                    K_eff,

                "TP":
                    tp,

                "FP":
                    K_eff - tp,

                "precision":
                    tp / K_eff,

                "division_recall":
                    tp / yva.sum(),
            }
        )


topk_1113e = pd.DataFrame(
    topk_rows
)

print(
    "\n========== FOLD0 TOP-K =========="
)

print(
    topk_1113e
    .round(6)
    .to_string(index=False)
)

### 11.13 — Frozen Multi-frame TemporalUNet Audit

The frozen TemporalUNet contains genuine cross-frame self-attention and accepts
arbitrary temporal window lengths. W4 input produced measurable representation
changes while preserving association ranking almost completely.

However, on the clean enriched division-parent population, W4 consistently
underperformed W2:

- inner-CV AP: 0.447072 → 0.439269
- inner-CV AUC: 0.816160 → 0.811563
- Fold0 AP: 0.513386 → 0.493715
- Fold0 AUC: 0.839360 → 0.825920

Fold0 Top-K division retrieval was identical for W2 and W4 through Top-100.

Decision:
FROZEN_W4_NOT_PROMOTED.

Interpretation:
Increasing temporal context in the existing position-free frozen temporal
attention changes the representation but does not improve division
discrimination. Full natural-prior W4 extraction is not justified.

In [ ]:
# ============================================================
# RESULT — 11.14A Clean Nonlinear Joint-Division Decoder Audit
# Purpose:
#   Test whether the existing 395-d parent–candidate joint
#   representation contains nonlinear division information
#   that the linear logistic probe fails to exploit.
#
# Protocol:
#   - Natural-prior population
#   - folds 1–4 ONLY for model/config selection
#   - Fold0 evaluated once
#   - Logistic baseline and shallow HGB decoders
#   - No temporal-history features
#
# Changes files: NO
# Runs training: YES — lightweight classifiers only
# Runs inference: NO
# Uses GT: NO new GT access
# Keep after running: YES
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

REPORT_DIR = (
    PROJECT
    / "reports/stage11_division_probe"
)

train = pd.read_pickle(
    REPORT_DIR
    / "stage11_multiframe_folds1to4.pkl"
)

val = pd.read_pickle(
    REPORT_DIR
    / "stage11_multiframe_fold0.pkl"
)

assert set(
    train["fold"].astype(int).unique()
) == {1, 2, 3, 4}

assert set(
    val["fold"].astype(int).unique()
) == {0}

assert len(train) == 86128
assert int(train["label"].sum()) == 100

assert len(val) == 20635
assert int(val["label"].sum()) == 25


print(
    "========== NATURAL-PRIOR POPULATION =========="
)

print(
    "train:",
    len(train),
    "/",
    int(train["label"].sum()),
    "positives",
)

print(
    "Fold0:",
    len(val),
    "/",
    int(val["label"].sum()),
    "positives",
)

print(
    "Fold0 prevalence:",
    f"{val['label'].mean():.6f}",
)


# ------------------------------------------------------------
# 1. Exact current 395-d representation
# ------------------------------------------------------------

def stack_col(df, col):
    return np.stack(
        df[col].to_numpy()
    ).astype(np.float32)


def build_X(df):
    return np.concatenate(
        [
            stack_col(df, "parent_q"),
            stack_col(df, "k_mean"),
            stack_col(df, "k_absdiff"),
            stack_col(df, "score_features"),
            stack_col(df, "geometry"),
        ],
        axis=1,
    )


X_all_train = build_X(train)
X_fold0 = build_X(val)

assert X_all_train.shape == (86128, 395)
assert X_fold0.shape == (20635, 395)

y_all_train = train["label"].to_numpy()
y_fold0 = val["label"].to_numpy()


# ------------------------------------------------------------
# 2. Predeclared models
# ------------------------------------------------------------

LOGISTIC_C = [
    0.001,
    0.003,
    0.01,
    0.03,
]

HGB_CONFIGS = {
    "HGB_small": {
        "learning_rate": 0.05,
        "max_iter": 100,
        "max_leaf_nodes": 7,
        "max_depth": 3,
        "min_samples_leaf": 50,
        "l2_regularization": 2.0,
    },

    "HGB_medium": {
        "learning_rate": 0.05,
        "max_iter": 150,
        "max_leaf_nodes": 15,
        "max_depth": 4,
        "min_samples_leaf": 50,
        "l2_regularization": 3.0,
    },

    "HGB_strong_reg": {
        "learning_rate": 0.03,
        "max_iter": 150,
        "max_leaf_nodes": 7,
        "max_depth": 3,
        "min_samples_leaf": 100,
        "l2_regularization": 5.0,
    },
}

INNER_FOLDS = [1, 2, 3, 4]


# ------------------------------------------------------------
# 3. Balanced sample weights
# ------------------------------------------------------------

def balanced_weights(y):

    y = np.asarray(y)

    n_pos = int(y.sum())
    n_neg = len(y) - n_pos

    assert n_pos > 0
    assert n_neg > 0

    w = np.ones(
        len(y),
        dtype=np.float64,
    )

    w[y == 1] = (
        n_neg / n_pos
    )

    return w


# ------------------------------------------------------------
# 4. Inner CV — Logistic
# ------------------------------------------------------------

inner_rows = []

for C in LOGISTIC_C:

    fold_metrics = []

    for held in INNER_FOLDS:

        tr_mask = (
            train["fold"].astype(int).to_numpy()
            != held
        )

        va_mask = ~tr_mask

        Xtr = X_all_train[tr_mask].astype(
            np.float64
        )

        Xva = X_all_train[va_mask].astype(
            np.float64
        )

        ytr = y_all_train[tr_mask]
        yva = y_all_train[va_mask]

        mu = Xtr.mean(axis=0)
        sd = Xtr.std(axis=0)

        sd[sd < 1e-6] = 1.0

        Xtr = (Xtr - mu) / sd
        Xva = (Xva - mu) / sd

        clf = LogisticRegression(
            C=C,
            class_weight="balanced",
            max_iter=5000,
            random_state=1640,
        )

        clf.fit(
            Xtr,
            ytr,
        )

        p = clf.predict_proba(
            Xva
        )[:, 1]

        fold_metrics.append(
            (
                average_precision_score(
                    yva,
                    p,
                ),
                roc_auc_score(
                    yva,
                    p,
                ),
            )
        )

    arr = np.asarray(
        fold_metrics
    )

    inner_rows.append({
        "model": "Logistic",
        "config": f"C={C}",
        "mean_AP": arr[:, 0].mean(),
        "sd_AP": arr[:, 0].std(),
        "mean_AUC": arr[:, 1].mean(),
    })


# ------------------------------------------------------------
# 5. Inner CV — shallow nonlinear HGB
# ------------------------------------------------------------

for name, cfg in HGB_CONFIGS.items():

    fold_metrics = []

    for held in INNER_FOLDS:

        tr_mask = (
            train["fold"].astype(int).to_numpy()
            != held
        )

        va_mask = ~tr_mask

        Xtr = X_all_train[tr_mask]
        Xva = X_all_train[va_mask]

        ytr = y_all_train[tr_mask]
        yva = y_all_train[va_mask]

        clf = HistGradientBoostingClassifier(
            **cfg,
            early_stopping=False,
            random_state=1640,
        )

        clf.fit(
            Xtr,
            ytr,
            sample_weight=balanced_weights(
                ytr
            ),
        )

        p = clf.predict_proba(
            Xva
        )[:, 1]

        fold_metrics.append(
            (
                average_precision_score(
                    yva,
                    p,
                ),
                roc_auc_score(
                    yva,
                    p,
                ),
            )
        )

    arr = np.asarray(
        fold_metrics
    )

    inner_rows.append({
        "model": "HGB",
        "config": name,
        "mean_AP": arr[:, 0].mean(),
        "sd_AP": arr[:, 0].std(),
        "mean_AUC": arr[:, 1].mean(),
    })


inner_1114a = (
    pd.DataFrame(inner_rows)
    .sort_values(
        [
            "mean_AP",
            "mean_AUC",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)


print(
    "\n========== INNER-CV MODEL SELECTION =========="
)

print(
    inner_1114a
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 6. Select best Logistic and best HGB independently
# ------------------------------------------------------------

best_logistic = (
    inner_1114a[
        inner_1114a["model"]
        == "Logistic"
    ]
    .iloc[0]
)

best_hgb = (
    inner_1114a[
        inner_1114a["model"]
        == "HGB"
    ]
    .iloc[0]
)


# ------------------------------------------------------------
# 7. Final Logistic — folds1–4 → Fold0
# ------------------------------------------------------------

C_best = float(
    best_logistic["config"]
    .split("=")[1]
)

Xtr = X_all_train.astype(
    np.float64
)

Xva = X_fold0.astype(
    np.float64
)

mu = Xtr.mean(axis=0)
sd = Xtr.std(axis=0)

sd[sd < 1e-6] = 1.0

Xtr = (Xtr - mu) / sd
Xva = (Xva - mu) / sd

log_clf = LogisticRegression(
    C=C_best,
    class_weight="balanced",
    max_iter=5000,
    random_state=1640,
)

log_clf.fit(
    Xtr,
    y_all_train,
)

p_log = log_clf.predict_proba(
    Xva
)[:, 1]


# ------------------------------------------------------------
# 8. Final HGB — folds1–4 → Fold0
# ------------------------------------------------------------

hgb_name = str(
    best_hgb["config"]
)

hgb_clf = HistGradientBoostingClassifier(
    **HGB_CONFIGS[hgb_name],
    early_stopping=False,
    random_state=1640,
)

hgb_clf.fit(
    X_all_train,
    y_all_train,
    sample_weight=balanced_weights(
        y_all_train
    ),
)

p_hgb = hgb_clf.predict_proba(
    X_fold0
)[:, 1]


# ------------------------------------------------------------
# 9. Fold0 metrics
# ------------------------------------------------------------

outer_rows = []

for name, p in [
    ("Logistic", p_log),
    ("HGB", p_hgb),
]:

    ap = average_precision_score(
        y_fold0,
        p,
    )

    outer_rows.append({
        "model": name,
        "AP": ap,
        "PR_lift":
            ap / y_fold0.mean(),
        "AUC":
            roc_auc_score(
                y_fold0,
                p,
            ),
    })


outer_1114a = pd.DataFrame(
    outer_rows
)

print(
    "\n========== FOLD0 NONLINEAR DECODER RESULTS =========="
)

print(
    outer_1114a
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 10. Operational Top-K
# ------------------------------------------------------------

topk_rows = []

for name, p in [
    ("Logistic", p_log),
    ("HGB", p_hgb),
]:

    order = np.argsort(
        -p,
        kind="stable",
    )

    for K in [
        25,
        50,
        100,
        250,
        500,
    ]:

        idx = order[:K]

        tp = int(
            y_fold0[idx].sum()
        )

        topk_rows.append({
            "model": name,
            "top_K": K,
            "TP": tp,
            "FP": K - tp,
            "precision": tp / K,
            "division_recall":
                tp / y_fold0.sum(),
        })


topk_1114a = pd.DataFrame(
    topk_rows
)

print(
    "\n========== FOLD0 TOP-K =========="
)

print(
    topk_1114a
    .round(6)
    .to_string(index=False)
)

### 11.14 — Nonlinear Joint-Representation Decoder Audit

A shallow nonlinear decoder was tested on the natural-prior 395-d
parent-candidate representation.

Population:
- folds 1–4: 86,128 rows / 100 divisions
- Fold0: 20,635 rows / 25 divisions

Best inner-CV:
- Logistic AP 0.022663, AUC 0.840352
- HGB AP 0.006448, AUC 0.826067

Fold0:
- Logistic AP 0.026677, AUC 0.833161
- HGB AP 0.009292, AUC 0.796598

HGB also retrieved fewer true divisions throughout the operational Top-K range.

Decision:
SHALLOW_TABULAR_NONLINEAR_DECODER_REJECTED.

Interpretation:
The existing joint representation contains useful division-ranking signal,
but a shallow tree-based nonlinear decoder does not expose additional
actionable information. Together with the failed frozen historical-context
and frozen-W4 experiments, the next hypothesis should introduce explicitly
supervised ordered temporal morphology rather than further post-hoc decoding
of the existing association representation.

In [ ]:
# ============================================================
# RESULT — 11.15A Temporal-Morphology Field-of-View Audit
# Purpose:
#   Measure physical displacement around a current parent over:
#
#       t-2 -> t
#       t-1 -> t
#       t   -> t+1 child / daughters
#
#   This determines the spatial crop size for an explicitly
#   ordered [t-2, t-1, t, t+1] morphology classifier.
#
# Changes files: NO
# Runs training/inference: NO
# Uses GT: YES — geometry audit only
# Keep after running: YES
# ============================================================

from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

REPORT_DIR = (
    PROJECT
    / "reports/stage11_division_probe"
)

JOINED_PATH = (
    REPORT_DIR
    / "stage11_multiframe_lineage_join.pkl"
)

sys.path.insert(
    0,
    str(PROJECT / "scripts"),
)

import run_stage11_multiframe_extract as runner


# Raw-voxel physical spacing.
SCALE_UM = np.asarray(
    [1.625, 0.40625, 0.40625],
    dtype=np.float64,
)

# After (1,4,4) downsampling:
# z = y = x = 1.625 um.
DS_SPACING_UM = 1.625


# ------------------------------------------------------------
# 1. Population with complete two-step ancestry
# ------------------------------------------------------------

joined = pd.read_pickle(
    JOINED_PATH
)

common = joined[
    joined["has_q_prev2"]
].copy()

print(
    "========== 11.15A POPULATION =========="
)

print(
    "rows     :",
    len(common),
)

print(
    "positives:",
    int(common["label"].sum()),
)

print(
    "datasets :",
    common["dataset"].nunique(),
)


# ------------------------------------------------------------
# 2. Geometry helpers
# ------------------------------------------------------------

def dist_um(a, b):
    delta = (
        np.asarray(a, dtype=np.float64)
        - np.asarray(b, dtype=np.float64)
    ) * SCALE_UM

    return float(
        np.linalg.norm(delta)
    )


records = []


# ------------------------------------------------------------
# 3. Reconstruct GT coordinates / children
# ------------------------------------------------------------

for dataset, sub in common.groupby(
    "dataset",
    sort=True,
):

    gt = runner.load_graph(
        TRAIN_ROOT
        / f"{dataset}.geff"
    )

    nodes = (
        gt.node_attrs(
            attr_keys=[
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
    )

    edges = (
        gt.edge_attrs(
            attr_keys=[
                "source_id",
                "target_id",
            ]
        )
        .to_pandas()
    )

    xyz = {
        int(r.node_id):
            np.asarray(
                [r.z, r.y, r.x],
                dtype=np.float64,
            )
        for r in nodes.itertuples(
            index=False
        )
    }

    children = {}

    for e in edges.itertuples(
        index=False
    ):
        children.setdefault(
            int(e.source_id),
            [],
        ).append(
            int(e.target_id)
        )


    for r in sub.itertuples(
        index=False
    ):

        cur = int(r.node_id)
        p1 = int(r.prev1_node_id)
        p2 = int(r.prev2_node_id)

        child_ids = children.get(
            cur,
            [],
        )

        if len(child_ids) not in (1, 2):
            continue

        next_distances = [
            dist_um(
                xyz[cur],
                xyz[c],
            )
            for c in child_ids
        ]

        records.append({
            "dataset":
                dataset,

            "fold":
                int(r.fold),

            "label":
                int(r.label),

            "d_prev1_um":
                dist_um(
                    xyz[cur],
                    xyz[p1],
                ),

            "d_prev2_um":
                dist_um(
                    xyz[cur],
                    xyz[p2],
                ),

            "d_next_max_um":
                max(
                    next_distances
                ),
        })


fov_1115a = pd.DataFrame(
    records
)

assert len(fov_1115a) > 0


# ------------------------------------------------------------
# 4. Quantiles
# ------------------------------------------------------------

def summarize(df, label):

    qs = [
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
        1.00,
    ]

    rows = []

    for col in [
        "d_prev1_um",
        "d_prev2_um",
        "d_next_max_um",
    ]:

        values = df[col]

        row = {
            "population":
                label,

            "metric":
                col,

            "n":
                len(values),
        }

        for q in qs:
            row[
                f"q{int(q * 100):02d}"
            ] = values.quantile(q)

        rows.append(row)

    return rows


summary_rows = []

summary_rows += summarize(
    fov_1115a,
    "ALL",
)

summary_rows += summarize(
    fov_1115a[
        fov_1115a["label"] == 1
    ],
    "DIVISION",
)

summary_1115a = pd.DataFrame(
    summary_rows
)


print(
    "\n========== PHYSICAL DISPLACEMENT QUANTILES =========="
)

print(
    summary_1115a
    .round(3)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 5. Positive-class crop requirement
#
# Use the maximum observed positive displacement across both
# historical ancestry and future daughter position, then add
# two downsampled voxels of margin.
# ------------------------------------------------------------

positive = fov_1115a[
    fov_1115a["label"] == 1
]

required_um = float(
    max(
        positive["d_prev2_um"].max(),
        positive["d_next_max_um"].max(),
    )
)

radius_vox = int(
    np.ceil(
        required_um
        / DS_SPACING_UM
    )
) + 2

patch_size = (
    2 * radius_vox
    + 1
)


print(
    "\n========== POSITIVE FOV REQUIREMENT =========="
)

print(
    "max required radius:",
    round(
        required_um,
        3,
    ),
    "um",
)

print(
    "downsample spacing :",
    DS_SPACING_UM,
    "um",
)

print(
    "recommended radius:",
    radius_vox,
    "voxels",
)

print(
    "recommended patch :",
    f"{patch_size} x {patch_size} x {patch_size}",
)

print(
    "physical width     :",
    round(
        patch_size
        * DS_SPACING_UM,
        2,
    ),
    "um",
)

In [ ]:
# ============================================================
# PATCH — 11.15B Temporal Patch Boundary / Size Audit
# Purpose:
#   Re-run the patch-size boundary audit with continuous GT
#   centroids mapped to the nearest VALID downsampled voxel.
#
# Fix:
#   np.rint() can map a valid continuous boundary centroid
#   (e.g. 63.75) to integer index 64. Clamp rounded centers
#   to [0, shape-1] before computing padding.
#
# Changes files: NO
# Runs training/inference: NO
# Uses GT: YES — current parent coordinate only
# Keep after running: NO — delete after successful verification
# ============================================================

from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

REPORT_DIR = (
    PROJECT
    / "reports/stage11_division_probe"
)

sys.path.insert(
    0,
    str(PROJECT / "scripts"),
)

import run_stage11_multiframe_extract as runner


# ------------------------------------------------------------
# 1. Natural-prior W4-capable population
# ------------------------------------------------------------

base = pd.concat(
    [
        pd.read_pickle(
            REPORT_DIR
            / "stage11_multiframe_folds1to4.pkl"
        ),
        pd.read_pickle(
            REPORT_DIR
            / "stage11_multiframe_fold0.pkl"
        ),
    ],
    ignore_index=True,
)

base = base[
    base["t"].astype(int) >= 2
].copy()

print(
    "========== 11.15B W4 POPULATION =========="
)

print("rows     :", len(base))
print("positives:", int(base["label"].sum()))
print("datasets :", base["dataset"].nunique())


# ------------------------------------------------------------
# 2. Recover current GT coordinates
# ------------------------------------------------------------

parts = []

for dataset, sub in base.groupby(
    "dataset",
    sort=True,
):

    gt = runner.load_graph(
        TRAIN_ROOT
        / f"{dataset}.geff"
    )

    nodes = (
        gt.node_attrs(
            attr_keys=[
                "node_id",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
    )

    coords = sub[
        [
            "dataset",
            "fold",
            "t",
            "node_id",
            "label",
        ]
    ].merge(
        nodes,
        on="node_id",
        how="left",
        validate="many_to_one",
    )

    assert not coords[
        ["z", "y", "x"]
    ].isna().any().any()

    parts.append(coords)

centers = pd.concat(
    parts,
    ignore_index=True,
)


# ------------------------------------------------------------
# 3. Continuous -> nearest VALID downsampled voxel
# ------------------------------------------------------------

ds_xyz = np.column_stack(
    [
        centers["z"].to_numpy(),
        centers["y"].to_numpy() / 4.0,
        centers["x"].to_numpy() / 4.0,
    ]
)

DS_SHAPE = np.asarray(
    [64, 64, 64],
    dtype=int,
)

rounded = np.rint(
    ds_xyz
).astype(int)

# Record how often ordinary rounding crosses an array boundary.
would_clip = (
    (rounded < 0)
    |
    (rounded >= DS_SHAPE)
).any(axis=1)

centers_vox = np.clip(
    rounded,
    0,
    DS_SHAPE - 1,
)

assert (centers_vox >= 0).all()
assert (centers_vox < DS_SHAPE).all()

print(
    "\n========== CENTER MAPPING SANITY =========="
)

print(
    "rows requiring index clipping:",
    int(would_clip.sum()),
)

print(
    "clipping rate:",
    round(
        float(would_clip.mean()),
        6,
    ),
)

print(
    "continuous ds min:",
    np.round(
        ds_xyz.min(axis=0),
        4,
    ),
)

print(
    "continuous ds max:",
    np.round(
        ds_xyz.max(axis=0),
        4,
    ),
)


# ------------------------------------------------------------
# 4. Patch boundary audit
# ------------------------------------------------------------

DIVISION_MAX_RADIUS_UM = 13.529
SPACING_UM = 1.625

positive = (
    centers["label"].to_numpy()
    == 1
)

rows = []

for patch_size in [
    19,
    21,
    23,
]:

    radius = (
        patch_size - 1
    ) // 2

    low_pad = np.maximum(
        radius - centers_vox,
        0,
    )

    high_pad = np.maximum(
        centers_vox
        + radius
        - (DS_SHAPE - 1),
        0,
    )

    pad = (
        low_pad
        + high_pad
    )

    requires_padding = (
        pad > 0
    ).any(axis=1)

    physical_radius = (
        radius
        * SPACING_UM
    )

    rows.append(
        {
            "patch":
                patch_size,

            "radius_vox":
                radius,

            "radius_um":
                physical_radius,

            "margin_vs_div_max_um":
                physical_radius
                - DIVISION_MAX_RADIUS_UM,

            "all_padding_rate":
                requires_padding.mean(),

            "division_padding_rate":
                requires_padding[
                    positive
                ].mean(),

            "median_pad_vox_if_padded":
                float(
                    np.median(
                        pad[
                            requires_padding
                        ].sum(axis=1)
                    )
                )
                if requires_padding.any()
                else 0.0,

            "max_pad_vox":
                int(
                    pad.sum(axis=1).max()
                ),
        }
    )

boundary_1115b = pd.DataFrame(
    rows
)

print(
    "\n========== PATCH SIZE TRADEOFF =========="
)

print(
    boundary_1115b
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 5. Fold0 positive sanity
# ------------------------------------------------------------

fold0_pos = (
    centers["label"].eq(1)
    &
    centers["fold"].astype(int).eq(0)
)

print(
    "\n========== FOLD0 POSITIVES =========="
)

print(
    "W4-capable positives:",
    int(fold0_pos.sum()),
)


print(
    "\n========== 11.15B DECISION INPUT =========="
)

print(
    "PATCH_BOUNDARY_AUDIT: PASS"
)

In [ ]:
# ============================================================
# RESULT — 11.15C Patch-Boundary Confound Audit
# Purpose:
#   Quantify whether 19^3 boundary-padding status correlates
#   with division labels, and count the clean fully-in-bounds
#   population available for a first morphology capacity test.
#
# Changes files: NO
# Runs training/inference: NO
# Uses GT: YES — reuses current parent coordinates
# Keep after running: YES
# ============================================================

import numpy as np
import pandas as pd

PATCH = 19
RADIUS = 9
SHAPE = np.asarray([64, 64, 64])

low_pad = np.maximum(
    RADIUS - centers_vox,
    0,
)

high_pad = np.maximum(
    centers_vox + RADIUS - (SHAPE - 1),
    0,
)

requires_padding = (
    (low_pad + high_pad) > 0
).any(axis=1)

audit = centers[
    ["fold", "label"]
].copy()

audit["padding"] = requires_padding
audit["interior"] = ~requires_padding


# ------------------------------------------------------------
# Per-fold clean population
# ------------------------------------------------------------

rows = []

for fold in range(5):

    f = audit[
        audit["fold"].astype(int) == fold
    ]

    pos = f[
        f["label"] == 1
    ]

    neg = f[
        f["label"] == 0
    ]

    interior = f[
        f["interior"]
    ]

    rows.append({
        "fold":
            fold,

        "rows":
            len(f),

        "positives":
            int(f["label"].sum()),

        "padding_rate_all":
            f["padding"].mean(),

        "padding_rate_pos":
            pos["padding"].mean(),

        "padding_rate_neg":
            neg["padding"].mean(),

        "interior_rows":
            len(interior),

        "interior_pos":
            int(interior["label"].sum()),
    })


boundary_conf_1115c = pd.DataFrame(
    rows
)

print(
    "========== 11.15C BOUNDARY CONFOUND BY FOLD =========="
)

print(
    boundary_conf_1115c
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# Overall 2x2 relationship
# ------------------------------------------------------------

tab = pd.crosstab(
    audit["label"],
    audit["padding"],
)

print(
    "\n========== LABEL x PADDING =========="
)

print(tab)


pos_pad = audit.loc[
    audit["label"] == 1,
    "padding",
].mean()

neg_pad = audit.loc[
    audit["label"] == 0,
    "padding",
].mean()

print(
    "\npositive padding rate:",
    round(float(pos_pad), 6)
)

print(
    "negative padding rate:",
    round(float(neg_pad), 6)
)

print(
    "absolute gap:",
    round(float(pos_pad - neg_pad), 6)
)


# ------------------------------------------------------------
# Clean candidate population
# ------------------------------------------------------------

clean = audit[
    audit["interior"]
]

clean_train = clean[
    clean["fold"].astype(int) != 0
]

clean_fold0 = clean[
    clean["fold"].astype(int) == 0
]

print(
    "\n========== FULLY-IN-BOUNDS 19^3 POPULATION =========="
)

print(
    "train folds1–4:",
    len(clean_train),
    "rows /",
    int(clean_train["label"].sum()),
    "positives",
)

print(
    "Fold0:",
    len(clean_fold0),
    "rows /",
    int(clean_fold0["label"].sum()),
    "positives",
)

print(
    "\n========== 11.15C DECISION INPUT =========="
)

print(
    "BOUNDARY_CONFOUND_AUDIT: PASS"
)

### 11.15E — Preliminary Ordered-Morphology OOF Capacity Result

On the fully in-bounds folds1–4 population (38,014 parents / 57 divisions),
a small 3D CNN using ordered [t-2,t-1,t,t+1] image patches substantially
outperformed a current-frame-only CNN.

Macro OOF:
- T1 current: AP 0.023361, AUC 0.709174
- T4 ordered: AP 0.084196, AUC 0.849234

T4 improved AP and AUC in all four held-out folds.

This is strong preliminary evidence for temporal morphology signal, but is not
yet promoted because T1 and T4 differ in input-channel count and the first run
used mode-dependent initialization / negative-sampling seeds.

Next gate:
same-capacity 4-channel current-frame-repeat control with paired seeds.

### 11.15F — Controlled Ordered Temporal Morphology Validation

The preliminary temporal-morphology gain survived a same-capacity,
paired-seed control.

Control:
- T4-repeat-current: [I_t, I_t, I_t, I_t]
- T4-ordered: [I_{t-2}, I_{t-1}, I_t, I_{t+1}]

Both models used:
- identical 4-channel CNN architecture
- identical initialization within each paired comparison
- identical negative samples each epoch
- 3 independent seeds
- 4-fold OOF evaluation on folds 1–4 only

Results:
- Ordered AP won 10/12 paired seed×fold comparisons.
- Ordered AUC won 12/12.
- Mean paired AP delta: +0.050595.
- Mean paired AUC delta: +0.105962.
- Ordered macro AP exceeded repeat-current macro AP for all three seeds.

Decision:
ORDERED_TEMPORAL_MORPHOLOGY_SIGNAL_CONFIRMED.

Interpretation:
The improvement is attributable to information contained in the ordered
multi-frame image sequence rather than simply additional input channels or a
single favorable random initialization.

The remaining question is whether this signal is complementary to the existing
parent–candidate joint representation.

### 11.15G — Morphology–Association Complementarity

Ordered temporal morphology and the existing 395-d parent–candidate
representation were evaluated using strict folds1–4 OOF predictions.

Macro OOF:
- Joint representation: AP 0.035066, AUC 0.851632
- 3-seed temporal morphology ensemble: AP 0.056544, AUC 0.855667
- Fixed 50/50 rank fusion: AP 0.114610, AUC 0.901961

Fusion exceeded the best individual AP in 3/4 folds and the best individual
AUC in all 4 folds.

Mean joint–morphology rank correlation was only 0.402, indicating substantial
non-redundant information.

Decision:
COMPLEMENTARY_ORDERED_TEMPORAL_MORPHOLOGY_SIGNAL_CONFIRMED.

The Fold0 outer protocol is now frozen before any Fold0 morphology prediction.

### 11.15 — Ordered Temporal Morphology: Final Outer Validation

A teacher-forced ordered temporal morphology signal was identified and
validated under a locked outer protocol.

The morphology model receives four fixed-order local image patches:

[t-2, t-1, t, t+1]

around the current parent cell. A controlled same-capacity experiment showed
that ordered temporal images outperform repeated-current-frame inputs across
3 seeds and 4 OOF folds.

The morphology signal was complementary to the existing 395-d
parent-candidate association representation.

Folds1–4 OOF:
- joint AP: 0.035066
- morphology ensemble AP: 0.056544
- fixed equal-rank fusion AP: 0.114610
- fusion AUC: 0.901961
- joint–morph rank correlation: 0.402

Locked Fold0 outer test:
- joint AP: 0.010227
- morphology ensemble AP: 0.022829
- fixed equal-rank fusion AP: 0.115910
- fusion AUC: 0.907346
- joint–morph rank correlation: 0.428
- fusion AP improvement over best single: +0.093081

At Top-100 on Fold0:
- joint: 1/12 divisions
- morphology: 2/12
- fusion: 5/12

Decision:
COMPLEMENTARY_ORDERED_TEMPORAL_MORPHOLOGY_SIGNAL_CONFIRMED_ON_OUTER_FOLD.

Limitation:
This remains a teacher-forced capacity result. Patch centers are GT parent
centroids and only fully in-bounds 19^3 patches were evaluated. It is not yet
a deployable submission component.

Next requirement:
replace GT centers with actual detected-parent centers and extend the patch
extractor to boundary cases before any competition-score integration.

In [ ]:
# ============================================================
# RESULT — 11.16B Persisted-GEFF Detection Fidelity Audit
# Purpose:
#   Verify that existing Baseline256 det=.995 prediction GEFF
#   contains the SAME detected-node coordinates used by the
#   validated Stage11 feature extractor.
#
# Method:
#   Reconstruct GT -> predicted-node matches from the persisted
#   GEFF and compare per-row physical match distances against
#   Stage11's stored match_distance_um fingerprint.
#
#   Two matching populations are tested:
#     A. eligible Stage11 GT parents only
#     B. all GT nodes in the frame, then restrict to Stage11 rows
#
# Changes files: NO
# Runs training/inference: NO
# Uses GT: YES — matching audit only
# Keep after running: YES
# ============================================================

from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

PRED_ROOT = (
    PROJECT
    / "predictions/heqiuyan/"
      "stage10_infer_baseline_256_det995/"
      "split_0"
)

REPORT_DIR = (
    PROJECT
    / "reports/stage11_division_probe"
)

DATASET = "6bba_df673a83"

sys.path.insert(
    0,
    str(PROJECT / "scripts"),
)

import run_stage11_multiframe_extract as runner
import stage11_multiframe_utils as mf


# ------------------------------------------------------------
# 1. Load historical Stage11 rows
# ------------------------------------------------------------

stage11 = pd.read_pickle(
    REPORT_DIR
    / "stage11_multiframe_fold0.pkl"
)

stage11 = (
    stage11[
        stage11["dataset"] == DATASET
    ]
    .copy()
    .sort_values(
        ["t", "node_id"]
    )
    .reset_index(drop=True)
)

assert len(stage11) == 645
assert int(stage11["label"].sum()) == 4


# ------------------------------------------------------------
# 2. Load GT + persisted prediction graph
# ------------------------------------------------------------

gt = runner.load_graph(
    TRAIN_ROOT
    / f"{DATASET}.geff"
)

pred = runner.load_graph(
    PRED_ROOT
    / f"{DATASET}.geff"
)

gt_nodes = (
    gt.node_attrs(
        attr_keys=[
            "node_id",
            "t",
            "z",
            "y",
            "x",
        ]
    )
    .to_pandas()
)

pred_nodes = (
    pred.node_attrs(
        attr_keys=[
            "node_id",
            "t",
            "z",
            "y",
            "x",
        ]
    )
    .to_pandas()
)


# ------------------------------------------------------------
# 3. Physical-distance helper
# ------------------------------------------------------------

SCALE_UM = np.asarray(
    [1.625, 0.40625, 0.40625],
    dtype=np.float64,
)


def physical_distance_um(a, b):

    d = (
        np.asarray(a, dtype=np.float64)
        -
        np.asarray(b, dtype=np.float64)
    ) * SCALE_UM

    return float(
        np.linalg.norm(d)
    )


# ------------------------------------------------------------
# 4. Reconstruct one matching strategy
# ------------------------------------------------------------

def reconstruct(strategy):

    reconstructed = []

    for t, target_rows in stage11.groupby(
        "t",
        sort=True,
    ):

        t = int(t)

        pred_frame = (
            pred_nodes[
                pred_nodes["t"].astype(int)
                == t
            ]
            .copy()
            .reset_index(drop=True)
        )

        assert len(pred_frame) > 0

        pred_xyz = pred_frame[
            ["z", "y", "x"]
        ].to_numpy(
            dtype=np.float32
        )


        if strategy == "eligible_only":

            gt_frame = (
                target_rows[
                    ["node_id"]
                ]
                .merge(
                    gt_nodes[
                        [
                            "node_id",
                            "z",
                            "y",
                            "x",
                        ]
                    ],
                    on="node_id",
                    how="left",
                    validate="one_to_one",
                )
                .reset_index(drop=True)
            )

        elif strategy == "all_gt":

            gt_frame = (
                gt_nodes[
                    gt_nodes["t"].astype(int)
                    == t
                ]
                .copy()
                .reset_index(drop=True)
            )

        else:
            raise ValueError(strategy)


        gt_xyz = gt_frame[
            ["z", "y", "x"]
        ].to_numpy(
            dtype=np.float32
        )


        matches = mf.match_gt_to_pred(
            gt_xyz,
            pred_xyz,
            SCALE_UM,
        )


        matched_by_gt_id = {}

        for gi, pi, dist in matches:

            gt_id = int(
                gt_frame.iloc[int(gi)][
                    "node_id"
                ]
            )

            p = pred_frame.iloc[
                int(pi)
            ]

            matched_by_gt_id[
                gt_id
            ] = {
                "pred_node_id":
                    int(p["node_id"]),

                "pred_z":
                    float(p["z"]),

                "pred_y":
                    float(p["y"]),

                "pred_x":
                    float(p["x"]),

                "distance_um":
                    float(dist),
            }


        for r in target_rows.itertuples(
            index=False
        ):

            node_id = int(r.node_id)

            m = matched_by_gt_id.get(
                node_id
            )

            if m is None:

                reconstructed.append({
                    "dataset":
                        DATASET,

                    "t":
                        t,

                    "node_id":
                        node_id,

                    "strategy":
                        strategy,

                    "matched":
                        False,

                    "stored_distance_um":
                        float(
                            r.match_distance_um
                        ),

                    "reconstructed_distance_um":
                        np.nan,

                    "distance_delta_um":
                        np.nan,
                })

                continue


            # Independent check that helper distance agrees
            # with physical coordinates.
            gt_row = gt_nodes[
                gt_nodes["node_id"]
                == node_id
            ].iloc[0]

            gt_xyz_one = np.asarray(
                [
                    gt_row["z"],
                    gt_row["y"],
                    gt_row["x"],
                ],
                dtype=np.float64,
            )

            pred_xyz_one = np.asarray(
                [
                    m["pred_z"],
                    m["pred_y"],
                    m["pred_x"],
                ],
                dtype=np.float64,
            )

            direct_dist = (
                physical_distance_um(
                    gt_xyz_one,
                    pred_xyz_one,
                )
            )

            assert np.isclose(
                direct_dist,
                m["distance_um"],
                atol=1e-6,
                rtol=0,
            )


            stored = float(
                r.match_distance_um
            )

            reconstructed.append({
                "dataset":
                    DATASET,

                "t":
                    t,

                "node_id":
                    node_id,

                "strategy":
                    strategy,

                "matched":
                    True,

                "pred_node_id":
                    m["pred_node_id"],

                "pred_z":
                    m["pred_z"],

                "pred_y":
                    m["pred_y"],

                "pred_x":
                    m["pred_x"],

                "stored_distance_um":
                    stored,

                "reconstructed_distance_um":
                    direct_dist,

                "distance_delta_um":
                    direct_dist - stored,
            })


    return pd.DataFrame(
        reconstructed
    )


# ------------------------------------------------------------
# 5. Test both plausible Stage11 matching semantics
# ------------------------------------------------------------

audit_parts = []

for strategy in [
    "eligible_only",
    "all_gt",
]:

    audit_parts.append(
        reconstruct(strategy)
    )

audit_1116b = pd.concat(
    audit_parts,
    ignore_index=True,
)


# ------------------------------------------------------------
# 6. Fidelity summary
# ------------------------------------------------------------

summary_rows = []

for strategy, sub in audit_1116b.groupby(
    "strategy",
    sort=False,
):

    matched = sub[
        sub["matched"]
    ]

    abs_delta = (
        matched[
            "distance_delta_um"
        ]
        .abs()
    )

    summary_rows.append({
        "strategy":
            strategy,

        "rows":
            len(sub),

        "matched":
            int(
                sub["matched"].sum()
            ),

        "missing":
            int(
                (~sub["matched"]).sum()
            ),

        "max_abs_delta_um":
            float(
                abs_delta.max()
            )
            if len(abs_delta)
            else np.nan,

        "median_abs_delta_um":
            float(
                abs_delta.median()
            )
            if len(abs_delta)
            else np.nan,

        "exact_1e6":
            int(
                (
                    abs_delta <= 1e-6
                ).sum()
            ),

        "exact_1e5":
            int(
                (
                    abs_delta <= 1e-5
                ).sum()
            ),
    })


summary_1116b = pd.DataFrame(
    summary_rows
)


print(
    "========== 11.16B MATCHING FIDELITY =========="
)

print(
    summary_1116b
    .round(9)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 7. Identify exact reproducing strategy
# ------------------------------------------------------------

exact = summary_1116b[
    (summary_1116b["matched"] == 645)
    &
    (summary_1116b["missing"] == 0)
    &
    (
        summary_1116b[
            "max_abs_delta_um"
        ] <= 1e-5
    )
]

print(
    "\n========== EXACT REPRODUCTION =========="
)

if len(exact) == 1:

    winning_strategy = str(
        exact.iloc[0]["strategy"]
    )

    print(
        "strategy:",
        winning_strategy,
    )

    winning = audit_1116b[
        audit_1116b["strategy"]
        == winning_strategy
    ].copy()

    print(
        "rows:",
        len(winning),
    )

    print(
        "max distance delta:",
        f"{winning['distance_delta_um'].abs().max():.9f}",
        "um",
    )

    print(
        "unique predicted nodes:",
        winning[
            "pred_node_id"
        ].nunique(),
    )

    print(
        "\n========== SAMPLE RECOVERED CENTERS =========="
    )

    print(
        winning[
            [
                "t",
                "node_id",
                "pred_node_id",
                "pred_z",
                "pred_y",
                "pred_x",
                "stored_distance_um",
                "reconstructed_distance_um",
            ]
        ]
        .head(10)
        .round(6)
        .to_string(index=False)
    )

    print(
        "\n========== 11.16B DECISION =========="
    )

    print(
        "PERSISTED_PREDICTION_COORDINATES: EXACT_COMPATIBLE"
    )

elif len(exact) > 1:

    print(
        "Multiple strategies reproduce exactly:",
        exact["strategy"].tolist(),
    )

    print(
        "\n========== 11.16B DECISION =========="
    )

    print(
        "PERSISTED_PREDICTION_COORDINATES: COMPATIBLE_MATCHING_SEMANTICS_AMBIGUOUS"
    )

else:

    print(
        "No strategy reproduces Stage11 match distances exactly."
    )

    print(
        "\nWorst discrepancies:"
    )

    print(
        audit_1116b[
            audit_1116b["matched"]
        ]
        .assign(
            abs_delta=lambda x:
                x["distance_delta_um"].abs()
        )
        .sort_values(
            "abs_delta",
            ascending=False,
        )
        [
            [
                "strategy",
                "t",
                "node_id",
                "stored_distance_um",
                "reconstructed_distance_um",
                "abs_delta",
            ]
        ]
        .head(10)
        .round(6)
        .to_string(index=False)
    )

    print(
        "\n========== 11.16B DECISION =========="
    )

    print(
        "PERSISTED_PREDICTION_COORDINATES: NOT_YET_PROVEN"
    )

### 11.16 — Predicted-Center Deployment Robustness

Persisted Baseline256 Fold0 detections exactly reproduced the Stage11
GT-to-detection match-distance fingerprint.

Detected-center displacement was moderate:
- 73.2% of parents had L∞ shift <= 1 downsampled voxel
- 93.5% had L∞ shift <= 2 voxels
- median 19^3 patch overlap was 89.75%

On the exact same 8,216 Fold0 rows / 12 divisions:

- joint AP: 0.010427
- GT-centered morphology AP: 0.023243
- detected-centered morphology AP: 0.036267
- GT-centered fusion AP: 0.116707
- detected-centered fusion AP: 0.072802

Detected-center morphology lost AUC but retained strong extreme-tail utility.
Detected-center fusion remained substantially stronger than joint-only and
recovered 4/12 divisions at Top-100 and 7/12 at Top-250.

Decision:
CENTER_SHIFT_ROBUST_ENOUGH_FOR_NEXT_DEPLOYMENT_GATE.

Localization robustness is not fully solved, but regeneration of folds1–4
detector predictions is not yet justified. Boundary-safe morphology is the
next deployment bottleneck.

### 11.17B — Reflection-Safe Temporal Morphology Validation

Ordered temporal morphology remained strongly predictive after extending the
training population to all W4-capable parents using reflection-padded 19^3
patches.

Population:
- 84,815 parents / 98 divisions
- boundary: 46,801 / 41
- interior: 38,014 / 57

Controlled comparison:
- T4-repeat-current: [I_t, I_t, I_t, I_t]
- T4-ordered: [I_{t-2}, I_{t-1}, I_t, I_{t+1}]
- identical architecture, initialization, negative samples
- 3 seeds × 4 OOF folds

Results:
- ALL: ordered AP won 12/12; AUC won 12/12
- BOUNDARY: ordered AP won 12/12; AUC won 11/12
- ALL mean paired AP delta: +0.078667
- ALL mean paired AUC delta: +0.102648
- BOUNDARY mean paired AP delta: +0.091647
- BOUNDARY mean paired AUC delta: +0.083223

Decision:
REFLECTION_SAFE_ORDERED_TEMPORAL_SIGNAL_CONFIRMED.

### 11.17C — Full-Population Morphology–Joint Complementarity

On all 84,815 W4-capable folds1–4 parents:

- joint macro AP: 0.021809
- reflection-safe morphology macro AP: 0.089271
- fixed equal-rank fusion macro AP: 0.069395

Fusion exceeded the best individual AP in only 2/4 folds.

On the boundary-only population:
- joint macro AP: 0.030196
- morphology macro AP: 0.104943
- fusion macro AP: 0.050069
- fusion exceeded the best single AP in 0/4 folds.

Joint–morphology rank correlation increased to 0.483.

Decision:
FULL_POPULATION_EQUAL_RANK_FUSION_REJECTED.

Reflection-safe ordered morphology remains strongly supported, but the
previous 50/50 joint fusion should not be promoted to the full deployment
population.

### 11.17E — Deployable Morphology Outer Validation

The reflection-safe ordered temporal morphology model was evaluated on the
untouched Fold0 deployment population using real Baseline256 detected-parent
centers.

Training:
- folds 1–4
- 84,815 parents / 98 divisions
- reflection-safe 19^3 patches

Outer Fold0:
- 20,324 parents / 25 divisions
- real detected centers
- 11,844 boundary parents / 12 divisions

Results:
- ALL: AP 0.102345, AUC 0.903519, PR lift 83.20x
- BOUNDARY: AP 0.065326, AUC 0.891639, PR lift 64.48x
- INTERIOR: AP 0.153014, AUC 0.912402

Tail recovery:
- Top-10: 2/25
- Top-25: 4/25
- Top-50: 6/25
- Top-100: 8/25
- Top-250: 10/25
- Top-500: 12/25

Three-seed rank correlations were approximately 0.90.

Decision:
DEPLOYABLE_REFLECTION_SAFE_TEMPORAL_MORPHOLOGY_CONFIRMED.

The remaining problem is no longer division-event detectability.
The next problem is converting high morphology scores into correct
daughter-edge proposals without introducing excessive false divisions.

In [ ]:
# ============================================================
# RESULT — 11.18A OOF Division-Event Budget Frontier
# Purpose:
#   Calibrate a deployment-scale proposal budget using ONLY
#   folds1–4 OOF morphology predictions.
#
#   Evaluate Top-K parent-frame proposals PER DATASET.
#   No Fold0 information is used for threshold/budget selection.
#
# Changes files: NO
# Runs training/inference: NO
# Uses GT: YES — OOF evaluation labels only
# Keep after running: YES
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

PROJECT = Path("/Users/heqiuyan/Desktop/biohub-cell-tracking")
RPT = PROJECT / "reports/stage11_division_probe"

PATH = RPT / "stage11_reflect_temporal_control_oof.pkl"

df = pd.read_pickle(PATH).copy()

SEEDS = [1640, 1641, 1642]
FOLDS = [1, 2, 3, 4]

pred_cols = [
    f"p_T4_ordered_{s}"
    for s in SEEDS
]

assert len(df) == 84815
assert int(df["label"].sum()) == 98


# ------------------------------------------------------------
# 1. Same fold-normalized rank ensemble used in Stage11.17
# ------------------------------------------------------------

rank_cols = []

for col in pred_cols:

    rcol = f"rank_{col}"
    rank_cols.append(rcol)

    df[rcol] = (
        df.groupby("fold")[col]
        .rank(
            pct=True,
            method="average",
        )
    )

df["morph_score"] = (
    df[rank_cols].mean(axis=1)
)

y = df["label"].to_numpy(np.int64)

print("========== 11.18A OOF POPULATION ==========")
print("rows      :", len(df))
print("positives :", int(y.sum()))
print("datasets  :", df["dataset"].nunique())
print("prevalence:", f"{y.mean():.8f}")
print(
    "micro AP  :",
    f"{average_precision_score(y, df['morph_score']):.6f}",
)


# ------------------------------------------------------------
# 2. How many true divisions exist per dataset?
# ------------------------------------------------------------

pos_per_dataset = (
    df.groupby("dataset")["label"]
    .sum()
    .astype(int)
)

dist = (
    pos_per_dataset
    .value_counts()
    .sort_index()
    .rename_axis("true_divisions")
    .rename("datasets")
)

print(
    "\n========== TRUE DIVISIONS PER DATASET =========="
)

print(dist.to_string())


# ------------------------------------------------------------
# 3. Deterministic rank inside each dataset
# ------------------------------------------------------------

df = df.sort_values(
    ["dataset", "morph_score", "t", "node_id"],
    ascending=[True, False, True, True],
).copy()

df["rank_in_dataset"] = (
    df.groupby("dataset")
    .cumcount()
    + 1
)


# ------------------------------------------------------------
# 4. Top-K event proposal frontier
# ------------------------------------------------------------

rows = []

for K in [1, 2, 3, 5, 8, 10, 15, 20]:

    selected = df[
        df["rank_in_dataset"] <= K
    ]

    tp = int(selected["label"].sum())
    n = len(selected)

    rows.append({
        "K_per_dataset": K,
        "proposals": n,
        "TP": tp,
        "FP": n - tp,
        "precision": tp / n,
        "recall": tp / 98,
        "lift_vs_prevalence":
            (tp / n)
            / (98 / 84815),
    })


frontier = pd.DataFrame(rows)

print(
    "\n========== TOP-K PER-DATASET FRONTIER =========="
)

print(
    frontier
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 5. Stability by fold for realistic low budgets
# ------------------------------------------------------------

fold_rows = []

for K in [1, 2, 3, 5]:

    selected = df[
        df["rank_in_dataset"] <= K
    ]

    for fold in FOLDS:

        total_pos = int(
            df.loc[
                df["fold"] == fold,
                "label",
            ].sum()
        )

        sub = selected[
            selected["fold"] == fold
        ]

        tp = int(sub["label"].sum())

        fold_rows.append({
            "K_per_dataset": K,
            "fold": fold,
            "proposals": len(sub),
            "TP": tp,
            "precision":
                tp / len(sub),
            "recall":
                tp / total_pos,
        })


fold_frontier = pd.DataFrame(
    fold_rows
)

print(
    "\n========== LOW-BUDGET STABILITY BY FOLD =========="
)

print(
    fold_frontier
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 6. Positive score concentration
# ------------------------------------------------------------

positive_rank = df.loc[
    df["label"] == 1,
    "rank_in_dataset",
]

print(
    "\n========== TRUE-DIVISION DATASET RANK =========="
)

print(
    "median:",
    float(positive_rank.median()),
)

print(
    "q25/q50/q75/q90:",
    np.round(
        positive_rank.quantile(
            [0.25, 0.50, 0.75, 0.90]
        ).to_numpy(),
        3,
    ),
)

print(
    "\n========== 11.18A DECISION INPUT =========="
)

print(
    "OOF_EVENT_BUDGET_FRONTIER: PASS"
)

In [ ]:
# ============================================================
# RESULT — 11.18B OOF WHEN→WHO Candidate-Pair Ceiling
# Purpose:
#   Measure whether Stage11 top1/top2 association candidates
#   contain BOTH GT daughters when morphology correctly identifies
#   a division parent-frame.
#
#   Also compute the strict actionable ceiling for the OOF
#   morphology Top-K-per-dataset proposal budgets.
#
# Changes files: NO
# Runs training/inference: NO
# Uses GT: YES — folds1–4 audit only
# Fold0 touched: NO
# Keep after running: YES
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import tracksdata as td

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

RPT = (
    PROJECT
    / "reports/stage11_division_probe"
)

FEATURE_PATH = (
    RPT
    / "stage11_multiframe_folds1to4.pkl"
)

MORPH_PATH = (
    RPT
    / "stage11_reflect_temporal_control_oof.pkl"
)

import sys
sys.path.insert(
    0,
    str(PROJECT / "scripts"),
)

import run_stage11_multiframe_extract as runner


SEEDS = [1640, 1641, 1642]

MAX_DIST_UM = 7.0

# After downsample=(1,4,4), all three axes are 1.625 µm.
DS_SPACING_UM = 1.625


# ------------------------------------------------------------
# 1. Reconstruct the exact OOF morphology ranking
# ------------------------------------------------------------

features = pd.read_pickle(
    FEATURE_PATH
)

features = features[
    features["t"].astype(int) >= 2
].copy()

morph = pd.read_pickle(
    MORPH_PATH
).copy()

KEY = [
    "dataset",
    "t",
    "node_id",
]

PRED_COLS = [
    f"p_T4_ordered_{s}"
    for s in SEEDS
]

df = features.merge(
    morph[
        KEY + PRED_COLS
    ],
    on=KEY,
    validate="one_to_one",
)

assert len(df) == 84815
assert int(df["label"].sum()) == 98


rank_cols = []

for col in PRED_COLS:

    rcol = f"rank_{col}"

    df[rcol] = (
        df.groupby("fold")[col]
        .rank(
            pct=True,
            method="average",
        )
    )

    rank_cols.append(rcol)


df["morph_score"] = (
    df[rank_cols]
    .mean(axis=1)
)


df = df.sort_values(
    [
        "dataset",
        "morph_score",
        "t",
        "node_id",
    ],
    ascending=[
        True,
        False,
        True,
        True,
    ],
).copy()

df["rank_in_dataset"] = (
    df.groupby("dataset")
    .cumcount()
    + 1
)


# ------------------------------------------------------------
# 2. GT daughter coordinates for the 98 division parents
# ------------------------------------------------------------

positives = df[
    df["label"] == 1
].copy()

assert len(positives) == 98

records = []

SOURCE = td.DEFAULT_ATTR_KEYS.EDGE_SOURCE
TARGET = td.DEFAULT_ATTR_KEYS.EDGE_TARGET
NODE_ID = td.DEFAULT_ATTR_KEYS.NODE_ID


for dataset, sub in positives.groupby(
    "dataset",
    sort=True,
):

    gt = runner.load_graph(
        TRAIN_ROOT
        / f"{dataset}.geff"
    )

    nodes = (
        gt.node_attrs(
            attr_keys=[
                NODE_ID,
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
    )

    edges = (
        gt.edge_attrs()
        .select(
            SOURCE,
            TARGET,
        )
        .to_pandas()
    )

    node_xyz = (
        nodes.set_index(NODE_ID)[
            ["z", "y", "x"]
        ]
    )

    outgoing = (
        edges.groupby(SOURCE)[TARGET]
        .apply(list)
        .to_dict()
    )


    for row in sub.itertuples(
        index=False
    ):

        parent_id = int(
            row.node_id
        )

        daughters = outgoing.get(
            parent_id,
            [],
        )

        assert len(daughters) == 2, (
            dataset,
            parent_id,
            daughters,
        )

        dxyz = (
            node_xyz.loc[
                daughters
            ]
            .to_numpy(
                dtype=np.float64
            )
        )

        # Original grid -> downsample grid.
        dxyz_ds = (
            dxyz
            /
            np.asarray(
                [1.0, 4.0, 4.0]
            )
        )

        records.append({
            "dataset":
                dataset,

            "t":
                int(row.t),

            "node_id":
                parent_id,

            "daughter1_xyz_ds":
                dxyz_ds[0],

            "daughter2_xyz_ds":
                dxyz_ds[1],
        })


daughter_df = pd.DataFrame(
    records
)

assert len(daughter_df) == 98


# ------------------------------------------------------------
# 3. Exact 2x2 candidate↔daughter bipartite test
# ------------------------------------------------------------

positives = positives.merge(
    daughter_df,
    on=KEY,
    validate="one_to_one",
)


def as_xyz(x):

    arr = np.asarray(
        x,
        dtype=np.float64,
    ).reshape(-1)

    assert arr.shape == (3,)

    return arr


pair_rows = []

for row in positives.itertuples(
    index=False
):

    c1 = as_xyz(
        row.top1_xyz
    )

    c2 = as_xyz(
        row.top2_xyz
    )

    d1 = as_xyz(
        row.daughter1_xyz_ds
    )

    d2 = as_xyz(
        row.daughter2_xyz_ds
    )


    dist = np.asarray([
        [
            np.linalg.norm(c1 - d1)
            * DS_SPACING_UM,

            np.linalg.norm(c1 - d2)
            * DS_SPACING_UM,
        ],
        [
            np.linalg.norm(c2 - d1)
            * DS_SPACING_UM,

            np.linalg.norm(c2 - d2)
            * DS_SPACING_UM,
        ],
    ])


    # Two possible bijections:
    #
    # c1→d1, c2→d2
    # c1→d2, c2→d1

    valid_direct = (
        dist[0, 0] <= MAX_DIST_UM
        and
        dist[1, 1] <= MAX_DIST_UM
    )

    valid_cross = (
        dist[0, 1] <= MAX_DIST_UM
        and
        dist[1, 0] <= MAX_DIST_UM
    )

    pair_ok = (
        valid_direct
        or
        valid_cross
    )

    any_ok = bool(
        (dist <= MAX_DIST_UM).any()
    )


    if pair_ok:
        matched_daughters = 2
    elif any_ok:
        matched_daughters = 1
    else:
        matched_daughters = 0


    best_pair_max_dist = min(
        max(
            dist[0, 0],
            dist[1, 1],
        ),
        max(
            dist[0, 1],
            dist[1, 0],
        ),
    )


    pair_rows.append({
        "dataset":
            row.dataset,

        "t":
            int(row.t),

        "node_id":
            int(row.node_id),

        "fold":
            int(row.fold),

        "rank_in_dataset":
            int(row.rank_in_dataset),

        "matched_daughters":
            matched_daughters,

        "pair_recoverable":
            pair_ok,

        "best_pair_max_dist_um":
            best_pair_max_dist,
    })


audit = pd.DataFrame(
    pair_rows
)

assert len(audit) == 98


# ------------------------------------------------------------
# 4. Global WHO ceiling
# ------------------------------------------------------------

print(
    "========== 11.18B GLOBAL WHO CEILING =========="
)

print(
    "true divisions:",
    len(audit),
)

print(
    ">=1 daughter covered:",
    int(
        (
            audit[
                "matched_daughters"
            ] >= 1
        ).sum()
    ),
    "/ 98",
)

print(
    "both daughters covered:",
    int(
        audit[
            "pair_recoverable"
        ].sum()
    ),
    "/ 98",
)

print(
    "pair coverage:",
    round(
        float(
            audit[
                "pair_recoverable"
            ].mean()
        ),
        6,
    ),
)

print(
    "best-pair max-distance "
    "q50/q75/q90/q95:",
    np.round(
        audit[
            "best_pair_max_dist_um"
        ]
        .quantile(
            [
                0.50,
                0.75,
                0.90,
                0.95,
            ]
        )
        .to_numpy(),
        3,
    ),
)


# ------------------------------------------------------------
# 5. Fold stability
# ------------------------------------------------------------

fold_rows = []

for fold in [
    1,
    2,
    3,
    4,
]:

    sub = audit[
        audit["fold"] == fold
    ]

    fold_rows.append({
        "fold":
            fold,

        "divisions":
            len(sub),

        "pair_recoverable":
            int(
                sub[
                    "pair_recoverable"
                ].sum()
            ),

        "pair_coverage":
            float(
                sub[
                    "pair_recoverable"
                ].mean()
            ),
    })


fold_summary = pd.DataFrame(
    fold_rows
)

print(
    "\n========== WHO CEILING BY FOLD =========="
)

print(
    fold_summary
    .round(6)
    .to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 6. Morphology WHEN + top1/top2 WHO strict ceiling
# ------------------------------------------------------------

budget_rows = []

for K in [
    1,
    2,
    3,
    5,
    8,
    10,
    15,
    20,
]:

    proposals = df[
        df["rank_in_dataset"] <= K
    ]

    event_tp = int(
        proposals[
            "label"
        ].sum()
    )

    selected_true = audit[
        audit[
            "rank_in_dataset"
        ] <= K
    ]

    actionable_tp = int(
        selected_true[
            "pair_recoverable"
        ].sum()
    )

    n_prop = len(
        proposals
    )

    budget_rows.append({
        "K_per_dataset":
            K,

        "proposals":
            n_prop,

        "event_TP":
            event_tp,

        "pair_recoverable_TP":
            actionable_tp,

        "event_precision":
            event_tp / n_prop,

        "actionable_precision_ceiling":
            actionable_tp / n_prop,

        "event_recall":
            event_tp / 98,

        "actionable_division_recall_ceiling":
            actionable_tp / 98,

        "WHO_success_given_correct_WHEN":
            (
                actionable_tp
                / event_tp
                if event_tp > 0
                else np.nan
            ),
    })


budget = pd.DataFrame(
    budget_rows
)

print(
    "\n========== WHEN→WHO BUDGET CEILING =========="
)

print(
    budget
    .round(6)
    .to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 7. Low-budget fold stability
# ------------------------------------------------------------

rows = []

for K in [
    1,
    2,
    3,
    5,
]:

    for fold in [
        1,
        2,
        3,
        4,
    ]:

        sub = audit[
            (
                audit[
                    "rank_in_dataset"
                ] <= K
            )
            &
            (
                audit["fold"]
                == fold
            )
        ]

        rows.append({
            "K_per_dataset":
                K,

            "fold":
                fold,

            "correct_WHEN":
                len(sub),

            "pair_recoverable":
                int(
                    sub[
                        "pair_recoverable"
                    ].sum()
                ),
        })


stability = pd.DataFrame(
    rows
)

print(
    "\n========== LOW-BUDGET WHO STABILITY =========="
)

print(
    stability.to_string(
        index=False
    )
)


print(
    "\n========== 11.18B DECISION INPUT =========="
)

print(
    "OOF_WHEN_WHO_PAIR_CEILING: PASS"
)

### 11.18B — WHEN→WHO Top-2 Candidate Ceiling

Across 98 folds1–4 OOF true division events:

- at least one GT daughter was covered by top1/top2 in 95/98 cases
- both daughters were covered in only 52/98 cases (53.1%)
- pair coverage varied substantially across folds:
  - Fold1: 56.0%
  - Fold2: 28.0%
  - Fold3: 69.6%
  - Fold4: 60.0%

For correctly identified high-morphology WHEN events, top1/top2 WHO success was
only approximately 52–63%.

Decision:
TOP2_PAIR_WHO_NOT_PROMOTED.

The dominant unresolved question is second-daughter recovery:
whether the missing daughter is detected but ranked below top2, or absent from
the detector candidate set entirely.

### 11.18C — Second-Daughter Bottleneck Decomposition

Across 98 folds1–4 OOF true divisions:

- both daughters detected: 86/98 (87.8%)
- detector-limited: 12/98 (12.2%)

Conditional on both daughters being detected:

- RAW @2: 52/86 = 60.5%
- RAW @5: 81/86 = 94.2%
- PROB @2: 61/86 = 70.9%
- PROB @3: 75/86 = 87.2%
- PROB @5: 83/86 = 96.5%

PROB ranking consistently dominated RAW ranking in the low-K regime.

Decision:
SECOND_DAUGHTER_ASSOCIATION_RANKING_IS_THE_PRIMARY_BOTTLENECK.

A bounded PROB Top-5 daughter search nearly saturates the detector ceiling.
Detector modification is not justified at this stage.

### 11.18D — Ungated Additive Second-Edge Policy

A directly executable additive policy was tested using morphology-ranked
parent-frames and the highest available PROB-ranked second child while
preserving the existing first edge.

The ungated policy was too aggressive:

- WHEN Top-1: 3 correct / 47 actions (6.38%)
- WHEN Top-2: 5 / 94 (5.32%)
- WHEN Top-3: 7 / 144 (4.86%)
- WHEN Top-5: 8 / 246 (3.25%)

However, association confidence strongly separated correct and incorrect
actions:

- correct candidate probability median: 0.381
- false/wrong median: 0.059
- false/wrong q75: 0.168
- all 8 correct second edges were PROB rank 2 or 3

Decision:
UNGATED_ADDITIVE_SECOND_EDGE_POLICY_REJECTED.

A bounded PROB Top-3 search with an association-confidence gate is the next
policy candidate.

In [ ]:
# ============================================================
# RESULT — 11.18E Confidence-Gated Additive Policy Frontier
# Purpose:
#   Using the existing folds1–4 OOF additive-policy replay,
#   find non-dominated operating points over:
#
#     WHEN_K        = morphology Top-K per dataset
#     WHO_rank_cap  = PROB rank <= 2 or <= 3
#     prob_threshold
#
#   No scalar utility is imposed yet. We retain the TP↑ / FP↓
#   Pareto frontier for exact official-metric replay next.
#
# Changes files: NO
# Runs training/inference: NO
# Uses GT: YES — folds1–4 OOF evaluation only
# Fold0 touched: NO
# Keep after running: YES
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

RPT = (
    PROJECT
    / "reports/stage11_division_probe"
)

AUDIT_PATH = (
    RPT
    / "stage11_additive_second_edge_tail.pkl"
)

FEATURE_PATH = (
    RPT
    / "stage11_multiframe_folds1to4.pkl"
)

audit = pd.read_pickle(
    AUDIT_PATH
).copy()

features = pd.read_pickle(
    FEATURE_PATH
)

features = features[
    features["t"].astype(int) >= 2
]

assert len(audit) == 515
assert int(audit["label"].sum()) == 31


# ------------------------------------------------------------
# 1. True zero-division datasets from FULL OOF population
# ------------------------------------------------------------

divisions_per_dataset = (
    features.groupby("dataset")["label"]
    .sum()
)

zero_div_datasets = set(
    divisions_per_dataset[
        divisions_per_dataset == 0
    ].index
)

assert len(zero_div_datasets) == 50

print(
    "========== 11.18E POPULATION =========="
)

print("tail rows          :", len(audit))
print("tail true events   :", int(audit["label"].sum()))
print("all true divisions :", int(features["label"].sum()))
print("zero-div datasets  :", len(zero_div_datasets))


# ------------------------------------------------------------
# 2. Full policy grid
# ------------------------------------------------------------

WHEN_KS = [1, 2, 3, 5]
RANK_CAPS = [2, 3]

THRESHOLDS = [
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
]

rows = []

for when_k in WHEN_KS:

    for rank_cap in RANK_CAPS:

        for threshold in THRESHOLDS:

            selected = audit[
                audit["action"]
                &
                (audit["rank_in_dataset"] <= when_k)
                &
                (audit["candidate_rank"] <= rank_cap)
                &
                (audit["candidate_prob"] >= threshold)
            ].copy()

            tp = int(
                selected["final_pair_correct"].sum()
            )

            fp = len(selected) - tp

            false_event = int(
                (
                    selected["label"] == 0
                ).sum()
            )

            wrong_true = int(
                (
                    (selected["label"] == 1)
                    &
                    (~selected["final_pair_correct"])
                ).sum()
            )

            zero_ds_actions = int(
                selected["dataset"]
                .isin(zero_div_datasets)
                .sum()
            )

            rows.append({
                "WHEN_K": when_k,
                "rank_cap": rank_cap,
                "prob_threshold": threshold,
                "actions": len(selected),
                "TP": tp,
                "FP": fp,
                "precision":
                    tp / len(selected)
                    if len(selected)
                    else np.nan,
                "recall_all_divisions":
                    tp / 98,
                "false_event_actions":
                    false_event,
                "wrong_true_actions":
                    wrong_true,
                "zero_div_dataset_actions":
                    zero_ds_actions,
                "datasets_acted":
                    selected["dataset"].nunique(),
            })


grid = pd.DataFrame(rows)


# ------------------------------------------------------------
# 3. TP↑ / FP↓ Pareto frontier
# ------------------------------------------------------------

pareto_mask = []

for i, row in grid.iterrows():

    dominated = (
        (grid["TP"] >= row["TP"])
        &
        (grid["FP"] <= row["FP"])
        &
        (
            (grid["TP"] > row["TP"])
            |
            (grid["FP"] < row["FP"])
        )
    ).any()

    pareto_mask.append(
        not dominated
    )


pareto = (
    grid.loc[pareto_mask]
    .drop_duplicates(
        subset=["TP", "FP"],
        keep="first",
    )
    .sort_values(
        ["FP", "TP"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

pareto["config_id"] = [
    f"P{i+1}"
    for i in range(len(pareto))
]

print(
    "\n========== TP/FP PARETO FRONTIER =========="
)

print(
    pareto[
        [
            "config_id",
            "WHEN_K",
            "rank_cap",
            "prob_threshold",
            "actions",
            "TP",
            "FP",
            "precision",
            "recall_all_divisions",
            "zero_div_dataset_actions",
        ]
    ]
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 4. Association threshold frontier with minimal WHO cap=3
# ------------------------------------------------------------

rank3 = grid[
    grid["rank_cap"] == 3
].copy()

print(
    "\n========== PROB TOP-3 FRONTIER =========="
)

print(
    rank3[
        [
            "WHEN_K",
            "prob_threshold",
            "actions",
            "TP",
            "FP",
            "precision",
            "zero_div_dataset_actions",
        ]
    ]
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 5. Fold stability of Pareto configurations
# ------------------------------------------------------------

fold_rows = []

for row in pareto.itertuples(index=False):

    selected = audit[
        audit["action"]
        &
        (audit["rank_in_dataset"] <= row.WHEN_K)
        &
        (audit["candidate_rank"] <= row.rank_cap)
        &
        (audit["candidate_prob"] >= row.prob_threshold)
    ]

    for fold in [1, 2, 3, 4]:

        s = selected[
            selected["fold"] == fold
        ]

        tp = int(
            s["final_pair_correct"].sum()
        )

        fold_rows.append({
            "config_id": row.config_id,
            "fold": fold,
            "actions": len(s),
            "TP": tp,
            "FP": len(s) - tp,
            "precision":
                tp / len(s)
                if len(s)
                else np.nan,
        })


fold_summary = pd.DataFrame(
    fold_rows
)

print(
    "\n========== PARETO FOLD STABILITY =========="
)

print(
    fold_summary
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 6. Compact operational shortlist
# ------------------------------------------------------------

shortlist = pareto[
    (pareto["TP"] >= 2)
    &
    (pareto["precision"] >= 0.15)
].copy()

print(
    "\n========== OFFICIAL-METRIC REPLAY SHORTLIST =========="
)

if len(shortlist):

    print(
        shortlist[
            [
                "config_id",
                "WHEN_K",
                "rank_cap",
                "prob_threshold",
                "actions",
                "TP",
                "FP",
                "precision",
                "zero_div_dataset_actions",
            ]
        ]
        .round(6)
        .to_string(index=False)
    )

else:
    print(
        "NONE meeting TP>=2 and precision>=0.15"
    )


print(
    "\n========== 11.18E DECISION INPUT =========="
)

print(
    "CONFIDENCE_GATED_POLICY_FRONTIER: PASS"
)

### 11.18E — Confidence-Gated Additive Policy Frontier

Association-confidence gating substantially improved the precision of the
additive second-edge policy, but no configuration was promoted from proxy
statistics alone.

Pareto operating points included:

- P2: 1 TP / 1 FP, precision 50.0%
- P3: 2 TP / 4 FP, precision 33.3%
- P4: 4 TP / 14 FP, precision 22.2%
- P5: 5 TP / 23 FP, precision 17.9%

However, fold stability remained weak and several policies still produced
actions in truly zero-division datasets.

Decision:
CONFIDENCE_GATING_HELPFUL_BUT_NOT_YET_PROMOTED.

Further threshold tuning is stopped. The next gate is exact official-metric
replay on complete prediction graphs.

In [ ]:
# ============================================================
# PATCH — 11.19E-P1 Correct Eligible-Only Center Recovery
# Purpose:
#   Re-run folds1–4 detected-center recovery using the now
#   confirmed Stage11 matching contract:
#
#       match ONLY Stage11 eligible parent rows per frame.
#
#   Then persist the exact 84,815-row detected-center table and
#   finish the localization/boundary census that 11.19E was
#   intended to produce.
#
# Changes files: YES
#   - reports/stage11_division_probe/
#       stage11_folds1to4_predicted_centers.pkl
#
# Runs training/inference: NO
# Uses GT: YES — matching audit only
# Fold0 touched: NO
# Keep after running: NO — delete after successful verification
# ============================================================

from pathlib import Path
import sys
import numpy as np
import pandas as pd

PROJECT = Path("/Users/heqiuyan/Desktop/biohub-cell-tracking")

TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

PRED_ROOT = (
    PROJECT
    / "predictions/heqiuyan/"
      "stage11_fullgraph_baseline256_folds1to4_det995/"
      "split_0"
)

RPT = PROJECT / "reports/stage11_division_probe"

FEATURE_PATH = RPT / "stage11_multiframe_folds1to4.pkl"
OUT = RPT / "stage11_folds1to4_predicted_centers.pkl"

sys.path.insert(0, str(PROJECT / "scripts"))

import run_stage11_multiframe_extract as runner
import stage11_multiframe_utils as mf

SCALE_UM = np.asarray(
    [1.625, 0.40625, 0.40625],
    dtype=np.float64,
)

RADIUS = 9


# ------------------------------------------------------------
# 1. Exact eligible W4 population
# ------------------------------------------------------------

stage11 = pd.read_pickle(FEATURE_PATH)

stage11 = (
    stage11[
        stage11["t"].astype(int) >= 2
    ]
    .reset_index(drop=True)
)

assert len(stage11) == 84815
assert int(stage11["label"].sum()) == 98
assert stage11["dataset"].nunique() == 103


# ------------------------------------------------------------
# 2. Eligible-only matching
# ------------------------------------------------------------

records = []

datasets = list(
    stage11.groupby("dataset", sort=True)
)

for di, (dataset, ds_rows) in enumerate(
    datasets,
    start=1,
):

    gt = runner.load_graph(
        TRAIN_ROOT / f"{dataset}.geff"
    )

    pred = runner.load_graph(
        PRED_ROOT / f"{dataset}.geff"
    )

    gt_nodes = (
        gt.node_attrs(
            attr_keys=[
                "node_id", "t",
                "z", "y", "x",
            ]
        )
        .to_pandas()
    )

    pred_nodes = (
        pred.node_attrs(
            attr_keys=[
                "node_id", "t",
                "z", "y", "x",
            ]
        )
        .to_pandas()
    )

    for t, target in ds_rows.groupby(
        "t",
        sort=True,
    ):

        t = int(t)

        # CRITICAL: only Stage11 eligible parents participate.
        g = (
            target[
                [
                    "node_id",
                    "fold",
                    "label",
                    "match_distance_um",
                ]
            ]
            .merge(
                gt_nodes[
                    gt_nodes["t"].astype(int) == t
                ][
                    ["node_id", "z", "y", "x"]
                ],
                on="node_id",
                validate="one_to_one",
            )
            .reset_index(drop=True)
        )

        p = (
            pred_nodes[
                pred_nodes["t"].astype(int) == t
            ]
            .reset_index(drop=True)
        )

        matches = mf.match_gt_to_pred(
            g[["z", "y", "x"]].to_numpy(np.float32),
            p[["z", "y", "x"]].to_numpy(np.float32),
            SCALE_UM,
        )

        assert len(matches) == len(g)

        for gi, pi, dist in matches:

            gr = g.iloc[int(gi)]
            pr = p.iloc[int(pi)]

            records.append({
                "dataset": dataset,
                "fold": int(gr["fold"]),
                "t": t,
                "node_id": int(gr["node_id"]),
                "label": int(gr["label"]),

                "stored_distance_um":
                    float(gr["match_distance_um"]),

                "reconstructed_distance_um":
                    float(dist),

                "pred_node_id":
                    int(pr["node_id"]),

                "pred_z": float(pr["z"]),
                "pred_y": float(pr["y"]),
                "pred_x": float(pr["x"]),

                "z": float(gr["z"]),
                "y": float(gr["y"]),
                "x": float(gr["x"]),
            })

    if di == 1 or di % 10 == 0 or di == len(datasets):
        print(f"{di:3d}/{len(datasets)} datasets")


centers = pd.DataFrame(records)

assert len(centers) == 84815
assert int(centers["label"].sum()) == 98


# ------------------------------------------------------------
# 3. Full fidelity gate
# ------------------------------------------------------------

abs_delta = (
    centers["reconstructed_distance_um"]
    - centers["stored_distance_um"]
).abs()

print(
    "\n========== CORRECTED GLOBAL MATCHING FIDELITY =========="
)

print("rows             :", len(centers))
print(
    "max abs delta um :",
    f"{float(abs_delta.max()):.9f}",
)
print(
    "median delta um  :",
    f"{float(abs_delta.median()):.9f}",
)
print(
    "exact <=1e-5     :",
    int((abs_delta <= 1e-5).sum()),
    "/",
    len(centers),
)

assert float(abs_delta.max()) <= 1e-5


# ------------------------------------------------------------
# 4. GT center vs real detected center
# ------------------------------------------------------------

gt_ds = np.column_stack([
    centers["z"],
    centers["y"] / 4.0,
    centers["x"] / 4.0,
])

pred_ds = np.column_stack([
    centers["pred_z"],
    centers["pred_y"] / 4.0,
    centers["pred_x"] / 4.0,
])

gt_vox = np.clip(
    np.rint(gt_ds).astype(int),
    0,
    63,
)

pred_vox = np.clip(
    np.rint(pred_ds).astype(int),
    0,
    63,
)

delta = pred_vox - gt_vox

centers["pred_cz"] = pred_vox[:, 0]
centers["pred_cy"] = pred_vox[:, 1]
centers["pred_cx"] = pred_vox[:, 2]

centers["shift_l2_vox"] = np.linalg.norm(
    delta,
    axis=1,
)

centers["shift_linf_vox"] = (
    np.abs(delta).max(axis=1)
)


print(
    "\n========== DETECTED-CENTER SHIFT =========="
)

for name, mask in [
    ("ALL", np.ones(len(centers), dtype=bool)),
    ("DIVISION", centers["label"].to_numpy() == 1),
]:

    s2 = centers.loc[mask, "shift_l2_vox"]
    si = centers.loc[mask, "shift_linf_vox"]

    print(f"\n{name}")
    print(
        "same center:",
        round(float((s2 == 0).mean()), 6),
    )
    print(
        "Linf <= 1 :",
        round(float((si <= 1).mean()), 6),
    )
    print(
        "Linf <= 2 :",
        round(float((si <= 2).mean()), 6),
    )
    print(
        "L2 q50/q90/q95/max:",
        np.round(
            s2.quantile(
                [.50, .90, .95, 1.00]
            ).to_numpy(),
            3,
        ),
    )


# ------------------------------------------------------------
# 5. 19^3 boundary migration
# ------------------------------------------------------------

gt_interior = (
    (gt_vox >= RADIUS).all(axis=1)
    &
    (gt_vox <= 63 - RADIUS).all(axis=1)
)

pred_interior = (
    (pred_vox >= RADIUS).all(axis=1)
    &
    (pred_vox <= 63 - RADIUS).all(axis=1)
)

centers["gt_interior"] = gt_interior
centers["pred_interior"] = pred_interior

rows = []

for name, mask in [
    ("ALL", np.ones(len(centers), dtype=bool)),
    ("DIVISION", centers["label"].to_numpy() == 1),
]:

    rows.append({
        "population": name,
        "rows": int(mask.sum()),
        "gt_interior":
            int(gt_interior[mask].sum()),
        "pred_interior":
            int(pred_interior[mask].sum()),
        "interior_to_boundary":
            int(
                (
                    gt_interior[mask]
                    & ~pred_interior[mask]
                ).sum()
            ),
        "boundary_to_interior":
            int(
                (
                    ~gt_interior[mask]
                    & pred_interior[mask]
                ).sum()
            ),
    })

migration = pd.DataFrame(rows)

print(
    "\n========== 19^3 BOUNDARY MIGRATION =========="
)

print(
    migration.to_string(index=False)
)


# ------------------------------------------------------------
# 6. Persist exact corrected artifact
# ------------------------------------------------------------

centers.to_pickle(OUT)

print("\nwritten:", OUT)

print(
    "\n========== 11.19E-P1 DECISION =========="
)

print(
    "FOLDS1TO4_ELIGIBLE_ONLY_DETECTED_CENTERS: PASS"
)

### 11.19E — Real Detected-Center Recovery

Stage11 matching semantics were resolved as ELIGIBLE_ONLY.

Using that contract, all 84,815 folds1–4 W4-capable parent rows reproduced
their frozen Stage11 GT-to-detection match distances:

- exact within 1e-5 µm: 84,815 / 84,815
- maximum absolute delta: 2.37e-7 µm

Detected-center localization shift:
- ALL: 70.6% within L∞ <= 1 voxel; 92.2% within <= 2
- DIVISION: 74.5% within <= 1; 93.9% within <= 2

19^3 boundary migration was limited:
- ALL interior→boundary: 1,176
- ALL boundary→interior: 1,150
- DIVISION interior→boundary: 1
- DIVISION boundary→interior: 0

Decision:
FOLDS1TO4_REAL_DETECTED_CENTERS_CONFIRMED.

In [ ]:
# ============================================================
# RESULT — 11.19H Deployment Event Budget + Audit-Reuse Census
# Purpose:
#   Recompute the morphology event budget using the REAL
#   detected-center OOF ranking and measure overlap with the
#   already-replayed 11.18D GT-center Top5 tail.
#
#   This determines how much association replay must actually
#   be redone before official Frozen-V9 metric evaluation.
#
# Changes files: YES
#   - reports/stage11_division_probe/
#       stage11_predcenter_top5_events.pkl
#
# Runs training/inference: NO
# Uses GT: folds1–4 labels for calibration only
# Fold0 touched: NO
# Keep after running: YES
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

RPT = (
    PROJECT
    / "reports/stage11_division_probe"
)

NEW_OOF = (
    RPT
    / "stage11_predcenter_morphology_oof.pkl"
)

OLD_AUDIT = (
    RPT
    / "stage11_additive_second_edge_tail.pkl"
)

TOP5_OUT = (
    RPT
    / "stage11_predcenter_top5_events.pkl"
)


# ------------------------------------------------------------
# 1. Deployment-realistic ranking
# ------------------------------------------------------------

df = pd.read_pickle(NEW_OOF).copy()

assert len(df) == 84815
assert int(df["label"].sum()) == 98
assert df["dataset"].nunique() == 103

df = df.sort_values(
    [
        "dataset",
        "rank_morph_ensemble",
        "t",
        "node_id",
    ],
    ascending=[
        True,
        False,
        True,
        True,
    ],
).reset_index(drop=True)

df["rank_in_dataset"] = (
    df.groupby("dataset")
    .cumcount()
    + 1
)

zero_div_datasets = set(
    df.groupby("dataset")["label"]
    .sum()
    .loc[lambda x: x == 0]
    .index
)

assert len(zero_div_datasets) == 50


# ------------------------------------------------------------
# 2. New deployment event-budget frontier
# ------------------------------------------------------------

rows = []

for k in [
    1, 2, 3, 5,
    8, 10, 15, 20,
]:

    sub = df[
        df["rank_in_dataset"] <= k
    ]

    tp = int(sub["label"].sum())
    n = len(sub)

    rows.append({
        "K": k,
        "proposals": n,
        "TP": tp,
        "precision": tp / n,
        "recall": tp / 98,
        "zero_div_proposals": int(
            sub["dataset"]
            .isin(zero_div_datasets)
            .sum()
        ),
    })

budget = pd.DataFrame(rows)

print(
    "========== DEPLOYMENT EVENT BUDGET =========="
)

print(
    budget
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 3. Fold stability at operational low K
# ------------------------------------------------------------

fold_rows = []

for k in [1, 2, 3, 5]:

    sub = df[
        df["rank_in_dataset"] <= k
    ]

    for fold in [1, 2, 3, 4]:

        s = sub[
            sub["fold"] == fold
        ]

        fold_rows.append({
            "K": k,
            "fold": fold,
            "proposals": len(s),
            "TP": int(s["label"].sum()),
        })

fold_df = pd.DataFrame(fold_rows)

print(
    "\n========== LOW-K FOLD STABILITY =========="
)

print(
    fold_df
    .pivot(
        index="K",
        columns="fold",
        values="TP",
    )
    .to_string()
)


# ------------------------------------------------------------
# 4. New Top5 population
# ------------------------------------------------------------

new_top5 = (
    df[
        df["rank_in_dataset"] <= 5
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(new_top5) == 515

new_top5.to_pickle(
    TOP5_OUT
)


# ------------------------------------------------------------
# 5. Compare against already-replayed 11.18D tail
# ------------------------------------------------------------

old = pd.read_pickle(
    OLD_AUDIT
)

KEY = [
    "dataset",
    "t",
    "node_id",
]

assert len(old) == 515
assert not old.duplicated(KEY).any()
assert not new_top5.duplicated(KEY).any()

old_keys = set(
    old[KEY]
    .itertuples(
        index=False,
        name=None,
    )
)

new_keys = set(
    new_top5[KEY]
    .itertuples(
        index=False,
        name=None,
    )
)

shared = old_keys & new_keys
entering = new_keys - old_keys
leaving = old_keys - new_keys

new_top5["old_audit_available"] = [
    tuple(x) in old_keys
    for x in new_top5[KEY]
    .itertuples(
        index=False,
        name=None,
    )
]

shared_true = int(
    new_top5.loc[
        new_top5["old_audit_available"],
        "label",
    ].sum()
)

entering_true = int(
    new_top5.loc[
        ~new_top5["old_audit_available"],
        "label",
    ].sum()
)


print(
    "\n========== TOP5 AUDIT REUSE =========="
)

print("old Top5 rows       :", len(old_keys))
print("new Top5 rows       :", len(new_keys))
print("shared              :", len(shared))
print("entering            :", len(entering))
print("leaving             :", len(leaving))
print(
    "Top5 overlap        :",
    round(
        len(shared) / 515,
        6,
    ),
)

print(
    "new Top5 true events:",
    int(new_top5["label"].sum()),
)

print(
    "shared true events  :",
    shared_true,
)

print(
    "entering true events:",
    entering_true,
)


print(
    "\nwritten:",
    TOP5_OUT,
)

print(
    "\n========== 11.19H DECISION =========="
)

print(
    "DEPLOYMENT_EVENT_BUDGET_AND_REUSE_CENSUS: PASS"
)

### 11.20A — Exact Official-Metric Conversion

Deployment-realistic temporal morphology was converted into additive
second-daughter edges on final Frozen-V9 graphs and evaluated directly
with the locked official competition metric across folds 1–4.

Baseline:
- score: 0.780416707
- adjusted Edge Jaccard: 0.779684107
- Division Jaccard: 0.007326007
- division TP / FP / FN: 2 / 173 / 98

Best train-selected policy:
- WHEN_K = 5
- association rank cap = 3
- candidate probability >= 0.075
- 103 actions across 71 datasets

Result:
- adjusted Edge Jaccard delta: -0.000136236
- Division Jaccard delta: +0.036872888
- official score delta: +0.003551053
- final score: 0.783967760
- division TP / FP / FN: 16 / 262 / 84

Fold score deltas:
- Fold 1: +0.004891
- Fold 2: +0.003865
- Fold 3: +0.002768
- Fold 4: +0.003216

The improvement was positive in every held-out fold and remained strong
across a broad neighboring probability-threshold range.

Decision:
TEMPORAL_MORPHOLOGY_OFFICIAL_METRIC_GAIN_CONFIRMED.

Locked policy:
MORPH-DIV-V1 = WHEN_K 5 / PROB rank <= 3 / probability >= 0.075.

### 11.20D — Full Deployment Candidate Dilution

Temporal morphology was evaluated over the complete observable FINAL-V9
candidate universe rather than the previous GT-matched candidate subset.

- observable parents: 1,656,156
- datasets: 103
- known division parents surviving the deployable universe: 75 / 98
- known divisions ranked <= 1 / 2 / 3 / 5: 1 / 3 / 4 / 4
- ranked <= 10 / 20: 8 / 10
- division-parent rank median: 1,303
- rank q25 / q75 / q90: 136.5 / 3,918 / 15,607.6

The morphology signal is therefore strongly diluted when evaluated over the
full observable parent population.

However, sparse GT prevents interpreting unannotated high-ranked candidates
as false divisions. The next gate is therefore exact official-metric replay
of the fully deployable Top5 policy before designing an additional parent
prefilter.

### 11.20F — Fully Deployable Official-Metric Conversion

The oracle-filtered parent universe was removed entirely.

The final deployment candidate population consisted of all observable
W4-capable FINAL Frozen-V9 parents with current outdegree = 1.
Morphology ranking and association replay therefore required no GT
information.

Across folds 1–4, all 88 pre-specified policy configurations produced
positive official-score deltas.

The best conservative policy was:

MORPH-DIV-V2
- morphology per-dataset Top 3
- association PROB-row rank <= 3
- association probability >= 0.10
- additive second edge after FINAL Frozen-V9

Result:
- baseline score: 0.780416707
- policy score: 0.781893060
- delta score: +0.001476352
- delta adjusted Edge Jaccard: +0.000035040
- delta Division Jaccard: +0.014413123
- division TP / FP / FN: 6 / 176 / 94

Fold score deltas:
- Fold 1: +0.000000
- Fold 2: +0.003112
- Fold 3: +0.001108
- Fold 4: +0.001745

K5/R3/P100 achieved identical official metrics but required 150
interventions versus 96 for K3/R3/P100. The lower-intervention K3
policy was therefore locked.

Decision:
FULLY_DEPLOYABLE_TEMPORAL_MORPHOLOGY_GAIN_CONFIRMED.
MORPH-DIV-V2 LOCKED.

In [ ]:
import warnings

warnings.filterwarnings(
    "ignore",
    message=r"Graph already matched, overwriting previous matching\.",
    category=UserWarning,
)

### 11.20I — Locked Fold0 Outer Validation

MORPH-DIV-V2 was evaluated on Fold0 only after its exact intervention
set had been frozen without Fold0 GT access.

Locked policy:
- observable FINAL Frozen-V9 W4-capable parents with outdegree = 1
- per-dataset morphology Top 3
- association PROB-row rank <= 3
- association probability >= 0.10
- additive second edge after FINAL Frozen-V9

Blind action lock:
- 34 actions across 22 / 25 Fold0 6bba datasets
- SHA256:
  853a4e45880969489355848d6b39c594666e4d07354c20b6d42aa784def6847a

Official Fold0 result:

Baseline:
- adjusted Edge Jaccard: 0.774471340
- Division Jaccard: 0.040816327
- division TP / FP / FN: 2 / 24 / 23
- score: 0.778552972

MORPH-DIV-V2:
- adjusted Edge Jaccard: 0.774480353
- Division Jaccard: 0.060000000
- division TP / FP / FN: 3 / 25 / 22
- score: 0.780480353

Delta:
- adjusted Edge Jaccard: +0.000009013
- Division Jaccard: +0.019183673
- official score: +0.001927381

The policy improved the locked official metric on the blinded Fold0
outer test while leaving ordinary edge quality effectively unchanged.

Decision:
FULLY_DEPLOYABLE_TEMPORAL_MORPHOLOGY_OUTER_VALIDATED.
MORPH-DIV-V2 PROMOTED.
POLICY REMAINS LOCKED.

In [ ]:
# ============================================================
# RESULT — 11.21A-P1 Full Hybrid + MORPH-DIV-V2 Evaluation
# Purpose:
#   Evaluate the complete current-best 40-dataset Fold0 hybrid:
#
#       44b6 -> Sparse9995 seed1640 -> Frozen V9
#       6bba -> Baseline256        -> Frozen V9
#                                      + locked MORPH-DIV-V2
#
#   First reproduce the previous full-hybrid checkpoint exactly,
#   then measure the incremental gain from MORPH-DIV-V2.
#
# Changes files: YES
#   reports/stage11_division_probe/
#       stage11_full_hybrid_morphdiv_v2_summary.json
#       stage11_full_hybrid_morphdiv_v2_datasets.pkl
#
# Runs training: NO
# Runs inference: NO
# Uses GT: YES — locked Fold0 evaluation
# Keep after running: YES
# ============================================================

from pathlib import Path
import hashlib
import json
import sys

import numpy as np
import pandas as pd


# ============================================================
# 0. Paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

PRED_ROOT = (
    PROJECT
    / "predictions/heqiuyan"
)

RPT = (
    PROJECT
    / "reports/stage11_division_probe"
)

CV_PATH = (
    PROJECT
    / "configs/cv_manifest.csv"
)

ACTION_PATH = (
    RPT
    / "stage11_fold0_morphdiv_v2_actions_locked.pkl"
)

LOCK_PATH = (
    RPT
    / "stage11_fold0_morphdiv_v2_action_lock.json"
)

SUMMARY_OUT = (
    RPT
    / "stage11_full_hybrid_morphdiv_v2_summary.json"
)

DATASET_OUT = (
    RPT
    / "stage11_full_hybrid_morphdiv_v2_datasets.pkl"
)


EXPECTED_ACTION_SHA = (
    "853a4e45880969489355848d6b39c594"
    "666e4d07354c20b6d42aa784def6847a"
)


# Previous locked current-best full Fold0 hybrid.
EXPECTED_BASELINE = {
    "edge_jaccard":
        0.781989031,

    "adj_edge_jaccard":
        0.774278344,

    "division_jaccard":
        0.031746032,

    "node_recall":
        0.959822345,

    "score":
        0.777452947,
}


sys.path.insert(
    0,
    str(PROJECT / "src"),
)

from biohub_cell_tracking import frozen_v9 as fv9


# ============================================================
# 1. Exact frozen prediction routes
# ============================================================

# 44b6 -> Sparse9995 seed1640
S9995_995 = (
    PRED_ROOT
    / "stage10_infer_sparse9995_256_det995"
    / "split_0"
)

S9995_9975 = (
    PRED_ROOT
    / "stage10_infer_sparse9995_256_det9975"
    / "split_0"
)

# 6bba -> Baseline256
B256_995 = (
    PRED_ROOT
    / "stage10_infer_baseline_256_det995"
    / "split_0"
)

B256_9975 = (
    PRED_ROOT
    / "stage10_infer_baseline_256_det9975"
    / "split_0"
)


for path in [
    S9995_995,
    S9995_9975,
    B256_995,
    B256_9975,
]:
    assert path.is_dir(), path


print(
    "========== 11.21A-P1 ROUTES =========="
)

print(
    "44b6 : Sparse9995 seed1640"
)

print(
    "6bba : Baseline256"
)


# ============================================================
# 2. Fold0 40-dataset corpus
# ============================================================

cv = pd.read_csv(
    CV_PATH
)

cv0 = (
    cv[
        cv["fold"]
        == 0
    ]
    .sort_values(
        [
            "prefix",
            "dataset",
        ]
    )
    .reset_index(
        drop=True
    )
)


assert len(cv0) == 40


names_44 = (
    cv0.loc[
        cv0[
            "dataset"
        ].str.startswith(
            "44b6_"
        ),
        "dataset",
    ]
    .tolist()
)

names_6 = (
    cv0.loc[
        cv0[
            "dataset"
        ].str.startswith(
            "6bba_"
        ),
        "dataset",
    ]
    .tolist()
)


assert len(names_44) == 15
assert len(names_6) == 25


estimated_nodes = dict(
    zip(
        cv0["dataset"],
        cv0[
            "estimated_nodes"
        ].astype(float),
    )
)

prefix_map = dict(
    zip(
        cv0["dataset"],
        cv0["prefix"],
    )
)


# ============================================================
# 3. Prediction coverage gates
# ============================================================

def assert_coverage(
    root,
    names,
):

    existing = {
        p.stem
        for p in root.glob(
            "*.geff"
        )
    }

    missing = sorted(
        set(names)
        - existing
    )

    assert not missing, (
        root,
        missing,
    )


assert_coverage(
    S9995_995,
    names_44,
)

assert_coverage(
    S9995_9975,
    names_44,
)

assert_coverage(
    B256_995,
    names_6,
)

assert_coverage(
    B256_9975,
    names_6,
)


print(
    "Fold0 corpus:",
    len(names_44),
    "44b6 +",
    len(names_6),
    "6bba =",
    len(cv0),
)


# ============================================================
# 4. Verify locked MORPH-DIV-V2 action set
# ============================================================

actions = pd.read_pickle(
    ACTION_PATH
).copy()

lock = json.loads(
    LOCK_PATH.read_text(
        encoding="utf-8"
    )
)


assert len(actions) == 34

assert (
    actions[
        "dataset"
    ].nunique()
    == 22
)

assert set(
    actions["dataset"]
).issubset(
    set(names_6)
)


canonical_records = []

for row in actions.itertuples(
    index=False
):

    canonical_records.append({
        "dataset":
            str(
                row.dataset
            ),

        "morph_rank":
            int(
                row.morph_rank
            ),

        "t":
            int(
                row.t
            ),

        "source_pred_node_id":
            int(
                row.source_pred_node_id
            ),

        "target_pred_node_id":
            int(
                row.target_pred_node_id
            ),

        "candidate_rank":
            int(
                row.candidate_rank
            ),

        "candidate_prob":
            format(
                float(
                    row.candidate_prob
                ),
                ".12f",
            ),
    })


canonical_text = json.dumps(
    canonical_records,
    sort_keys=True,
    separators=(",", ":"),
)

action_sha = hashlib.sha256(
    canonical_text.encode(
        "utf-8"
    )
).hexdigest()


assert (
    action_sha
    == EXPECTED_ACTION_SHA
)

assert (
    lock["action_sha256"]
    == EXPECTED_ACTION_SHA
)


print(
    "MORPH-DIV-V2 actions:",
    len(actions),
)

print(
    "action SHA verified:",
    "YES"
)


# ============================================================
# 5. Route-specific Frozen-V9 policies
# ============================================================

policies_44 = (
    fv9.build_frozen_v9_policies(
        names_44,
        S9995_995,
        S9995_9975,
        TRAIN_ROOT,
        prefix_map=
            prefix_map,
        config=
            fv9.FROZEN_V9_CONFIG,
        progress=False,
    )
)


policies_6 = (
    fv9.build_frozen_v9_policies(
        names_6,
        B256_995,
        B256_9975,
        TRAIN_ROOT,
        prefix_map=
            prefix_map,
        config=
            fv9.FROZEN_V9_CONFIG,
        progress=False,
    )
)


# ============================================================
# 6. Locked edge insertion
# ============================================================

def add_locked_action(
    graph,
    nodes,
    action,
):

    src = int(
        action.source_pred_node_id
    )

    tgt = int(
        action.target_pred_node_id
    )


    dist = fv9.recovered_edge_dist(
        nodes,
        src,
        tgt,
        fv9.FROZEN_V9_CONFIG,
    )


    attrs = {
        "edge_dist":
            float(dist),

        "edge_prob":
            float(
                action.candidate_prob
            ),
    }


    if (
        "matched_edge_mask"
        in set(
            graph.edge_attr_keys()
        )
    ):
        attrs[
            "matched_edge_mask"
        ] = False


    graph.add_edge(
        src,
        tgt,
        attrs,
    )


# ============================================================
# 7. Evaluate 40-dataset current-best baseline and new hybrid
# ============================================================

baseline_records = []

policy_records = []

dataset_rows = []


for i, row in enumerate(
    cv0.itertuples(
        index=False
    ),
    start=1,
):

    name = str(
        row.dataset
    )

    is_44 = name.startswith(
        "44b6_"
    )


    if is_44:

        pred995 = S9995_995
        pred9975 = S9995_9975
        policies = policies_44

    else:

        pred995 = B256_995
        pred9975 = B256_9975
        policies = policies_6


    # --------------------------------------------------------
    # A. Fresh previous current-best hybrid graph
    # --------------------------------------------------------

    (
        base_graph,
        base_scale,
        _,
    ) = fv9.apply_frozen_v9(
        name,
        pred995,
        pred9975,
        policies,
        TRAIN_ROOT,
        config=
            fv9.FROZEN_V9_CONFIG,
    )


    gt_base = fv9.load_graph(
        TRAIN_ROOT
        / f"{name}.geff"
    )


    bm = fv9.evaluate_graph(
        base_graph,
        gt_base,
        base_scale,
        estimated_nodes[
            name
        ],
        fv9.FROZEN_V9_CONFIG,
    )


    baseline_records.append(
        bm
    )


    # --------------------------------------------------------
    # B. Separate fresh policy graph
    # --------------------------------------------------------

    (
        policy_graph,
        policy_scale,
        _,
    ) = fv9.apply_frozen_v9(
        name,
        pred995,
        pred9975,
        policies,
        TRAIN_ROOT,
        config=
            fv9.FROZEN_V9_CONFIG,
    )


    dataset_actions = (
        actions[
            actions[
                "dataset"
            ]
            == name
        ]
        if not is_44
        else actions.iloc[
            0:0
        ]
    )


    if len(
        dataset_actions
    ):

        nodes = (
            policy_graph.node_attrs(
                attr_keys=[
                    "node_id",
                    "t",
                    "z",
                    "y",
                    "x",
                ]
            )
            .to_pandas()
            .set_index(
                "node_id"
            )
        )


        edges = (
            policy_graph.edge_attrs(
                unpack=True
            )
            .to_pandas()
        )


        outdeg = (
            edges
            .groupby(
                "source_id"
            )
            .size()
            .astype(int)
            .to_dict()
        )


        indeg = (
            edges
            .groupby(
                "target_id"
            )
            .size()
            .astype(int)
            .to_dict()
        )


        edge_set = set(
            zip(
                edges[
                    "source_id"
                ].astype(int),

                edges[
                    "target_id"
                ].astype(int),
            )
        )


        ordered_actions = (
            dataset_actions
            .sort_values(
                [
                    "morph_rank",
                    "t",
                    "source_pred_node_id",
                    "target_pred_node_id",
                ]
            )
        )


        for action in (
            ordered_actions
            .itertuples(
                index=False
            )
        ):

            src = int(
                action.source_pred_node_id
            )

            tgt = int(
                action.target_pred_node_id
            )


            assert src in nodes.index
            assert tgt in nodes.index

            assert (
                int(
                    outdeg.get(
                        src,
                        0,
                    )
                )
                == 1
            )

            assert (
                int(
                    indeg.get(
                        tgt,
                        0,
                    )
                )
                == 0
            )

            assert (
                (
                    src,
                    tgt,
                )
                not in edge_set
            )

            assert (
                int(
                    nodes.loc[
                        tgt,
                        "t",
                    ]
                )
                ==
                int(
                    nodes.loc[
                        src,
                        "t",
                    ]
                )
                + 1
            )


            add_locked_action(
                policy_graph,
                nodes,
                action,
            )


            edge_set.add(
                (
                    src,
                    tgt,
                )
            )

            outdeg[
                src
            ] = 2

            indeg[
                tgt
            ] = 1


    # --------------------------------------------------------
    # C. Evaluate new full hybrid
    # --------------------------------------------------------

    gt_policy = fv9.load_graph(
        TRAIN_ROOT
        / f"{name}.geff"
    )


    pm = fv9.evaluate_graph(
        policy_graph,
        gt_policy,
        policy_scale,
        estimated_nodes[
            name
        ],
        fv9.FROZEN_V9_CONFIG,
    )


    policy_records.append(
        pm
    )


    dataset_rows.append({
        "dataset":
            name,

        "route":
            (
                "Sparse9995"
                if is_44
                else "Baseline256"
            ),

        "actions":
            int(
                len(
                    dataset_actions
                )
            ),

        "baseline_edge":
            float(
                bm[
                    "edge_jaccard"
                ]
            ),

        "policy_edge":
            float(
                pm[
                    "edge_jaccard"
                ]
            ),

        "delta_edge":
            float(
                pm[
                    "edge_jaccard"
                ]
                -
                bm[
                    "edge_jaccard"
                ]
            ),

        "baseline_adj_edge":
            float(
                bm[
                    "adj_edge_jaccard"
                ]
            ),

        "policy_adj_edge":
            float(
                pm[
                    "adj_edge_jaccard"
                ]
            ),

        "delta_adj_edge":
            float(
                pm[
                    "adj_edge_jaccard"
                ]
                -
                bm[
                    "adj_edge_jaccard"
                ]
            ),
    })


    if (
        i == 1
        or i % 10 == 0
        or i == len(cv0)
    ):

        print(
            f"{i:2d}/{len(cv0)} datasets evaluated"
        )


# ============================================================
# 8. Aggregate
# ============================================================

baseline = fv9.summarise(
    baseline_records
)

policy = fv9.summarise(
    policy_records
)


# ============================================================
# 9. Critical reproduction gate
# ============================================================

print(
    "\n========== CURRENT-BEST HYBRID REPRODUCTION =========="
)


for key, expected in (
    EXPECTED_BASELINE.items()
):

    observed = float(
        baseline[
            key
        ]
    )

    delta = abs(
        observed
        - expected
    )


    print(
        f"{key:22s}: "
        f"{observed:.9f} | "
        f"expected={expected:.9f} | "
        f"delta={delta:.3e}"
    )


    assert (
        delta <= 1e-9
    ), (
        key,
        observed,
        expected,
        delta,
    )


print(
    "REPRODUCTION: PASS"
)


# ============================================================
# 10. New full-hybrid result
# ============================================================

print(
    "\n========== FULL HYBRID + MORPH-DIV-V2 =========="
)


for key in [
    "edge_jaccard",
    "adj_edge_jaccard",
    "division_jaccard",
    "node_recall",
    "score",
]:

    print(
        f"{key:22s}: "
        f"{float(policy[key]):.9f}"
    )


print(
    "division TP/FP/FN     :",
    int(
        policy[
            "division_tp"
        ]
    ),
    "/",
    int(
        policy[
            "division_fp"
        ]
    ),
    "/",
    int(
        policy[
            "division_fn"
        ]
    ),
)


# ============================================================
# 11. Incremental Stage11 gain
# ============================================================

print(
    "\n========== COMPLETE HYBRID DELTA =========="
)


for key in [
    "edge_jaccard",
    "adj_edge_jaccard",
    "division_jaccard",
    "score",
]:

    print(
        f"delta {key:16s}: "
        f"{float(policy[key] - baseline[key]):+.9f}"
    )


# ============================================================
# 12. Persist
# ============================================================

dataset_df = pd.DataFrame(
    dataset_rows
)


def json_safe(
    d,
):

    result = {}

    for k, v in d.items():

        if isinstance(
            v,
            (
                np.integer,
                int,
            ),
        ):
            result[k] = int(v)

        elif isinstance(
            v,
            (
                np.floating,
                float,
            ),
        ):
            result[k] = float(v)

        else:
            result[k] = v

    return result


payload = {
    "stage":
        "11.21A-P1",

    "baseline_name":
        "Frozen-V9 current-best hybrid",

    "policy_name":
        "Frozen-V9 current-best hybrid + MORPH-DIV-V2",

    "action_sha256":
        EXPECTED_ACTION_SHA,

    "baseline":
        json_safe(
            baseline
        ),

    "policy":
        json_safe(
            policy
        ),

    "delta": {
        "edge_jaccard":
            float(
                policy[
                    "edge_jaccard"
                ]
                -
                baseline[
                    "edge_jaccard"
                ]
            ),

        "adj_edge_jaccard":
            float(
                policy[
                    "adj_edge_jaccard"
                ]
                -
                baseline[
                    "adj_edge_jaccard"
                ]
            ),

        "division_jaccard":
            float(
                policy[
                    "division_jaccard"
                ]
                -
                baseline[
                    "division_jaccard"
                ]
            ),

        "score":
            float(
                policy[
                    "score"
                ]
                -
                baseline[
                    "score"
                ]
            ),
    },
}


SUMMARY_OUT.write_text(
    json.dumps(
        payload,
        indent=2,
        allow_nan=False,
    ),
    encoding="utf-8",
)


dataset_df.to_pickle(
    DATASET_OUT
)


print(
    "\nwritten:",
    SUMMARY_OUT
)

print(
    "written:",
    DATASET_OUT
)


print(
    "\n========== 11.21A-P1 DECISION =========="
)

print(
    "FULL_HYBRID_MORPHDIV_V2_EVALUATED: PASS"
)

### Stage 11 Final — Fully Deployable Division Recovery

MORPH-DIV-V2 completed the full deployment and validation chain.

Locked policy:
- FINAL Frozen-V9 observable parents
- W4-capable
- current outdegree = 1
- temporal morphology Top 3 per dataset
- association PROB-row rank <= 3
- association probability >= 0.10
- additive second edge only

Fold0 action set was frozen before GT evaluation:
- 34 interventions
- 22 / 25 6bba datasets
- SHA256:
  853a4e45880969489355848d6b39c594666e4d07354c20b6d42aa784def6847a

Full 40-dataset hybrid reproduction:
- score: 0.777452947
- exact reproduction: PASS

Full hybrid + MORPH-DIV-V2:
- Edge Jaccard: 0.781996647
- adjusted Edge Jaccard: 0.774285647
- Division Jaccard: 0.046875000
- node recall: 0.959822345
- division TP / FP / FN: 3 / 34 / 27
- official score: 0.778973147

Incremental gain:
- Edge Jaccard: +0.000007615
- adjusted Edge Jaccard: +0.000007304
- Division Jaccard: +0.015128968
- official score: +0.001520200

Decision:
MORPH-DIV-V2 PROMOTED.
STAGE 11 COMPLETE.
CURRENT-BEST FOLD0 SCORE = 0.778973147.

In [ ]:
# ============================================================
# RESULT — 11.21B Stage11 Current-Best Checkpoint Freeze
# Purpose:
#   Freeze the complete validated Stage11 current-best pipeline
#   and its exact provenance before beginning Stage12.
#
# Changes files: YES
#   reports/stage11_division_probe/
#       stage11_current_best_checkpoint.json
#
# Runs training/inference: NO
# Uses GT: NO
# Keep after running: YES
# ============================================================

from pathlib import Path
import hashlib
import json


PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

RPT = (
    PROJECT
    / "reports/stage11_division_probe"
)

OUT = (
    RPT
    / "stage11_current_best_checkpoint.json"
)


# ------------------------------------------------------------
# 1. Immutable artifacts
# ------------------------------------------------------------

files = {
    "morphdiv_action_lock":
        RPT
        / "stage11_fold0_morphdiv_v2_action_lock.json",

    "morphdiv_actions":
        RPT
        / "stage11_fold0_morphdiv_v2_actions_locked.pkl",

    "fold0_outer_summary":
        RPT
        / "stage11_fold0_morphdiv_v2_outer_summary.json",

    "full_hybrid_summary":
        RPT
        / "stage11_full_hybrid_morphdiv_v2_summary.json",
}


for name, path in files.items():

    assert path.is_file(), (
        name,
        path,
    )


def sha256_file(path):

    h = hashlib.sha256()

    with path.open("rb") as f:

        for block in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


artifact_hashes = {
    key: {
        "path":
            str(path),

        "sha256":
            sha256_file(path),
    }

    for key, path
    in files.items()
}


# ------------------------------------------------------------
# 2. Verify locked action identity
# ------------------------------------------------------------

action_lock = json.loads(
    files[
        "morphdiv_action_lock"
    ].read_text(
        encoding="utf-8"
    )
)


EXPECTED_ACTION_SHA = (
    "853a4e45880969489355848d6b39c594"
    "666e4d07354c20b6d42aa784def6847a"
)


assert (
    action_lock["action_sha256"]
    == EXPECTED_ACTION_SHA
)

assert (
    int(
        action_lock["locked_actions"]
    )
    == 34
)


# ------------------------------------------------------------
# 3. Verify final full-hybrid metric
# ------------------------------------------------------------

hybrid = json.loads(
    files[
        "full_hybrid_summary"
    ].read_text(
        encoding="utf-8"
    )
)


score = float(
    hybrid[
        "policy"
    ][
        "score"
    ]
)


assert abs(
    score
    - 0.778973147
) <= 1e-9


# ------------------------------------------------------------
# 4. Freeze complete checkpoint
# ------------------------------------------------------------

checkpoint = {
    "stage":
        "Stage11",

    "status":
        "COMPLETE",

    "checkpoint_name":
        "FULL_HYBRID_PLUS_MORPH_DIV_V2",

    "routes": {
        "44b6": {
            "predictor":
                "Sparse9995 seed1640",

            "postprocess":
                "Frozen V9",
        },

        "6bba": {
            "predictor":
                "Baseline256",

            "postprocess": [
                "Frozen V9",
                "MORPH-DIV-V2",
            ],
        },
    },

    "morph_div_v2": {
        "candidate_universe":
            (
                "FINAL Frozen-V9 W4-capable "
                "parents with outdegree=1"
            ),

        "morphology_top_k":
            3,

        "association_rank_cap":
            3,

        "association_prob_threshold":
            0.10,

        "mutation":
            "additive second edge",

        "locked_actions_fold0":
            34,

        "action_sha256":
            EXPECTED_ACTION_SHA,
    },

    "fold0_metrics": {
        "edge_jaccard":
            0.781996647,

        "adj_edge_jaccard":
            0.774285647,

        "division_jaccard":
            0.046875000,

        "node_recall":
            0.959822345,

        "division_tp":
            3,

        "division_fp":
            34,

        "division_fn":
            27,

        "score":
            0.778973147,
    },

    "previous_best_score":
        0.777452947,

    "incremental_gain": {
        "edge_jaccard":
            0.000007615,

        "adj_edge_jaccard":
            0.000007304,

        "division_jaccard":
            0.015128968,

        "score":
            0.001520200,
    },

    "artifact_hashes":
        artifact_hashes,

    "policy_locked":
        True,

    "retuning_allowed":
        False,
}


OUT.write_text(
    json.dumps(
        checkpoint,
        indent=2,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# 5. Verify persisted checkpoint
# ------------------------------------------------------------

written = json.loads(
    OUT.read_text(
        encoding="utf-8"
    )
)


assert (
    written[
        "checkpoint_name"
    ]
    == "FULL_HYBRID_PLUS_MORPH_DIV_V2"
)

assert (
    written[
        "morph_div_v2"
    ][
        "action_sha256"
    ]
    == EXPECTED_ACTION_SHA
)

assert abs(
    float(
        written[
            "fold0_metrics"
        ][
            "score"
        ]
    )
    - 0.778973147
) <= 1e-9


print(
    "========== STAGE11 CURRENT-BEST CHECKPOINT =========="
)

print(
    "pipeline : FULL_HYBRID_PLUS_MORPH_DIV_V2"
)

print(
    "score    :",
    f"{written['fold0_metrics']['score']:.9f}",
)

print(
    "adj edge :",
    f"{written['fold0_metrics']['adj_edge_jaccard']:.9f}",
)

print(
    "division :",
    f"{written['fold0_metrics']['division_jaccard']:.9f}",
)

print(
    "actions  :",
    written[
        "morph_div_v2"
    ][
        "locked_actions_fold0"
    ],
)

print(
    "policy locked:",
    written[
        "policy_locked"
    ],
)

print(
    "\nwritten:",
    OUT,
)

print(
    "\n========== 11.21B DECISION =========="
)

print(
    "STAGE11_CURRENT_BEST_FROZEN: PASS"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.00A Current-Best Residual Metric Anatomy
# Purpose:
#   Inspect the exact aggregate metric/error counts of the frozen
#   Stage11 current-best checkpoint before choosing a Stage12
#   research direction.
#
#   No new model or policy is tested here.
#
# Changes files: NO
# Runs training/inference: NO
# Uses GT: NO — reads already-computed Stage11 metric artifact
# Keep after running: YES
# ============================================================

from pathlib import Path
import json


PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

RPT = (
    PROJECT
    / "reports/stage11_division_probe"
)

SUMMARY = (
    RPT
    / "stage11_full_hybrid_morphdiv_v2_summary.json"
)

CHECKPOINT = (
    RPT
    / "stage11_current_best_checkpoint.json"
)


assert SUMMARY.is_file()
assert CHECKPOINT.is_file()


summary = json.loads(
    SUMMARY.read_text(
        encoding="utf-8"
    )
)

checkpoint = json.loads(
    CHECKPOINT.read_text(
        encoding="utf-8"
    )
)


policy = summary["policy"]


print(
    "========== STAGE12.00A CURRENT-BEST =========="
)

print(
    "checkpoint:",
    checkpoint["checkpoint_name"],
)

print(
    "score     :",
    f"{float(policy['score']):.9f}",
)

print(
    "edge J    :",
    f"{float(policy['edge_jaccard']):.9f}",
)

print(
    "adj edge J:",
    f"{float(policy['adj_edge_jaccard']):.9f}",
)

print(
    "division J:",
    f"{float(policy['division_jaccard']):.9f}",
)

print(
    "node recall:",
    f"{float(policy['node_recall']):.9f}",
)


print(
    "\n========== FULL METRIC CONTRACT =========="
)

for key in sorted(
    policy.keys()
):

    print(
        f"{key:28s}: {policy[key]}"
    )


print(
    "\n========== SCORE LEVERAGE =========="
)

print(
    "+0.001 adjusted-edge J -> +0.001000 score"
)

print(
    "+0.010 division J      -> +0.001000 score"
)


print(
    "\n========== 12.00A DECISION =========="
)

print(
    "CURRENT_BEST_RESIDUAL_METRIC_ANATOMY: PASS"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.00B Official Evaluator Residual Schema Probe
# Purpose:
#   Inspect the exact post-evaluation graph schemas exposed by
#   the locked official evaluator.
#
#   We need this before constructing edge TP / FP / FN residual
#   counts for Stage12.
#
# Changes files: NO
# Runs training/inference: NO
# Uses GT: YES — diagnostic inspection only
# Changes policy: NO
# Keep after running: YES
# ============================================================

from pathlib import Path
import json
import sys

import pandas as pd


PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

PRED_ROOT = (
    PROJECT
    / "predictions/heqiuyan"
)

RPT = (
    PROJECT
    / "reports/stage11_division_probe"
)

CV_PATH = (
    PROJECT
    / "configs/cv_manifest.csv"
)

ACTION_PATH = (
    RPT
    / "stage11_fold0_morphdiv_v2_actions_locked.pkl"
)


sys.path.insert(
    0,
    str(PROJECT / "src"),
)

from biohub_cell_tracking import frozen_v9 as fv9


# ============================================================
# 1. Exact frozen Fold0 6bba route
# ============================================================

PRED995 = (
    PRED_ROOT
    / "stage10_infer_baseline_256_det995"
    / "split_0"
)

PRED9975 = (
    PRED_ROOT
    / "stage10_infer_baseline_256_det9975"
    / "split_0"
)


cv = pd.read_csv(
    CV_PATH
)

cv0_6 = (
    cv[
        (cv["fold"] == 0)
        &
        cv["dataset"].str.startswith("6bba_")
    ]
    .sort_values("dataset")
    .reset_index(drop=True)
)

assert len(cv0_6) == 25


names = cv0_6[
    "dataset"
].tolist()

name = names[0]


estimated_nodes = dict(
    zip(
        cv0_6["dataset"],
        cv0_6["estimated_nodes"].astype(float),
    )
)

prefix_map = dict(
    zip(
        cv0_6["dataset"],
        cv0_6["prefix"],
    )
)


# ============================================================
# 2. Exact Frozen-V9 graph
# ============================================================

policies = fv9.build_frozen_v9_policies(
    names,
    PRED995,
    PRED9975,
    TRAIN_ROOT,
    prefix_map=prefix_map,
    config=fv9.FROZEN_V9_CONFIG,
    progress=False,
)


graph, scale, _ = fv9.apply_frozen_v9(
    name,
    PRED995,
    PRED9975,
    policies,
    TRAIN_ROOT,
    config=fv9.FROZEN_V9_CONFIG,
)


# ============================================================
# 3. Apply already-locked MORPH-DIV-V2 actions for this dataset
# ============================================================

actions = pd.read_pickle(
    ACTION_PATH
)

dataset_actions = (
    actions[
        actions["dataset"] == name
    ]
    .sort_values(
        [
            "morph_rank",
            "t",
            "source_pred_node_id",
            "target_pred_node_id",
        ]
    )
)


if len(dataset_actions):

    nodes = (
        graph.node_attrs(
            attr_keys=[
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
        .set_index("node_id")
    )

    edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
    )

    outdeg = (
        edges
        .groupby("source_id")
        .size()
        .astype(int)
        .to_dict()
    )

    indeg = (
        edges
        .groupby("target_id")
        .size()
        .astype(int)
        .to_dict()
    )

    edge_set = set(
        zip(
            edges["source_id"].astype(int),
            edges["target_id"].astype(int),
        )
    )


    for a in dataset_actions.itertuples(
        index=False
    ):

        src = int(
            a.source_pred_node_id
        )

        tgt = int(
            a.target_pred_node_id
        )

        assert outdeg.get(src, 0) == 1
        assert indeg.get(tgt, 0) == 0
        assert (src, tgt) not in edge_set

        dist = fv9.recovered_edge_dist(
            nodes,
            src,
            tgt,
            fv9.FROZEN_V9_CONFIG,
        )

        attrs = {
            "edge_dist":
                float(dist),

            "edge_prob":
                float(
                    a.candidate_prob
                ),
        }

        if (
            "matched_edge_mask"
            in set(
                graph.edge_attr_keys()
            )
        ):
            attrs[
                "matched_edge_mask"
            ] = False

        graph.add_edge(
            src,
            tgt,
            attrs,
        )

        edge_set.add(
            (src, tgt)
        )

        outdeg[src] = 2
        indeg[tgt] = 1


# ============================================================
# 4. Schema BEFORE official evaluation
# ============================================================

gt_graph = fv9.load_graph(
    TRAIN_ROOT
    / f"{name}.geff"
)


print(
    "========== BEFORE OFFICIAL EVALUATION =========="
)

print(
    "dataset:",
    name
)

print(
    "prediction node attrs:",
    sorted(
        graph.node_attr_keys()
    )
)

print(
    "prediction edge attrs:",
    sorted(
        graph.edge_attr_keys()
    )
)

print(
    "GT node attrs:",
    sorted(
        gt_graph.node_attr_keys()
    )
)

print(
    "GT edge attrs:",
    sorted(
        gt_graph.edge_attr_keys()
    )
)


# ============================================================
# 5. Locked official evaluation
# ============================================================

metric = fv9.evaluate_graph(
    graph,
    gt_graph,
    scale,
    estimated_nodes[name],
    fv9.FROZEN_V9_CONFIG,
)


print(
    "\n========== SINGLE-DATASET METRIC =========="
)

for key in sorted(
    metric.keys()
):
    print(
        f"{key:28s}: {metric[key]}"
    )


# ============================================================
# 6. Schema AFTER official evaluation
# ============================================================

print(
    "\n========== AFTER OFFICIAL EVALUATION =========="
)

print(
    "prediction node attrs:",
    sorted(
        graph.node_attr_keys()
    )
)

print(
    "prediction edge attrs:",
    sorted(
        graph.edge_attr_keys()
    )
)

print(
    "GT node attrs:",
    sorted(
        gt_graph.node_attr_keys()
    )
)

print(
    "GT edge attrs:",
    sorted(
        gt_graph.edge_attr_keys()
    )
)


# ============================================================
# 7. Inspect edge tables and any evaluator-added fields
# ============================================================

pred_edges = (
    graph.edge_attrs(
        unpack=True
    )
    .to_pandas()
)

gt_edges = (
    gt_graph.edge_attrs(
        unpack=True
    )
    .to_pandas()
)


print(
    "\n========== PRED EDGE COLUMNS =========="
)

print(
    pred_edges.columns.tolist()
)


print(
    "\n========== GT EDGE COLUMNS =========="
)

print(
    gt_edges.columns.tolist()
)


interesting_pred = [
    c
    for c in pred_edges.columns
    if (
        "match" in c.lower()
        or "mask" in c.lower()
        or "gt" in c.lower()
    )
]

interesting_gt = [
    c
    for c in gt_edges.columns
    if (
        "match" in c.lower()
        or "mask" in c.lower()
        or "pred" in c.lower()
    )
]


print(
    "\n========== EVALUATOR-ADDED PRED FIELDS =========="
)

print(
    interesting_pred
)


for c in interesting_pred:

    print(
        f"\n{c}:"
    )

    print(
        pred_edges[c]
        .value_counts(
            dropna=False
        )
        .to_string()
    )


print(
    "\n========== EVALUATOR-ADDED GT FIELDS =========="
)

print(
    interesting_gt
)


for c in interesting_gt:

    print(
        f"\n{c}:"
    )

    print(
        gt_edges[c]
        .value_counts(
            dropna=False
        )
        .to_string()
    )


print(
    "\n========== 12.00B DECISION =========="
)

print(
    "OFFICIAL_EVALUATOR_RESIDUAL_SCHEMA_PROBED: PASS"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.00D Full Current-Best Edge Residual Census
# Purpose:
#   Decompose every official edge FN in the frozen Stage11
#   current-best 40-dataset Fold0 pipeline into:
#
#       DETECTION_LIMITED
#           one or both GT edge endpoints have no matched
#           predicted node
#
#       LINKING_LIMITED
#           both GT endpoints already exist as matched predicted
#           nodes, but their correct edge is absent
#
#   Also classify exact official FPs using source-locked:
#
#       pred_valid == True
#       matched_edge_mask == False
#
#   No metric logic is reimplemented.
#
# Changes files: YES
#   reports/stage12_residual_audit/
#       stage12_edge_fn_residuals.pkl
#       stage12_edge_fp_residuals.pkl
#       stage12_edge_residual_datasets.pkl
#       stage12_edge_residual_summary.json
#
# Runs training/inference: NO
# Uses GT: YES — residual research audit
# Changes frozen Stage11 policy: NO
# Keep after running: YES
# ============================================================

from pathlib import Path
import hashlib
import json
import sys

import numpy as np
import pandas as pd


# ============================================================
# 0. Paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

PRED_ROOT = (
    PROJECT
    / "predictions/heqiuyan"
)

STAGE11_RPT = (
    PROJECT
    / "reports/stage11_division_probe"
)

RPT = (
    PROJECT
    / "reports/stage12_residual_audit"
)

RPT.mkdir(
    parents=True,
    exist_ok=True,
)

CV_PATH = (
    PROJECT
    / "configs/cv_manifest.csv"
)

ACTION_PATH = (
    STAGE11_RPT
    / "stage11_fold0_morphdiv_v2_actions_locked.pkl"
)

CHECKPOINT_PATH = (
    STAGE11_RPT
    / "stage11_current_best_checkpoint.json"
)

FN_OUT = (
    RPT
    / "stage12_edge_fn_residuals.pkl"
)

FP_OUT = (
    RPT
    / "stage12_edge_fp_residuals.pkl"
)

DATASET_OUT = (
    RPT
    / "stage12_edge_residual_datasets.pkl"
)

SUMMARY_OUT = (
    RPT
    / "stage12_edge_residual_summary.json"
)


sys.path.insert(
    0,
    str(PROJECT / "src"),
)

sys.path.insert(
    0,
    str(PROJECT / "external/official/src"),
)

from biohub_cell_tracking import frozen_v9 as fv9
from tracking_cellmot.metrics import _evaluate_matched_graph


# ============================================================
# 1. Verify frozen Stage11 checkpoint
# ============================================================

checkpoint = json.loads(
    CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)

assert (
    checkpoint["checkpoint_name"]
    == "FULL_HYBRID_PLUS_MORPH_DIV_V2"
)

assert abs(
    float(
        checkpoint[
            "fold0_metrics"
        ][
            "score"
        ]
    )
    - 0.778973147
) <= 1e-9


def sha256_file(path):

    h = hashlib.sha256()

    with path.open("rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


expected_action_file_sha = (
    checkpoint[
        "artifact_hashes"
    ][
        "morphdiv_actions"
    ][
        "sha256"
    ]
)

assert (
    sha256_file(ACTION_PATH)
    == expected_action_file_sha
)


# ============================================================
# 2. Frozen current-best routes
# ============================================================

S9995_995 = (
    PRED_ROOT
    / "stage10_infer_sparse9995_256_det995"
    / "split_0"
)

S9995_9975 = (
    PRED_ROOT
    / "stage10_infer_sparse9995_256_det9975"
    / "split_0"
)

B256_995 = (
    PRED_ROOT
    / "stage10_infer_baseline_256_det995"
    / "split_0"
)

B256_9975 = (
    PRED_ROOT
    / "stage10_infer_baseline_256_det9975"
    / "split_0"
)


# ============================================================
# 3. Fold0 corpus
# ============================================================

cv = pd.read_csv(
    CV_PATH
)

cv0 = (
    cv[
        cv["fold"] == 0
    ]
    .sort_values(
        ["prefix", "dataset"]
    )
    .reset_index(drop=True)
)

assert len(cv0) == 40

names_44 = (
    cv0.loc[
        cv0["dataset"].str.startswith("44b6_"),
        "dataset",
    ]
    .tolist()
)

names_6 = (
    cv0.loc[
        cv0["dataset"].str.startswith("6bba_"),
        "dataset",
    ]
    .tolist()
)

assert len(names_44) == 15
assert len(names_6) == 25

prefix_map = dict(
    zip(
        cv0["dataset"],
        cv0["prefix"],
    )
)

estimated_nodes = dict(
    zip(
        cv0["dataset"],
        cv0["estimated_nodes"].astype(float),
    )
)


# ============================================================
# 4. Build Frozen-V9 route policies
# ============================================================

policies_44 = fv9.build_frozen_v9_policies(
    names_44,
    S9995_995,
    S9995_9975,
    TRAIN_ROOT,
    prefix_map=prefix_map,
    config=fv9.FROZEN_V9_CONFIG,
    progress=False,
)

policies_6 = fv9.build_frozen_v9_policies(
    names_6,
    B256_995,
    B256_9975,
    TRAIN_ROOT,
    prefix_map=prefix_map,
    config=fv9.FROZEN_V9_CONFIG,
    progress=False,
)

actions = pd.read_pickle(
    ACTION_PATH
)


# ============================================================
# 5. Current-best graph builder
# ============================================================

def build_current_best(name):

    is_44 = name.startswith(
        "44b6_"
    )

    if is_44:
        p995 = S9995_995
        p9975 = S9995_9975
        policies = policies_44
    else:
        p995 = B256_995
        p9975 = B256_9975
        policies = policies_6

    graph, scale, _ = fv9.apply_frozen_v9(
        name,
        p995,
        p9975,
        policies,
        TRAIN_ROOT,
        config=fv9.FROZEN_V9_CONFIG,
    )

    # MORPH-DIV-V2 applies only to 6bba.
    ds_actions = (
        actions[
            actions["dataset"] == name
        ]
        if not is_44
        else actions.iloc[0:0]
    )

    if len(ds_actions) == 0:
        return graph, scale

    nodes = (
        graph.node_attrs(
            attr_keys=[
                "node_id", "t", "z", "y", "x",
            ]
        )
        .to_pandas()
        .set_index("node_id")
    )

    edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
    )

    outdeg = (
        edges.groupby("source_id")
        .size()
        .astype(int)
        .to_dict()
    )

    indeg = (
        edges.groupby("target_id")
        .size()
        .astype(int)
        .to_dict()
    )

    edge_set = set(
        zip(
            edges["source_id"].astype(int),
            edges["target_id"].astype(int),
        )
    )

    ordered = ds_actions.sort_values(
        [
            "morph_rank",
            "t",
            "source_pred_node_id",
            "target_pred_node_id",
        ]
    )

    for a in ordered.itertuples(
        index=False
    ):

        src = int(
            a.source_pred_node_id
        )

        tgt = int(
            a.target_pred_node_id
        )

        assert outdeg.get(src, 0) == 1
        assert indeg.get(tgt, 0) == 0
        assert (src, tgt) not in edge_set

        dist = fv9.recovered_edge_dist(
            nodes,
            src,
            tgt,
            fv9.FROZEN_V9_CONFIG,
        )

        graph.add_edge(
            src,
            tgt,
            {
                "edge_dist":
                    float(dist),

                "edge_prob":
                    float(
                        a.candidate_prob
                    ),
            },
        )

        outdeg[src] = 2
        indeg[tgt] = 1
        edge_set.add(
            (src, tgt)
        )

    return graph, scale


# ============================================================
# 6. Residual audit
# ============================================================

metric_records = []
dataset_rows = []
fn_rows = []
fp_rows = []


for i, name in enumerate(
    cv0["dataset"],
    start=1,
):

    prefix = prefix_map[name]

    graph, scale = build_current_best(
        name
    )

    gt = fv9.load_graph(
        TRAIN_ROOT
        / f"{name}.geff"
    )

    # Official evaluation performs node matching in place.
    metric = fv9.evaluate_graph(
        graph,
        gt,
        scale,
        estimated_nodes[name],
        fv9.FROZEN_V9_CONFIG,
    )

    metric_records.append(
        metric
    )

    # Exact official pred_valid / matched mask logic.
    edge_eval = (
        _evaluate_matched_graph(
            graph,
            gt,
        )
        .to_pandas()
    )

    pred_nodes = (
        graph.node_attrs(
            unpack=True
        )
        .to_pandas()
    )

    pred_edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
    )

    gt_edges = (
        gt.edge_attrs(
            unpack=True
        )
        .to_pandas()
    )


    # --------------------------------------------------------
    # Node matching maps
    # --------------------------------------------------------

    pred_nodes[
        "match_node_id"
    ] = (
        pd.to_numeric(
            pred_nodes[
                "match_node_id"
            ],
            errors="coerce",
        )
        .fillna(-1)
        .astype(int)
    )

    matched_nodes = (
        pred_nodes[
            pred_nodes[
                "match_node_id"
            ] >= 0
        ]
    )

    # Official matching is one-to-one.
    assert not matched_nodes[
        "match_node_id"
    ].duplicated().any()

    pred_to_gt = dict(
        zip(
            matched_nodes[
                "node_id"
            ].astype(int),

            matched_nodes[
                "match_node_id"
            ].astype(int),
        )
    )

    gt_to_pred = {
        gt_id: pred_id
        for pred_id, gt_id
        in pred_to_gt.items()
    }


    # --------------------------------------------------------
    # Exact TP / FP sets from official edge accounting
    # --------------------------------------------------------

    matched_mask = (
        edge_eval[
            "matched_edge_mask"
        ].astype(bool)
    )

    valid_mask = (
        edge_eval[
            "pred_valid"
        ].astype(bool)
    )

    tp_edges = edge_eval[
        matched_mask
    ]

    valid_fp_edges = edge_eval[
        valid_mask
        &
        ~matched_mask
    ]


    assert len(tp_edges) == int(
        metric["edge_tp"]
    )

    assert len(valid_fp_edges) == int(
        metric["edge_fp"]
    )


    # --------------------------------------------------------
    # Convert TP prediction edges back to GT edge identities
    # --------------------------------------------------------

    matched_gt_pairs = set()

    for e in tp_edges.itertuples(
        index=False
    ):

        ps = int(
            e.source_id
        )

        pt = int(
            e.target_id
        )

        assert ps in pred_to_gt
        assert pt in pred_to_gt

        matched_gt_pairs.add(
            (
                pred_to_gt[ps],
                pred_to_gt[pt],
            )
        )


    assert (
        len(matched_gt_pairs)
        == int(metric["edge_tp"])
    )


    gt_edge_set = set(
        zip(
            gt_edges[
                "source_id"
            ].astype(int),

            gt_edges[
                "target_id"
            ].astype(int),
        )
    )

    fn_pairs = (
        gt_edge_set
        - matched_gt_pairs
    )

    assert len(fn_pairs) == int(
        metric["edge_fn"]
    )


    # --------------------------------------------------------
    # Degree context
    # --------------------------------------------------------

    pred_outdeg = (
        pred_edges.groupby(
            "source_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )

    pred_indeg = (
        pred_edges.groupby(
            "target_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )

    gt_outdeg = (
        gt_edges.groupby(
            "source_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )

    valid_fp_out = (
        valid_fp_edges.groupby(
            "source_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )

    valid_fp_in = (
        valid_fp_edges.groupby(
            "target_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    # --------------------------------------------------------
    # FN root-cause decomposition
    # --------------------------------------------------------

    for gsrc, gtgt in sorted(
        fn_pairs
    ):

        ps = gt_to_pred.get(
            gsrc
        )

        pt = gt_to_pred.get(
            gtgt
        )

        src_present = (
            ps is not None
        )

        tgt_present = (
            pt is not None
        )

        if (
            not src_present
            and not tgt_present
        ):
            cause = (
                "DETECTION_BOTH_MISSING"
            )

        elif not src_present:
            cause = (
                "DETECTION_SOURCE_MISSING"
            )

        elif not tgt_present:
            cause = (
                "DETECTION_TARGET_MISSING"
            )

        else:
            cause = (
                "LINKING_LIMITED"
            )


        if cause == "LINKING_LIMITED":

            src_out = int(
                pred_outdeg.get(
                    ps,
                    0,
                )
            )

            tgt_in = int(
                pred_indeg.get(
                    pt,
                    0,
                )
            )

            if tgt_in == 0:
                topology = (
                    f"SRC_OUT{src_out}_TARGET_FREE"
                )
            else:
                topology = (
                    f"SRC_OUT{src_out}_TARGET_OCCUPIED"
                )

            src_fp = int(
                valid_fp_out.get(
                    ps,
                    0,
                )
            )

            tgt_fp = int(
                valid_fp_in.get(
                    pt,
                    0,
                )
            )

        else:

            src_out = (
                int(
                    pred_outdeg.get(
                        ps,
                        0,
                    )
                )
                if ps is not None
                else -1
            )

            tgt_in = (
                int(
                    pred_indeg.get(
                        pt,
                        0,
                    )
                )
                if pt is not None
                else -1
            )

            topology = (
                "DETECTION_LIMITED"
            )

            src_fp = 0
            tgt_fp = 0


        fn_rows.append({
            "dataset":
                name,

            "prefix":
                prefix,

            "gt_source_id":
                int(gsrc),

            "gt_target_id":
                int(gtgt),

            "pred_source_id":
                (
                    int(ps)
                    if ps is not None
                    else -1
                ),

            "pred_target_id":
                (
                    int(pt)
                    if pt is not None
                    else -1
                ),

            "cause":
                cause,

            "gt_parent_outdegree":
                int(
                    gt_outdeg.get(
                        gsrc,
                        0,
                    )
                ),

            "pred_source_outdegree":
                src_out,

            "pred_target_indegree":
                tgt_in,

            "link_topology":
                topology,

            "source_has_valid_fp_out":
                bool(
                    src_fp > 0
                ),

            "target_has_valid_fp_in":
                bool(
                    tgt_fp > 0
                ),

            "swap_supported":
                bool(
                    src_fp > 0
                    or tgt_fp > 0
                ),
        })


    # --------------------------------------------------------
    # Exact official FP endpoint-support decomposition
    # --------------------------------------------------------

    for e in valid_fp_edges.itertuples(
        index=False
    ):

        ps = int(
            e.source_id
        )

        pt = int(
            e.target_id
        )

        sg = pred_to_gt.get(
            ps
        )

        tg = pred_to_gt.get(
            pt
        )

        if (
            sg is not None
            and tg is not None
        ):
            support = (
                "BOTH_ENDPOINTS_GT_MATCHED"
            )

        elif sg is not None:
            support = (
                "SOURCE_ONLY_GT_MATCHED"
            )

        elif tg is not None:
            support = (
                "TARGET_ONLY_GT_MATCHED"
            )

        else:
            support = (
                "NEITHER_ENDPOINT_GT_MATCHED"
            )


        fp_rows.append({
            "dataset":
                name,

            "prefix":
                prefix,

            "source_pred_node_id":
                ps,

            "target_pred_node_id":
                pt,

            "source_gt_node_id":
                (
                    int(sg)
                    if sg is not None
                    else -1
                ),

            "target_gt_node_id":
                (
                    int(tg)
                    if tg is not None
                    else -1
                ),

            "endpoint_support":
                support,

            "pred_source_outdegree":
                int(
                    pred_outdeg.get(
                        ps,
                        0,
                    )
                ),

            "pred_target_indegree":
                int(
                    pred_indeg.get(
                        pt,
                        0,
                    )
                ),
        })


    dataset_rows.append({
        "dataset":
            name,

        "prefix":
            prefix,

        "edge_tp":
            int(
                metric["edge_tp"]
            ),

        "edge_fp":
            int(
                metric["edge_fp"]
            ),

        "edge_fn":
            int(
                metric["edge_fn"]
            ),

        "node_recall":
            float(
                metric["node_recall"]
            ),
    })


    if (
        i == 1
        or i % 10 == 0
        or i == 40
    ):
        print(
            f"{i:2d}/40 datasets"
        )


# ============================================================
# 7. Global consistency gates
# ============================================================

fn_df = pd.DataFrame(
    fn_rows
)

fp_df = pd.DataFrame(
    fp_rows
)

dataset_df = pd.DataFrame(
    dataset_rows
)


total_tp = int(
    dataset_df[
        "edge_tp"
    ].sum()
)

total_fp = int(
    dataset_df[
        "edge_fp"
    ].sum()
)

total_fn = int(
    dataset_df[
        "edge_fn"
    ].sum()
)


assert len(fn_df) == total_fn
assert len(fp_df) == total_fp


summary_metric = fv9.summarise(
    metric_records
)


print(
    "\n========== CURRENT-BEST REPRODUCTION =========="
)

print(
    "edge TP/FP/FN:",
    total_tp,
    "/",
    total_fp,
    "/",
    total_fn,
)

print(
    "edge J       :",
    f"{summary_metric['edge_jaccard']:.9f}",
)

print(
    "adj edge J   :",
    f"{summary_metric['adj_edge_jaccard']:.9f}",
)

print(
    "score        :",
    f"{summary_metric['score']:.9f}",
)


assert abs(
    summary_metric[
        "edge_jaccard"
    ]
    - 0.781996647
) <= 1e-9

assert abs(
    summary_metric[
        "score"
    ]
    - 0.778973147
) <= 1e-9


# ============================================================
# 8. FN root cause
# ============================================================

fn_df[
    "root_cause"
] = np.where(
    fn_df[
        "cause"
    ].eq(
        "LINKING_LIMITED"
    ),
    "LINKING_LIMITED",
    "DETECTION_LIMITED",
)


print(
    "\n========== EDGE FN ROOT CAUSE =========="
)

root_counts = (
    fn_df[
        "root_cause"
    ]
    .value_counts()
)

for key in [
    "DETECTION_LIMITED",
    "LINKING_LIMITED",
]:

    n = int(
        root_counts.get(
            key,
            0,
        )
    )

    print(
        f"{key:20s}: "
        f"{n:5d} "
        f"({n / total_fn:.3%})"
    )


print(
    "\n========== DETECTION-LIMITED DETAIL =========="
)

print(
    fn_df.loc[
        fn_df[
            "root_cause"
        ]
        == "DETECTION_LIMITED",
        "cause",
    ]
    .value_counts()
    .to_string()
)


print(
    "\n========== FN ROOT CAUSE BY PREFIX =========="
)

print(
    pd.crosstab(
        fn_df["prefix"],
        fn_df["root_cause"],
    ).to_string()
)


# ============================================================
# 9. Linking-limited topology
# ============================================================

link = fn_df[
    fn_df[
        "root_cause"
    ]
    == "LINKING_LIMITED"
].copy()


print(
    "\n========== LINKING-LIMITED TOPOLOGY =========="
)

print(
    link[
        "link_topology"
    ]
    .value_counts()
    .to_string()
)


print(
    "\n========== LINKING FN — GT PARENT TYPE =========="
)

print(
    link[
        "gt_parent_outdegree"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)


print(
    "\n========== LINKING FN — VALID-FP SUBSTITUTION SUPPORT =========="
)

print(
    "linking FNs:",
    len(link),
)

print(
    "source has valid FP out:",
    int(
        link[
            "source_has_valid_fp_out"
        ].sum()
    ),
)

print(
    "target has valid FP in :",
    int(
        link[
            "target_has_valid_fp_in"
        ].sum()
    ),
)

print(
    "either side supports swap:",
    int(
        link[
            "swap_supported"
        ].sum()
    ),
)

if len(link):

    print(
        "swap-supported fraction:",
        f"{link['swap_supported'].mean():.3%}"
    )


# ============================================================
# 10. Official FP anatomy
# ============================================================

print(
    "\n========== OFFICIAL VALID FP ENDPOINT SUPPORT =========="
)

print(
    fp_df[
        "endpoint_support"
    ]
    .value_counts()
    .to_string()
)


print(
    "\n========== VALID FP SUPPORT BY PREFIX =========="
)

print(
    pd.crosstab(
        fp_df["prefix"],
        fp_df["endpoint_support"],
    ).to_string()
)


# ============================================================
# 11. Persist
# ============================================================

fn_df.to_pickle(
    FN_OUT
)

fp_df.to_pickle(
    FP_OUT
)

dataset_df.to_pickle(
    DATASET_OUT
)


payload = {
    "stage":
        "12.00D",

    "current_best_score":
        float(
            summary_metric[
                "score"
            ]
        ),

    "edge_tp":
        total_tp,

    "edge_fp":
        total_fp,

    "edge_fn":
        total_fn,

    "detection_limited_fn":
        int(
            (
                fn_df[
                    "root_cause"
                ]
                == "DETECTION_LIMITED"
            ).sum()
        ),

    "linking_limited_fn":
        int(
            (
                fn_df[
                    "root_cause"
                ]
                == "LINKING_LIMITED"
            ).sum()
        ),

    "linking_swap_supported":
        int(
            link[
                "swap_supported"
            ].sum()
        ),
}


SUMMARY_OUT.write_text(
    json.dumps(
        payload,
        indent=2,
    ),
    encoding="utf-8",
)


print(
    "\nwritten:",
    FN_OUT
)

print(
    "written:",
    FP_OUT
)

print(
    "written:",
    DATASET_OUT
)

print(
    "written:",
    SUMMARY_OUT
)


print(
    "\n========== 12.00D DECISION =========="
)

print(
    "CURRENT_BEST_EDGE_RESIDUAL_CENSUS: PASS"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.01A Linking-FN Raw Edge Provenance
# Purpose:
#   Determine whether each current-best LINKING_LIMITED FN:
#
#   A) already existed as an edge in one of the raw prediction
#      graphs but was lost by Frozen-V9/postprocessing, or
#
#   B) was absent from both raw graphs, implying an upstream
#      association-generation/ranking miss.
#
# Changes files: YES
#   reports/stage12_residual_audit/
#       stage12_linking_fn_raw_edge_provenance.pkl
#       stage12_linking_fn_raw_edge_provenance_summary.json
#
# Runs training/inference: NO
# Uses GT: NO NEW GT READ
#   Uses the already-frozen GT-derived 12.00D residual artifact.
# Changes Stage11 policy: NO
# Keep after running: YES
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd

from biohub_cell_tracking import frozen_v9 as fv9


PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

PRED_ROOT = (
    PROJECT
    / "predictions/heqiuyan"
)

RPT = (
    PROJECT
    / "reports/stage12_residual_audit"
)

FN_PATH = (
    RPT
    / "stage12_edge_fn_residuals.pkl"
)

OUT = (
    RPT
    / "stage12_linking_fn_raw_edge_provenance.pkl"
)

SUMMARY_OUT = (
    RPT
    / "stage12_linking_fn_raw_edge_provenance_summary.json"
)


# ============================================================
# 1. Frozen route roots
# ============================================================

ROUTES = {
    "44b6": {
        "det995":
            PRED_ROOT
            / "stage10_infer_sparse9995_256_det995"
            / "split_0",

        "det9975":
            PRED_ROOT
            / "stage10_infer_sparse9995_256_det9975"
            / "split_0",
    },

    "6bba": {
        "det995":
            PRED_ROOT
            / "stage10_infer_baseline_256_det995"
            / "split_0",

        "det9975":
            PRED_ROOT
            / "stage10_infer_baseline_256_det9975"
            / "split_0",
    },
}


for route in ROUTES.values():

    for root in route.values():

        assert root.is_dir(), root


# ============================================================
# 2. Locked linking-FN population
# ============================================================

fn = pd.read_pickle(
    FN_PATH
)

link = (
    fn[
        fn["root_cause"]
        == "LINKING_LIMITED"
    ]
    .copy()
    .reset_index(drop=True)
)


assert len(link) == 2290

assert (
    link["pred_source_id"] >= 0
).all()

assert (
    link["pred_target_id"] >= 0
).all()


print(
    "========== 12.01A LINKING POPULATION =========="
)

print(
    "linking FNs:",
    len(link)
)

print(
    "44b6:",
    int(
        (link["prefix"] == "44b6").sum()
    )
)

print(
    "6bba:",
    int(
        (link["prefix"] == "6bba").sum()
    )
)


# ============================================================
# 3. Raw-edge lookup helper
# ============================================================

def edge_lookup(root, dataset):

    graph = fv9.load_graph(
        root
        / f"{dataset}.geff"
    )

    nodes = (
        graph.node_attrs(
            attr_keys=[
                "node_id",
            ]
        )
        .to_pandas()
    )

    node_ids = set(
        nodes["node_id"]
        .astype(int)
    )

    edges = (
        graph.edge_attrs(
            attr_keys=[
                "edge_prob",
                "edge_dist",
            ],
            unpack=True,
        )
        .to_pandas()
    )

    assert not edges[
        [
            "source_id",
            "target_id",
        ]
    ].duplicated().any()

    lookup = {}

    for row in edges.itertuples(
        index=False
    ):

        lookup[
            (
                int(row.source_id),
                int(row.target_id),
            )
        ] = (
            float(row.edge_prob),
            float(row.edge_dist),
        )

    return node_ids, lookup


# ============================================================
# 4. Audit exact correct-edge provenance
# ============================================================

records = []


for i, (
    dataset,
    rows,
) in enumerate(
    link.groupby(
        "dataset",
        sort=True,
    ),
    start=1,
):

    prefix = str(
        rows.iloc[0]["prefix"]
    )

    route = ROUTES[
        prefix
    ]


    nodes995, edges995 = edge_lookup(
        route["det995"],
        dataset,
    )

    nodes9975, edges9975 = edge_lookup(
        route["det9975"],
        dataset,
    )


    for row in rows.itertuples(
        index=False
    ):

        src = int(
            row.pred_source_id
        )

        tgt = int(
            row.pred_target_id
        )

        pair = (
            src,
            tgt,
        )


        # Final-V9 matched nodes should always originate
        # from the loose det995 prediction universe.
        assert src in nodes995
        assert tgt in nodes995


        e995 = edges995.get(
            pair
        )

        e9975 = edges9975.get(
            pair
        )


        present995 = (
            e995 is not None
        )

        present9975 = (
            e9975 is not None
        )


        if (
            present995
            and present9975
        ):

            provenance = (
                "RAW_BOTH_PRESENT"
            )

        elif present995:

            provenance = (
                "RAW995_ONLY"
            )

        elif present9975:

            provenance = (
                "RAW9975_ONLY"
            )

        else:

            provenance = (
                "RAW_ASSOCIATION_MISS"
            )


        raw_probs = [
            x[0]
            for x in [
                e995,
                e9975,
            ]
            if x is not None
        ]


        records.append({
            "dataset":
                dataset,

            "prefix":
                prefix,

            "gt_source_id":
                int(
                    row.gt_source_id
                ),

            "gt_target_id":
                int(
                    row.gt_target_id
                ),

            "pred_source_id":
                src,

            "pred_target_id":
                tgt,

            "link_topology":
                row.link_topology,

            "gt_parent_outdegree":
                int(
                    row.gt_parent_outdegree
                ),

            "swap_supported":
                bool(
                    row.swap_supported
                ),

            "raw995_present":
                present995,

            "raw9975_present":
                present9975,

            "raw995_prob":
                (
                    e995[0]
                    if e995 is not None
                    else np.nan
                ),

            "raw9975_prob":
                (
                    e9975[0]
                    if e9975 is not None
                    else np.nan
                ),

            "raw995_dist":
                (
                    e995[1]
                    if e995 is not None
                    else np.nan
                ),

            "raw9975_dist":
                (
                    e9975[1]
                    if e9975 is not None
                    else np.nan
                ),

            "best_raw_prob":
                (
                    max(raw_probs)
                    if raw_probs
                    else np.nan
                ),

            "provenance":
                provenance,
        })


    if (
        i == 1
        or i % 10 == 0
        or i == link["dataset"].nunique()
    ):

        print(
            f"{i:2d}/"
            f"{link['dataset'].nunique()} datasets"
        )


audit = pd.DataFrame(
    records
)


assert len(audit) == 2290


# ============================================================
# 5. Main provenance verdict
# ============================================================

print(
    "\n========== LINKING-FN RAW PROVENANCE =========="
)

counts = (
    audit[
        "provenance"
    ]
    .value_counts()
)


for key in [
    "RAW_BOTH_PRESENT",
    "RAW995_ONLY",
    "RAW9975_ONLY",
    "RAW_ASSOCIATION_MISS",
]:

    n = int(
        counts.get(
            key,
            0,
        )
    )

    print(
        f"{key:22s}: "
        f"{n:5d} "
        f"({n / len(audit):.3%})"
    )


postprocess_lost = int(
    (
        audit[
            "provenance"
        ]
        != "RAW_ASSOCIATION_MISS"
    ).sum()
)

association_miss = int(
    (
        audit[
            "provenance"
        ]
        == "RAW_ASSOCIATION_MISS"
    ).sum()
)


print(
    "\nPOSTPROCESS-LOST :",
    postprocess_lost,
    f"({postprocess_lost / len(audit):.3%})"
)

print(
    "RAW ASSOC MISS   :",
    association_miss,
    f"({association_miss / len(audit):.3%})"
)


# ============================================================
# 6. Prefix / topology decomposition
# ============================================================

print(
    "\n========== PROVENANCE BY PREFIX =========="
)

print(
    pd.crosstab(
        audit["prefix"],
        audit["provenance"],
    ).to_string()
)


print(
    "\n========== PROVENANCE BY FINAL TOPOLOGY =========="
)

print(
    pd.crosstab(
        audit["link_topology"],
        audit["provenance"],
    ).to_string()
)


# ============================================================
# 7. Raw confidence of edges that V9 lost
# ============================================================

lost = audit[
    audit[
        "provenance"
    ]
    != "RAW_ASSOCIATION_MISS"
]


print(
    "\n========== LOST TRUE-EDGE RAW CONFIDENCE =========="
)

if len(lost):

    print(
        "n:",
        len(lost)
    )

    print(
        "best raw prob q10/q25/q50/q75/q90:",
        np.round(
            lost[
                "best_raw_prob"
            ]
            .quantile(
                [
                    .10,
                    .25,
                    .50,
                    .75,
                    .90,
                ]
            )
            .to_numpy(),
            6,
        )
    )


print(
    "\n========== LOST TRUE EDGE + FP-SWAP SUPPORT =========="
)

lost_swap = (
    lost[
        "swap_supported"
    ]
)

print(
    "lost edges with swap support:",
    int(
        lost_swap.sum()
    ),
    "/",
    len(lost),
)

if len(lost):

    print(
        "fraction:",
        f"{lost_swap.mean():.3%}"
    )


# ============================================================
# 8. Persist
# ============================================================

audit.to_pickle(
    OUT
)


summary = {
    "stage":
        "12.01A",

    "linking_fn":
        int(
            len(audit)
        ),

    "postprocess_lost":
        postprocess_lost,

    "raw_association_miss":
        association_miss,

    "postprocess_lost_fraction":
        float(
            postprocess_lost
            / len(audit)
        ),

    "raw_association_miss_fraction":
        float(
            association_miss
            / len(audit)
        ),
}


SUMMARY_OUT.write_text(
    json.dumps(
        summary,
        indent=2,
    ),
    encoding="utf-8",
)


print(
    "\nwritten:",
    OUT
)

print(
    "written:",
    SUMMARY_OUT
)


print(
    "\n========== 12.01A DECISION =========="
)

print(
    "LINKING_FN_RAW_EDGE_PROVENANCE: PASS"
)

In [ ]:
# ============================================================
# RUN — Stage 12.02B Association Selector Barrier Audit
# Purpose:
#   Diagnose WHY the 1,552 known 6bba RAW association misses
#   are absent from the persisted raw association graph.
#
#   For every correct source -> target candidate, measure:
#
#       - correct target-column rank
#       - raw source outdegree
#       - raw target indegree
#       - whether a free legal slot exists
#       - probability of the retained competing edge
#
#   This separates:
#
#       LOW_CONFIDENCE_FREE_SLOT
#       TARGET_OCCUPIED
#       SOURCE_CAPACITY_BLOCKED
#       BOTH_BLOCKED
#
# Changes files: YES
#   reports/stage12_residual_audit/
#       stage12_6bba_assoc_selector_barrier_audit.pkl
#       stage12_6bba_assoc_selector_barrier_summary.json
#
# Runs training: NO
# Runs inference: YES — sparse association replay only
# Uses new GT: NO
#   Uses already-frozen residual identities.
# Changes frozen policy: NO
# Keep after running: NO
#   Delete this RUN cell after successful verification;
#   retain artifacts + RESULT Markdown.
# ============================================================

from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import torch


# ============================================================
# 0. Paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

PRED995 = (
    PROJECT
    / "predictions/heqiuyan/"
      "stage10_infer_baseline_256_det995/"
      "split_0"
)

RPT = (
    PROJECT
    / "reports/stage12_residual_audit"
)

PROV_PATH = (
    RPT
    / "stage12_linking_fn_raw_edge_provenance.pkl"
)

OUT = (
    RPT
    / "stage12_6bba_assoc_selector_barrier_audit.pkl"
)

SUMMARY_OUT = (
    RPT
    / "stage12_6bba_assoc_selector_barrier_summary.json"
)

CHECKPOINT = (
    PROJECT
    / "models/official_baseline/"
      "stage10_pilot_baseline_256/"
      "split_0/edge_predictor_best.pth"
)


sys.path.insert(
    0,
    str(PROJECT / "scripts"),
)

import run_stage11_multiframe_extract as runner
from replay_stage11_association import DatasetReplay


# ============================================================
# 1. Frozen residual population
# ============================================================

prov = pd.read_pickle(
    PROV_PATH
)

miss = (
    prov[
        (prov["prefix"] == "6bba")
        &
        (
            prov["provenance"]
            == "RAW_ASSOCIATION_MISS"
        )
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(miss) == 1552


# ============================================================
# 2. Frozen Baseline256 model
# ============================================================

device = runner.resolve_device(
    "mps"
)

model, window_size, downsample = runner.load_model(
    CHECKPOINT,
    device,
)

assert window_size == 2
assert tuple(downsample) == (1, 4, 4)

DS = np.asarray(
    downsample,
    dtype=np.float64,
)


def as_numpy(x):

    if torch.is_tensor(x):
        return x.detach().cpu().numpy()

    return np.asarray(x)


def replay_ids(
    coords,
    persisted_nodes,
):

    c = as_numpy(coords)

    if c.shape[1] == 4:
        c = c[:, -3:]

    xyz = np.rint(
        c * DS
    ).astype(np.int64)

    lookup = {
        (
            int(r.z),
            int(r.y),
            int(r.x),
        ):
            int(r.node_id)

        for r in persisted_nodes.itertuples(
            index=False
        )
    }

    ids = np.asarray(
        [
            lookup.get(
                tuple(map(int, p)),
                -1,
            )
            for p in xyz
        ],
        dtype=np.int64,
    )

    return ids


# ============================================================
# 3. Replay required pairs
# ============================================================

records = []

datasets = sorted(
    miss["dataset"].unique()
)


for di, dataset in enumerate(
    datasets,
    start=1,
):

    pred_graph = runner.load_graph(
        PRED995
        / f"{dataset}.geff"
    )

    nodes = (
        pred_graph.node_attrs(
            attr_keys=[
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
    )

    edges = (
        pred_graph.edge_attrs(
            attr_keys=[
                "edge_prob",
            ],
            unpack=True,
        )
        .to_pandas()
    )

    node_time = dict(
        zip(
            nodes["node_id"].astype(int),
            nodes["t"].astype(int),
        )
    )

    out_map = (
        edges.groupby("source_id")
        .size()
        .astype(int)
        .to_dict()
    )

    in_map = (
        edges.groupby("target_id")
        .size()
        .astype(int)
        .to_dict()
    )

    incoming_prob = {
        int(r.target_id):
            float(r.edge_prob)

        for r in edges.itertuples(
            index=False
        )
    }

    ds_rows = (
        miss[
            miss["dataset"] == dataset
        ]
        .copy()
    )

    ds_rows["t"] = (
        ds_rows[
            "pred_source_id"
        ]
        .map(node_time)
    )

    assert ds_rows["t"].notna().all()

    ds_rows["t"] = (
        ds_rows["t"]
        .astype(int)
    )


    replay = DatasetReplay(
        dataset,
        TRAIN_ROOT,
        model,
        window_size,
        downsample,
        device,
    )


    for t, group in ds_rows.groupby(
        "t",
        sort=True,
    ):

        t = int(t)

        c_src, c_tgt, raw, probs = (
            replay.replay_pair(t)
        )

        probs = as_numpy(
            probs
        ).astype(
            np.float64,
            copy=False,
        )

        src_nodes = (
            nodes[
                nodes["t"].astype(int) == t
            ]
            .reset_index(drop=True)
        )

        tgt_nodes = (
            nodes[
                nodes["t"].astype(int) == t + 1
            ]
            .reset_index(drop=True)
        )

        src_ids = replay_ids(
            c_src,
            src_nodes,
        )

        tgt_ids = replay_ids(
            c_tgt,
            tgt_nodes,
        )

        assert probs.shape == (
            len(src_ids),
            len(tgt_ids),
        )


        for row in group.itertuples(
            index=False
        ):

            src = int(
                row.pred_source_id
            )

            tgt = int(
                row.pred_target_id
            )

            si = np.flatnonzero(
                src_ids == src
            )

            tj = np.flatnonzero(
                tgt_ids == tgt
            )

            assert len(si) == 1
            assert len(tj) == 1

            i = int(si[0])
            j = int(tj[0])

            correct_prob = float(
                probs[i, j]
            )

            # ----------------------------------------
            # Column competition:
            # softmax(dim=0) => sources compete for target.
            # ----------------------------------------

            column = probs[:, j]

            column_order = np.argsort(
                -column,
                kind="stable",
            )

            column_rank = (
                int(
                    np.flatnonzero(
                        column_order == i
                    )[0]
                )
                + 1
            )

            column_top1 = float(
                column[
                    column_order[0]
                ]
            )

            column_margin = float(
                correct_prob
                - column_top1
            )


            source_out = int(
                out_map.get(
                    src,
                    0,
                )
            )

            target_in = int(
                in_map.get(
                    tgt,
                    0,
                )
            )

            source_capacity_free = (
                source_out < 2
            )

            target_free = (
                target_in == 0
            )


            if (
                source_capacity_free
                and target_free
            ):

                barrier = (
                    "LOW_CONFIDENCE_FREE_SLOT"
                )

            elif (
                source_capacity_free
                and not target_free
            ):

                barrier = (
                    "TARGET_OCCUPIED"
                )

            elif (
                not source_capacity_free
                and target_free
            ):

                barrier = (
                    "SOURCE_CAPACITY_BLOCKED"
                )

            else:

                barrier = (
                    "BOTH_BLOCKED"
                )


            retained_in_prob = (
                incoming_prob.get(
                    tgt,
                    np.nan,
                )
            )


            records.append({
                "dataset":
                    dataset,

                "t":
                    t,

                "pred_source_id":
                    src,

                "pred_target_id":
                    tgt,

                "correct_prob":
                    correct_prob,

                "column_rank":
                    column_rank,

                "column_top1_prob":
                    column_top1,

                "column_margin":
                    column_margin,

                "source_raw_outdegree":
                    source_out,

                "target_raw_indegree":
                    target_in,

                "source_capacity_free":
                    source_capacity_free,

                "target_free":
                    target_free,

                "barrier":
                    barrier,

                "retained_incoming_prob":
                    retained_in_prob,

                "correct_beats_retained":
                    bool(
                        np.isfinite(
                            retained_in_prob
                        )
                        and
                        correct_prob
                        > retained_in_prob
                    ),
            })


    if (
        di == 1
        or di % 5 == 0
        or di == len(datasets)
    ):

        print(
            f"{di:2d}/{len(datasets)} datasets"
        )


    if torch.backends.mps.is_available():
        torch.mps.empty_cache()


audit = pd.DataFrame(
    records
)

assert len(audit) == 1552


# ============================================================
# 4. Column-rank anatomy
# ============================================================

print(
    "\n========== CORRECT SOURCE COLUMN RANK =========="
)

for k in [
    1,
    2,
    3,
    5,
    10,
]:

    n = int(
        (
            audit[
                "column_rank"
            ]
            <= k
        ).sum()
    )

    print(
        f"column rank <= {k:2d}: "
        f"{n:4d} / {len(audit)} "
        f"({n/len(audit):.3%})"
    )


print(
    "column rank q25/q50/q75/q90:",
    np.round(
        audit[
            "column_rank"
        ]
        .quantile(
            [.25, .50, .75, .90]
        )
        .to_numpy(),
        3,
    )
)


# ============================================================
# 5. Raw selector barrier
# ============================================================

print(
    "\n========== RAW SELECTOR BARRIER =========="
)

barrier_counts = (
    audit[
        "barrier"
    ]
    .value_counts()
)

print(
    barrier_counts.to_string()
)


print(
    "\n========== RAW SOURCE OUTDEGREE =========="
)

print(
    audit[
        "source_raw_outdegree"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)


print(
    "\n========== RAW TARGET INDEGREE =========="
)

print(
    audit[
        "target_raw_indegree"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)


# ============================================================
# 6. Free-slot opportunity
# ============================================================

free = audit[
    audit[
        "barrier"
    ]
    == "LOW_CONFIDENCE_FREE_SLOT"
]


print(
    "\n========== FREE-SLOT LOW-CONFIDENCE OPPORTUNITY =========="
)

print(
    "rows:",
    len(free),
    "/",
    len(audit),
    f"({len(free)/len(audit):.3%})",
)


if len(free):

    print(
        "column-rank1:",
        int(
            (
                free[
                    "column_rank"
                ]
                == 1
            ).sum()
        ),
        "/",
        len(free),
    )

    print(
        "correct prob q10/q25/q50/q75/q90:",
        np.round(
            free[
                "correct_prob"
            ]
            .quantile(
                [
                    .10,
                    .25,
                    .50,
                    .75,
                    .90,
                ]
            )
            .to_numpy(),
            6,
        )
    )


# ============================================================
# 7. Occupied-target competition
# ============================================================

occupied = audit[
    audit[
        "target_raw_indegree"
    ]
    > 0
].copy()


print(
    "\n========== OCCUPIED-TARGET COMPETITION =========="
)

print(
    "rows:",
    len(occupied)
)


if len(occupied):

    margins = (
        occupied[
            "correct_prob"
        ]
        -
        occupied[
            "retained_incoming_prob"
        ]
    )

    print(
        "correct prob q25/q50/q75:",
        np.round(
            occupied[
                "correct_prob"
            ]
            .quantile(
                [.25, .50, .75]
            )
            .to_numpy(),
            6,
        )
    )

    print(
        "retained prob q25/q50/q75:",
        np.round(
            occupied[
                "retained_incoming_prob"
            ]
            .quantile(
                [.25, .50, .75]
            )
            .to_numpy(),
            6,
        )
    )

    print(
        "correct-retained margin q25/q50/q75:",
        np.round(
            margins.quantile(
                [.25, .50, .75]
            )
            .to_numpy(),
            6,
        )
    )

    print(
        "correct beats retained:",
        int(
            (
                margins > 0
            ).sum()
        ),
        "/",
        len(occupied),
    )


# ============================================================
# 8. Persist
# ============================================================

audit.to_pickle(
    OUT
)


payload = {
    "stage":
        "12.02B",

    "population":
        int(
            len(audit)
        ),

    "column_rank1":
        int(
            (
                audit[
                    "column_rank"
                ]
                == 1
            ).sum()
        ),

    "free_slot":
        int(
            len(free)
        ),

    "free_slot_column_rank1":
        int(
            (
                free[
                    "column_rank"
                ]
                == 1
            ).sum()
        ),

    "target_occupied":
        int(
            (
                audit[
                    "target_raw_indegree"
                ]
                > 0
            ).sum()
        ),

    "barrier_counts":
        {
            str(k):
                int(v)

            for k, v
            in barrier_counts.items()
        },
}


SUMMARY_OUT.write_text(
    json.dumps(
        payload,
        indent=2,
    ),
    encoding="utf-8",
)


print(
    "\nwritten:",
    OUT
)

print(
    "written:",
    SUMMARY_OUT
)


print(
    "\n========== 12.02B DECISION =========="
)

print(
    "ASSOCIATION_SELECTOR_BARRIER_AUDITED: PASS"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.02C Free-Slot Recovery Gate Anatomy
# Purpose:
#   Characterize the 772 LOW_CONFIDENCE_FREE_SLOT residuals
#   using only already-computed 12.02A / 12.02B artifacts.
#
#   Main questions:
#
#       1. How many are source-outdegree 0 vs 1?
#       2. How many are simultaneously row-rank1 + column-rank1?
#       3. What fraction survive observable probability gates?
#       4. Is there a conservative continuation-only pocket?
#
# Changes files: YES
#   reports/stage12_residual_audit/
#       stage12_free_slot_gate_anatomy.pkl
#
# Runs training/inference: NO
# Uses new GT: NO
# Changes frozen policy: NO
# Keep after running: YES
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

RPT = (
    PROJECT
    / "reports/stage12_residual_audit"
)

RANK_PATH = (
    RPT
    / "stage12_6bba_raw_assoc_miss_rank_audit.pkl"
)

BARRIER_PATH = (
    RPT
    / "stage12_6bba_assoc_selector_barrier_audit.pkl"
)

OUT = (
    RPT
    / "stage12_free_slot_gate_anatomy.pkl"
)


rank = pd.read_pickle(
    RANK_PATH
)

barrier = pd.read_pickle(
    BARRIER_PATH
)


assert len(rank) == 1552
assert len(barrier) == 1552


# ============================================================
# 1. Exact residual-key merge
# ============================================================

KEYS = [
    "dataset",
    "t",
    "pred_source_id",
    "pred_target_id",
]


merged = barrier.merge(
    rank[
        KEYS
        + [
            "prob_rank",
            "raw_rank",
        ]
    ],
    on=KEYS,
    how="left",
    validate="one_to_one",
)


assert len(merged) == 1552

assert (
    merged[
        "prob_rank"
    ]
    .notna()
    .all()
)


free = (
    merged[
        merged[
            "barrier"
        ]
        == "LOW_CONFIDENCE_FREE_SLOT"
    ]
    .copy()
    .reset_index(drop=True)
)


assert len(free) == 772


# ============================================================
# 2. Basic free-slot topology
# ============================================================

print(
    "========== FREE-SLOT SOURCE TOPOLOGY =========="
)

print(
    free[
        "source_raw_outdegree"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)


print(
    "\n========== FREE-SLOT ROW-RANK =========="
)

for k in [
    1,
    2,
    3,
    5,
]:

    n = int(
        (
            free[
                "prob_rank"
            ]
            <= k
        ).sum()
    )

    print(
        f"row rank <= {k}: "
        f"{n} / {len(free)} "
        f"({n/len(free):.3%})"
    )


print(
    "\n========== FREE-SLOT COLUMN-RANK =========="
)

for k in [
    1,
    2,
    3,
    5,
]:

    n = int(
        (
            free[
                "column_rank"
            ]
            <= k
        ).sum()
    )

    print(
        f"column rank <= {k}: "
        f"{n} / {len(free)} "
        f"({n/len(free):.3%})"
    )


# ============================================================
# 3. Mutual-rank agreement
# ============================================================

free[
    "mutual_rank1"
] = (
    free[
        "prob_rank"
    ].eq(1)
    &
    free[
        "column_rank"
    ].eq(1)
)


free[
    "mutual_rank2"
] = (
    free[
        "prob_rank"
    ].le(2)
    &
    free[
        "column_rank"
    ].le(2)
)


print(
    "\n========== MUTUAL RANK AGREEMENT =========="
)

print(
    "row1 & column1:",
    int(
        free[
            "mutual_rank1"
        ].sum()
    ),
    "/",
    len(free),
)

print(
    "row<=2 & column<=2:",
    int(
        free[
            "mutual_rank2"
        ].sum()
    ),
    "/",
    len(free),
)


# ============================================================
# 4. Conservative continuation-only pocket
#
# source_raw_outdegree == 0:
# adding an edge does NOT create a division.
# ============================================================

cont = (
    free[
        free[
            "source_raw_outdegree"
        ]
        == 0
    ]
    .copy()
)


print(
    "\n========== CONTINUATION-ONLY FREE SLOT =========="
)

print(
    "source outdegree0:",
    len(cont),
    "/",
    len(free),
    f"({len(cont)/len(free):.3%})",
)


print(
    "mutual rank1:",
    int(
        cont[
            "mutual_rank1"
        ].sum()
    )
)

print(
    "mutual rank<=2:",
    int(
        cont[
            "mutual_rank2"
        ].sum()
    )
)


# ============================================================
# 5. Probability-gated observable frontier
#
# These counts are NOT deployment policy performance.
# They only measure how many known residuals fall inside each
# possible observable gate.
# ============================================================

rows = []


for source_out in [
    0,
    1,
]:

    subset = free[
        free[
            "source_raw_outdegree"
        ]
        == source_out
    ]


    for mutual_k in [
        1,
        2,
        3,
    ]:

        rank_mask = (
            subset[
                "prob_rank"
            ].le(
                mutual_k
            )
            &
            subset[
                "column_rank"
            ].le(
                mutual_k
            )
        )


        for p in [
            0.10,
            0.15,
            0.20,
            0.25,
            0.30,
            0.35,
            0.40,
            0.45,
        ]:

            selected = (
                rank_mask
                &
                subset[
                    "correct_prob"
                ].ge(p)
            )


            rows.append({
                "source_outdegree":
                    source_out,

                "mutual_rank_cap":
                    mutual_k,

                "prob_threshold":
                    p,

                "known_residuals":
                    int(
                        selected.sum()
                    ),
            })


frontier = pd.DataFrame(
    rows
)


print(
    "\n========== OUTDEG0 MUTUAL-RANK FRONTIER =========="
)

print(
    frontier[
        frontier[
            "source_outdegree"
        ]
        == 0
    ]
    .pivot(
        index="prob_threshold",
        columns="mutual_rank_cap",
        values="known_residuals",
    )
    .to_string()
)


print(
    "\n========== OUTDEG1 MUTUAL-RANK FRONTIER =========="
)

print(
    frontier[
        frontier[
            "source_outdegree"
        ]
        == 1
    ]
    .pivot(
        index="prob_threshold",
        columns="mutual_rank_cap",
        values="known_residuals",
    )
    .to_string()
)


# ============================================================
# 6. Probability distributions by topology
# ============================================================

print(
    "\n========== FREE-SLOT PROBABILITY BY SOURCE OUTDEGREE =========="
)

for outdeg in [
    0,
    1,
]:

    s = free.loc[
        free[
            "source_raw_outdegree"
        ]
        == outdeg,
        "correct_prob",
    ]

    print(
        f"outdegree {outdeg}: "
        f"n={len(s)} | "
        f"q10/q25/q50/q75/q90="
        f"{np.round(s.quantile([.1,.25,.5,.75,.9]).to_numpy(), 6)}"
    )


# ============================================================
# 7. Persist
# ============================================================

free.to_pickle(
    OUT
)


print(
    "\nwritten:",
    OUT
)


print(
    "\n========== 12.02C DECISION =========="
)

print(
    "FREE_SLOT_RECOVERY_GATE_ANATOMY: PASS"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.03B-P1 Folds1–4 MORPH-DIV-V2 Exact Lock
# Purpose:
#   Extract and cryptographically lock the exact Stage11
#   K3_R3_P100 action set that Stage12 must treat as baseline.
#
#   Also reproduce both folds1–4 aggregate metric checkpoints
#   directly from the Stage11 per-dataset artifact.
#
# Changes files: YES
#   reports/stage12_residual_audit/
#       stage12_folds14_morphdiv_v2_lock.json
#
# Runs training/inference: NO
# Uses GT: NO NEW GT READ
# Changes frozen policy: NO
# Keep after running: YES
# ============================================================

from pathlib import Path
import hashlib
import json
import sys

import pandas as pd


PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

S11 = (
    PROJECT
    / "reports/stage11_division_probe"
)

S12 = (
    PROJECT
    / "reports/stage12_residual_audit"
)

S12.mkdir(
    parents=True,
    exist_ok=True,
)


GRID_PATH = (
    S11
    / "stage11_deployable_final_v9_policy_grid.pkl"
)

ACTIONS_PATH = (
    S11
    / "stage11_deployable_final_v9_policy_grid_actions.pkl"
)

DATASETS_PATH = (
    S11
    / "stage11_deployable_final_v9_policy_grid_datasets.pkl"
)

LOCK_OUT = (
    S12
    / "stage12_folds14_morphdiv_v2_lock.json"
)


sys.path.insert(
    0,
    str(PROJECT / "src"),
)

from biohub_cell_tracking import frozen_v9 as fv9


# ============================================================
# 1. Load Stage11 frozen artifacts
# ============================================================

grid = pd.read_pickle(
    GRID_PATH
)

actions_all = pd.read_pickle(
    ACTIONS_PATH
)

datasets = pd.read_pickle(
    DATASETS_PATH
)


assert len(grid) == 88
assert len(datasets) == 103 * 89


CONFIG = "K3_R3_P100"


row = (
    grid[
        grid["config_id"] == CONFIG
    ]
    .iloc[0]
)


actions = (
    actions_all[
        actions_all["config_id"] == CONFIG
    ]
    .copy()
    .sort_values(
        [
            "fold",
            "dataset",
            "morph_rank",
            "t",
            "source_pred_node_id",
            "target_pred_node_id",
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# 2. Exact action contract
# ============================================================

assert int(
    row["WHEN_K"]
) == 3

assert int(
    row["rank_cap"]
) == 3

assert abs(
    float(row["prob_threshold"]) - 0.10
) < 1e-12

assert len(actions) == 96

assert (
    actions["dataset"].nunique()
    == 63
)

assert (
    actions["morph_rank"] <= 3
).all()

assert (
    actions["candidate_rank"] <= 3
).all()

assert (
    actions["candidate_prob"] >= 0.10
).all()

assert set(
    actions["fold"].unique()
) == {1, 2, 3, 4}


# ============================================================
# 3. Reproduce aggregate metrics from per-dataset rows
# ============================================================

base_rows = (
    datasets[
        datasets["config_id"] == "BASELINE"
    ]
)

k3_rows = (
    datasets[
        datasets["config_id"] == CONFIG
    ]
)


assert len(base_rows) == 103
assert len(k3_rows) == 103


base = fv9.summarise(
    base_rows.to_dict(
        "records"
    )
)

k3 = fv9.summarise(
    k3_rows.to_dict(
        "records"
    )
)


print(
    "========== FOLDS1-4 BASELINE REPRODUCTION =========="
)

print(
    "score       :",
    f"{base['score']:.9f}"
)

print(
    "edge J      :",
    f"{base['edge_jaccard']:.9f}"
)

print(
    "adj edge J  :",
    f"{base['adj_edge_jaccard']:.9f}"
)

print(
    "division J  :",
    f"{base['division_jaccard']:.9f}"
)


print(
    "\n========== MORPH-DIV-V2 REPRODUCTION =========="
)

print(
    "config      :",
    CONFIG
)

print(
    "actions     :",
    len(actions)
)

print(
    "datasets    :",
    actions["dataset"].nunique()
)

print(
    "score       :",
    f"{k3['score']:.9f}"
)

print(
    "edge J      :",
    f"{k3['edge_jaccard']:.9f}"
)

print(
    "adj edge J  :",
    f"{k3['adj_edge_jaccard']:.9f}"
)

print(
    "division J  :",
    f"{k3['division_jaccard']:.9f}"
)

print(
    "delta score :",
    f"{k3['score'] - base['score']:+.9f}"
)


# Frozen Stage11 checkpoint gates.
assert abs(
    base["score"] - 0.780416707
) <= 1e-9

assert abs(
    k3["score"] - 0.781893060
) <= 1e-9

assert abs(
    (
        k3["score"]
        - base["score"]
    )
    - 0.001476352
) <= 1e-9


# ============================================================
# 4. Canonical action-set SHA
# ============================================================

LOCK_COLS = [
    "dataset",
    "fold",
    "morph_rank",
    "t",
    "source_pred_node_id",
    "target_pred_node_id",
    "candidate_rank",
    "candidate_prob",
]

blob = (
    actions[
        LOCK_COLS
    ]
    .to_csv(
        index=False,
        float_format="%.17g",
    )
    .encode()
)

action_sha = hashlib.sha256(
    blob
).hexdigest()


print(
    "\n========== EXACT ACTION LOCK =========="
)

print(
    "actions:",
    len(actions)
)

print(
    "SHA256 :",
    action_sha
)


# ============================================================
# 5. Persist Stage12 baseline lock
# ============================================================

payload = {
    "stage":
        "12.03B-P1",

    "baseline":
        "FROZEN_V9_PLUS_MORPH_DIV_V2",

    "config_id":
        CONFIG,

    "actions":
        int(len(actions)),

    "datasets_acted":
        int(
            actions["dataset"].nunique()
        ),

    "action_sha256":
        action_sha,

    "baseline_score":
        float(base["score"]),

    "morphdiv_score":
        float(k3["score"]),

    "delta_score":
        float(
            k3["score"]
            - base["score"]
        ),

    "source_actions_artifact":
        str(
            ACTIONS_PATH.relative_to(
                PROJECT
            )
        ),
}


LOCK_OUT.write_text(
    json.dumps(
        payload,
        indent=2,
    ),
    encoding="utf-8",
)


print(
    "\nwritten:",
    LOCK_OUT
)


print(
    "\n========== 12.03B-P1 DECISION =========="
)

print(
    "FOLD14_MORPHDIV_V2_BASELINE_LOCKED: PASS"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.03E Lock MUTUAL-CONT-V1
# Purpose:
#   Freeze the Stage12 continuation-recovery policy selected
#   exclusively from the predeclared folds1–4 development grid.
#
# Policy:
#   baseline = Baseline256 -> Frozen V9 -> MORPH-DIV-V2
#   source FINAL outdegree == 0
#   target FINAL indegree == 0
#   PROB row rank == 1
#   PROB column rank == 1
#   candidate_prob >= 0.25
#   additive continuation edge only
#
# Changes files: YES
#   reports/stage12_residual_audit/
#       stage12_mutual_cont_v1_policy_lock.json
#
# Runs training/inference: NO
# Uses GT: NO NEW GT READ
# Changes Stage11 policy: NO
# Keep after running: YES
# ============================================================

from pathlib import Path
import hashlib
import json

import pandas as pd


PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

RPT = (
    PROJECT
    / "reports/stage12_residual_audit"
)

GRID_PATH = (
    RPT
    / "stage12_mutual_best_policy_grid.pkl"
)

UNIVERSE_PATH = (
    RPT
    / "stage12_folds14_mutual_best_universe.pkl"
)

OUT = (
    RPT
    / "stage12_mutual_cont_v1_policy_lock.json"
)


grid = pd.read_pickle(
    GRID_PATH
)

universe = pd.read_pickle(
    UNIVERSE_PATH
)


row = (
    grid[
        grid["config_id"].eq(
            "MB_P250"
        )
    ]
    .iloc[0]
)


assert abs(
    float(row["threshold"]) - 0.25
) < 1e-12

assert int(
    row["actions"]
) == 46951

assert abs(
    float(row["score"])
    - 0.788841246
) <= 1e-9

assert abs(
    float(row["delta_score"])
    - 0.006948186
) <= 1e-9


# Exact development action identities under the locked rule.
dev_actions = (
    universe[
        universe[
            "candidate_prob"
        ].ge(0.25)
    ]
    .sort_values(
        [
            "fold",
            "dataset",
            "t",
            "source_pred_node_id",
            "target_pred_node_id",
        ]
    )
    .reset_index(drop=True)
)

assert len(dev_actions) == 46951


LOCK_COLS = [
    "dataset",
    "fold",
    "t",
    "source_pred_node_id",
    "target_pred_node_id",
    "candidate_prob",
]

blob = (
    dev_actions[
        LOCK_COLS
    ]
    .to_csv(
        index=False,
        float_format="%.17g",
    )
    .encode()
)

dev_action_sha = hashlib.sha256(
    blob
).hexdigest()


payload = {
    "stage":
        "12.03E",

    "policy_name":
        "MUTUAL-CONT-V1",

    "status":
        "LOCKED_BEFORE_FOLD0_OUTER_EVALUATION",

    "baseline":
        "BASELINE256_FROZEN_V9_MORPH_DIV_V2",

    "domain":
        "6bba",

    "rule": {
        "source_final_outdegree":
            0,

        "target_final_indegree":
            0,

        "prob_row_rank":
            1,

        "prob_column_rank":
            1,

        "candidate_prob_min":
            0.25,

        "operation":
            "ADD_SINGLE_CONTINUATION_EDGE",
    },

    "development": {
        "folds":
            [1, 2, 3, 4],

        "actions":
            int(len(dev_actions)),

        "baseline_score":
            0.781893060,

        "policy_score":
            float(row["score"]),

        "delta_score":
            float(row["delta_score"]),

        "action_sha256":
            dev_action_sha,
    },

    "prohibited_after_lock": [
        "change probability threshold",
        "change row-rank gate",
        "change column-rank gate",
        "change source outdegree gate",
        "change target indegree gate",
        "add geometry gate based on Fold0 result",
    ],
}


OUT.write_text(
    json.dumps(
        payload,
        indent=2,
    ),
    encoding="utf-8",
)


print(
    "========== MUTUAL-CONT-V1 POLICY =========="
)

print(
    "threshold       : 0.25"
)

print(
    "development actions:",
    len(dev_actions)
)

print(
    "development score  :",
    f"{row['score']:.9f}"
)

print(
    "development delta  :",
    f"{row['delta_score']:+.9f}"
)

print(
    "dev action SHA256  :",
    dev_action_sha
)


print(
    "\n========== POLICY CONTRACT =========="
)

print(
    "source FINAL outdegree == 0"
)

print(
    "target FINAL indegree  == 0"
)

print(
    "PROB row rank          == 1"
)

print(
    "PROB column rank       == 1"
)

print(
    "candidate_prob         >= 0.25"
)

print(
    "operation              = single continuation edge"
)


print(
    "\nwritten:",
    OUT
)


print(
    "\n========== 12.03E DECISION =========="
)

print(
    "MUTUAL_CONT_V1_LOCKED_BEFORE_FOLD0: PASS"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.04C Full 40-Dataset MUTUAL-CONT-V1 Integration
# Purpose:
#   Integrate the promoted MUTUAL-CONT-V1 Fold0 result into the
#   exact 40-dataset current-best hybrid.
#
#   44b6:
#       Sparse9995 seed1640 -> Frozen V9
#
#   6bba OLD:
#       Baseline256 -> Frozen V9 -> MORPH-DIV-V2
#
#   6bba NEW:
#       Baseline256 -> Frozen V9 -> MORPH-DIV-V2
#       -> MUTUAL-CONT-V1
#
#   First reproduce the frozen Stage11 40-dataset checkpoint,
#   then compute the new integrated score.
#
# Changes files: YES
#   reports/stage12_residual_audit/
#       stage12_full_hybrid_mutual_cont_v1_datasets.pkl
#       stage12_full_hybrid_mutual_cont_v1_summary.json
#       stage12_current_best_checkpoint.json
#
# Runs training/inference: NO
# Uses GT: YES — final integration evaluation
# Changes locked policies: NO
# Keep after running: YES
# ============================================================

from pathlib import Path
import json
import sys

import pandas as pd


PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

CV_PATH = (
    PROJECT
    / "configs/cv_manifest.csv"
)

P44_995 = (
    PROJECT
    / "predictions/heqiuyan/"
      "stage10_infer_sparse9995_256_det995/"
      "split_0"
)

P44_9975 = (
    PROJECT
    / "predictions/heqiuyan/"
      "stage10_infer_sparse9995_256_det9975/"
      "split_0"
)

S12 = (
    PROJECT
    / "reports/stage12_residual_audit"
)

OUTER_PATH = (
    S12
    / "stage12_fold0_mutual_cont_v1_outer_datasets.pkl"
)

ACTION_LOCK_PATH = (
    S12
    / "stage12_fold0_mutual_cont_v1_action_lock.json"
)

DATASET_OUT = (
    S12
    / "stage12_full_hybrid_mutual_cont_v1_datasets.pkl"
)

SUMMARY_OUT = (
    S12
    / "stage12_full_hybrid_mutual_cont_v1_summary.json"
)

CHECKPOINT_OUT = (
    S12
    / "stage12_current_best_checkpoint.json"
)


sys.path.insert(
    0,
    str(PROJECT / "src"),
)

from biohub_cell_tracking import frozen_v9 as fv9


# ============================================================
# 1. Fold0 corpus
# ============================================================

cv = pd.read_csv(
    CV_PATH
)

cv0 = (
    cv[
        cv["fold"].eq(0)
    ]
    .sort_values(
        ["prefix", "dataset"]
    )
    .reset_index(drop=True)
)

assert len(cv0) == 40


c44 = (
    cv0[
        cv0["dataset"].str.startswith(
            "44b6_"
        )
    ]
    .copy()
)

c6 = (
    cv0[
        cv0["dataset"].str.startswith(
            "6bba_"
        )
    ]
    .copy()
)

assert len(c44) == 15
assert len(c6) == 25


names44 = c44["dataset"].tolist()

prefix44 = dict(
    zip(
        c44["dataset"],
        c44["prefix"],
    )
)

estimated44 = dict(
    zip(
        c44["dataset"],
        c44["estimated_nodes"].astype(float),
    )
)


# ============================================================
# 2. Re-evaluate unchanged 44b6 route
# ============================================================

policies44 = fv9.build_frozen_v9_policies(
    names44,
    P44_995,
    P44_9975,
    TRAIN_ROOT,
    prefix_map=prefix44,
    config=fv9.FROZEN_V9_CONFIG,
    progress=False,
)


rows44 = []


for i, name in enumerate(
    names44,
    start=1,
):

    graph, scale, _ = fv9.apply_frozen_v9(
        name,
        P44_995,
        P44_9975,
        policies44,
        TRAIN_ROOT,
        config=fv9.FROZEN_V9_CONFIG,
    )

    gt = fv9.load_graph(
        TRAIN_ROOT
        / f"{name}.geff"
    )

    metric = fv9.evaluate_graph(
        graph,
        gt,
        scale,
        estimated44[name],
        fv9.FROZEN_V9_CONFIG,
    )

    rows44.append({
        "dataset":
            name,

        "route":
            "44b6_SPARSE9995_FROZEN_V9",

        "actions":
            0,

        **metric,
    })


assert len(rows44) == 15


# ============================================================
# 3. Load immutable 6bba locked outer result
# ============================================================

outer = pd.read_pickle(
    OUTER_PATH
)

assert len(outer) == 25
assert outer["dataset"].nunique() == 25


action_lock = json.loads(
    ACTION_LOCK_PATH.read_text(
        encoding="utf-8"
    )
)

assert (
    action_lock["action_sha256"]
    ==
    "28a54a53524eea310cd4f3d10d258f05"
    "cc73e706c96695a4b2f0d5362c454a2c"
)

assert int(
    action_lock["actions"]
) == 10107


METRIC_KEYS = [
    "edge_tp",
    "edge_fp",
    "edge_fn",
    "division_tp",
    "division_fp",
    "division_fn",
    "num_pred_nodes",
    "node_recall",
    "total_node_ratio",
    "edge_jaccard",
    "adj_edge_jaccard",
]


def extract_6bba(
    prefix,
    route,
):

    rows = []

    for r in outer.itertuples(
        index=False
    ):

        metric = {
            key:
                getattr(
                    r,
                    f"{prefix}_{key}",
                )
            for key in METRIC_KEYS
        }

        rows.append({
            "dataset":
                r.dataset,

            "route":
                route,

            "actions":
                (
                    int(r.actions)
                    if prefix == "new"
                    else 0
                ),

            **metric,
        })

    return rows


rows6_old = extract_6bba(
    "base",
    "6bba_BASELINE256_FROZEN_V9_MORPHDIV_V2",
)

rows6_new = extract_6bba(
    "new",
    "6bba_BASELINE256_FROZEN_V9_MORPHDIV_V2_MUTUAL_CONT_V1",
)


# ============================================================
# 4. Reproduce frozen Stage11 40-dataset checkpoint
# ============================================================

old_rows = (
    rows44
    + rows6_old
)

assert len(old_rows) == 40


old_metric_rows = [
    {
        key: r[key]
        for key in METRIC_KEYS
    }
    for r in old_rows
]


old = fv9.summarise(
    old_metric_rows
)


print(
    "========== OLD CURRENT-BEST REPRODUCTION =========="
)

print(
    "edge J      :",
    f"{old['edge_jaccard']:.9f}"
)

print(
    "adj edge J  :",
    f"{old['adj_edge_jaccard']:.9f}"
)

print(
    "division J  :",
    f"{old['division_jaccard']:.9f}"
)

print(
    "node recall :",
    f"{old['node_recall']:.9f}"
)

print(
    "score       :",
    f"{old['score']:.9f}"
)


assert abs(
    old["edge_jaccard"]
    - 0.781996647
) <= 1e-9

assert abs(
    old["adj_edge_jaccard"]
    - 0.774285647
) <= 1e-9

assert abs(
    old["division_jaccard"]
    - 0.046875000
) <= 1e-9

assert abs(
    old["node_recall"]
    - 0.959822345
) <= 1e-9

assert abs(
    old["score"]
    - 0.778973147
) <= 1e-9


# ============================================================
# 5. New 40-dataset integration
# ============================================================

new_rows = (
    rows44
    + rows6_new
)

assert len(new_rows) == 40


new_metric_rows = [
    {
        key: r[key]
        for key in METRIC_KEYS
    }
    for r in new_rows
]


new = fv9.summarise(
    new_metric_rows
)


old_tp = sum(
    int(r["edge_tp"])
    for r in old_rows
)

old_fp = sum(
    int(r["edge_fp"])
    for r in old_rows
)

old_fn = sum(
    int(r["edge_fn"])
    for r in old_rows
)

new_tp = sum(
    int(r["edge_tp"])
    for r in new_rows
)

new_fp = sum(
    int(r["edge_fp"])
    for r in new_rows
)

new_fn = sum(
    int(r["edge_fn"])
    for r in new_rows
)


print(
    "\n========== NEW FULL HYBRID + MUTUAL-CONT-V1 =========="
)

print(
    "edge TP/FP/FN:",
    new_tp,
    "/",
    new_fp,
    "/",
    new_fn,
)

print(
    "edge J      :",
    f"{new['edge_jaccard']:.9f}"
)

print(
    "adj edge J  :",
    f"{new['adj_edge_jaccard']:.9f}"
)

print(
    "division J  :",
    f"{new['division_jaccard']:.9f}"
)

print(
    "node recall :",
    f"{new['node_recall']:.9f}"
)

print(
    "score       :",
    f"{new['score']:.9f}"
)


print(
    "\n========== FULL 40-DATASET DELTAS =========="
)

print(
    "edge TP:",
    f"{new_tp-old_tp:+d}"
)

print(
    "edge FP:",
    f"{new_fp-old_fp:+d}"
)

print(
    "edge FN:",
    f"{new_fn-old_fn:+d}"
)

print(
    "edge J:",
    f"{new['edge_jaccard']-old['edge_jaccard']:+.9f}"
)

print(
    "adj edge J:",
    f"{new['adj_edge_jaccard']-old['adj_edge_jaccard']:+.9f}"
)

print(
    "division J:",
    f"{new['division_jaccard']-old['division_jaccard']:+.9f}"
)

print(
    "node recall:",
    f"{new['node_recall']-old['node_recall']:+.9f}"
)

print(
    "score:",
    f"{new['score']-old['score']:+.9f}"
)


assert (
    new["score"]
    > old["score"]
)


# ============================================================
# 6. Persist exact 40-dataset rows
# ============================================================

final_df = pd.DataFrame(
    new_rows
).sort_values(
    "dataset"
).reset_index(
    drop=True
)

final_df.to_pickle(
    DATASET_OUT
)


payload = {
    "stage":
        "12.04C",

    "pipeline":
        "FULL_HYBRID_PLUS_MORPH_DIV_V2_PLUS_MUTUAL_CONT_V1",

    "mutual_cont_v1_actions":
        10107,

    "mutual_cont_v1_action_sha256":
        action_lock[
            "action_sha256"
        ],

    "old": {
        "edge_jaccard":
            float(old["edge_jaccard"]),

        "adj_edge_jaccard":
            float(old["adj_edge_jaccard"]),

        "division_jaccard":
            float(old["division_jaccard"]),

        "node_recall":
            float(old["node_recall"]),

        "score":
            float(old["score"]),
    },

    "new": {
        "edge_tp":
            int(new_tp),

        "edge_fp":
            int(new_fp),

        "edge_fn":
            int(new_fn),

        "edge_jaccard":
            float(new["edge_jaccard"]),

        "adj_edge_jaccard":
            float(new["adj_edge_jaccard"]),

        "division_jaccard":
            float(new["division_jaccard"]),

        "node_recall":
            float(new["node_recall"]),

        "score":
            float(new["score"]),
    },

    "delta_score":
        float(
            new["score"]
            - old["score"]
        ),
}


SUMMARY_OUT.write_text(
    json.dumps(
        payload,
        indent=2,
    ),
    encoding="utf-8",
)


checkpoint = {
    "checkpoint_name":
        "FULL_HYBRID_PLUS_MORPH_DIV_V2_PLUS_MUTUAL_CONT_V1",

    "status":
        "STAGE12_CURRENT_BEST",

    "fold0_metrics": {
        "score":
            float(new["score"]),

        "edge_jaccard":
            float(new["edge_jaccard"]),

        "adj_edge_jaccard":
            float(new["adj_edge_jaccard"]),

        "division_jaccard":
            float(new["division_jaccard"]),

        "node_recall":
            float(new["node_recall"]),
    },

    "policies": {
        "MORPH-DIV-V2":
            "LOCKED",

        "MUTUAL-CONT-V1":
            "LOCKED_AND_PROMOTED",
    },

    "mutual_cont_v1": {
        "actions":
            10107,

        "action_sha256":
            action_lock[
                "action_sha256"
            ],

        "threshold":
            0.25,

        "row_rank":
            1,

        "column_rank":
            1,

        "source_final_outdegree":
            0,

        "target_final_indegree":
            0,
    },
}


CHECKPOINT_OUT.write_text(
    json.dumps(
        checkpoint,
        indent=2,
    ),
    encoding="utf-8",
)


print(
    "\nwritten:",
    DATASET_OUT
)

print(
    "written:",
    SUMMARY_OUT
)

print(
    "written:",
    CHECKPOINT_OUT
)


print(
    "\n========== 12.04C DECISION =========="
)

print(
    "FULL_HYBRID_MUTUAL_CONT_V1_INTEGRATED: PASS"
)